# 0. Setup & Imports
Install / import all required packages, set random seeds for reproducibility, and define global paths & hyper-parameters.

**【KeyBERT representation note】**
- This notebook now uses BERTopic's built-in `KeyBERTInspired` representation layer instead of running a separate post-hoc KeyBERT pass.
- Topic keywords are generated directly inside BERTopic, and then MiniMax is used to produce Chinese topic titles and one-sentence summaries.

**【中文说明】**
- 本节为全笔记本的环境准备：安装/导入依赖、固定随机种子；路径与多数超参数见 `config/cluster_keybert_from_cluster.json`，首节代码运行时加载。
- 聚类仍使用 `allenai/specter2_base` 文档向量缓存；主题表示层额外使用真正的 `allenai/specter2` adapter 语义编码器来支持 BERTopic 的 KeyBERT 表示层。
- 同时配置 MiniMax 中文主题总结接口与缓存文件，保证 notebook 可重复运行且主题结构清晰。
- 设备自动选择 CUDA → MPS → CPU，以便在 GPU 或 Apple Silicon 上加速编码。


In [ ]:
# Centralized setup: config parsing, reproducibility, path bootstrap, and shared imports.

import os
import json
import random
import warnings
import time
import hashlib
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Sequence

# Avoid TensorFlow/Keras import path in transformers to prevent Keras 3 incompatibility.
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("TRANSFORMERS_NO_TF", "1")

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from sklearn.preprocessing import normalize

from sentence_transformers import SentenceTransformer
from umap import UMAP
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.neighbors import kneighbors_graph
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from bertopic import BERTopic
from bertopic.backend import BaseEmbedder
from bertopic.representation import KeyBERTInspired
from scipy.spatial.distance import jensenshannon
from scipy.cluster.hierarchy import linkage, dendrogram

from scripts.cluster_pipeline.config_utils import (
    ensure_output_dirs,
    load_json_config,
    set_reproducibility,
    unpack_runtime_config,
)
from scripts.cluster_pipeline.device_utils import select_device

_CONFIG_PATH = Path("config/cluster_keybert_from_cluster.json")
_cfg = load_json_config(_CONFIG_PATH)
_runtime = unpack_runtime_config(_cfg, ENGLISH_STOP_WORDS)

SEED = _runtime["SEED"]
set_reproducibility(SEED)

DATA_PATH = _runtime["DATA_PATH"]
OUTPUT_DIR = _runtime["OUTPUT_DIR"]
_created_dirs = ensure_output_dirs(OUTPUT_DIR, [_runtime["HIERARCHICAL_SUBDIR"]])
HIER_DIR = _created_dirs[_runtime["HIERARCHICAL_SUBDIR"]]

SPECTER_MODEL = _runtime["SPECTER_MODEL"]
SPECTER2_BASE_MODEL = _runtime["SPECTER2_BASE_MODEL"]
SPECTER2_ADAPTER_MODEL = _runtime["SPECTER2_ADAPTER_MODEL"]
SPECTER2_CACHE_ROOT = _runtime["SPECTER2_CACHE_ROOT"]
SPECTER2_KEYBERT_BATCH_SIZE = _runtime["SPECTER2_KEYBERT_BATCH_SIZE"]
TOP_N_KEYWORDS = _runtime["TOP_N_KEYWORDS"]

MINIMAX_API_BASE = _runtime["MINIMAX_API_BASE"]
MINIMAX_CHAT_ENDPOINT = _runtime["MINIMAX_CHAT_ENDPOINT"]
MINIMAX_MODEL = _runtime["MINIMAX_MODEL"]
MINIMAX_API_KEY = os.getenv(_runtime["MINIMAX_API_KEY_ENV"], "")
MINIMAX_TIMEOUT = _runtime["MINIMAX_TIMEOUT"]
MINIMAX_MAX_RETRIES = _runtime["MINIMAX_MAX_RETRIES"]
MINIMAX_RETRY_SLEEP = _runtime["MINIMAX_RETRY_SLEEP"]
TOPIC_SUMMARY_CACHE = os.path.join(OUTPUT_DIR, _runtime["TOPIC_SUMMARY_CACHE_FILENAME"])

UMAP_N_NEIGHBORS = _runtime["UMAP_N_NEIGHBORS"]
UMAP_N_COMPONENTS = _runtime["UMAP_N_COMPONENTS"]
UMAP_MIN_DIST = _runtime["UMAP_MIN_DIST"]
UMAP_METRIC = _runtime["UMAP_METRIC"]

AGGLO_N_CLUSTERS = _runtime["AGGLO_N_CLUSTERS"]
AGGLO_LINKAGE = _runtime["AGGLO_LINKAGE"]
AGGLO_METRIC = _runtime["AGGLO_METRIC"]

AGGLO_SEARCH_SUBSET_N = _runtime["AGGLO_SEARCH_SUBSET_N"]
AGGLO_SEARCH_TOPK_FULL = _runtime["AGGLO_SEARCH_TOPK_FULL"]
AGGLO_SEARCH_MAX_EVAL_POINTS = _runtime["AGGLO_SEARCH_MAX_EVAL_POINTS"]
AGGLO_SEARCH_LINKAGE_METRIC_COARSE = _runtime["AGGLO_SEARCH_LINKAGE_METRIC_COARSE"]
AGGLO_SEARCH_LINKAGE_METRIC_FULL = _runtime["AGGLO_SEARCH_LINKAGE_METRIC_FULL"]

CUSTOM_STOPWORDS = _runtime["CUSTOM_STOPWORDS"]
NGRAM_RANGE = _runtime["NGRAM_RANGE"]
MAX_FEATURES = _runtime["MAX_FEATURES"]

BATCH_SIZE = _runtime["BATCH_SIZE"]
TOP_K_TOPICS = _runtime["TOP_K_TOPICS"]

DEVICE = select_device()
print(f"Device selected: {DEVICE}")
print(f"Config file: {_CONFIG_PATH.resolve()}")
print(f"Output directory: {Path(OUTPUT_DIR).resolve()}")
print(f"Hierarchical output directory: {HIER_DIR.resolve()}")

# 1. Load Data & Cleaning
Read the SCIE CSV, normalise column names, handle missing titles/abstracts, construct the combined text field, and report data statistics.

**【中文说明】**
- 从 WoS 导出的 SCIE CSV 读入后，将常用字段映射为 `title`/`abstract`/`year` 等统一列名。
- 清洗规则：标题与摘要不能同时缺失；拼接为 `text` 供嵌入；过短文本剔除；国家名规范为 CN/US/其他。
- 保留全样本做主题建模，同时生成 `country_code` 供后续中美对比分析。


In [ ]:
# Load and clean data via reusable pipeline utility.

from scripts.cluster_pipeline.data_prep import load_and_clean_papers

df, data_stats = load_and_clean_papers(DATA_PATH, min_text_len=20)
HAS_YEAR = bool(data_stats["has_year"])

print(f"Raw rows: {data_stats['raw_rows']:,}")
print(f"Rows after cleaning: {data_stats['clean_rows']:,} (dropped {data_stats['dropped_rows']:,})")
print(f"Missing title: {data_stats['missing_title']:,}")
print(f"Missing abstract: {data_stats['missing_abstract']:,}")

if "country" in df.columns:
    print("\nCountry distribution (top 10):")
    print(df["country"].value_counts().head(10).to_string())

if HAS_YEAR:
    print("\nYear range:")
    print(f"  {df['year'].min()} - {df['year'].max()}")
    print(df["year"].value_counts().sort_index().tail(10).to_string())
else:
    print("\nYear column not available; time-based analysis may be skipped.")

print(f"\nCN papers: {data_stats['cn_papers']:,}")
print(f"US papers: {data_stats['us_papers']:,}")
print(f"Other:     {data_stats['other_papers']:,}")

docs = df["text"].tolist()
print(f"\nDocuments ready for embedding: {len(docs):,}")

# 2. Embedding with SPECTER2
Load the `allenai/specter2` sentence-transformer, encode all documents with progress bar, and L2-normalise the embedding matrix.

**【中文说明】**
- 使用 SPECTER2（SentenceTransformer 接口）将每篇论文的 `text` 编码为高维向量，并进行 L2 归一化，使余弦相似度等价于点积。
- 向量可缓存为 `.npy`，避免重复下载模型与重复编码，加快迭代。


In [ ]:
# 【单元格中文说明】
# - 以下为可选路径：用 HuggingFace adapters 加载带 adapter 的 SPECTER2，当前整段注释保留作参考。
# - 若需与论文完全一致的可尝试取消注释；默认流程使用下一单元格的 SentenceTransformer 基座模型。

# ── 2.1 Load SPECTER2 model via adapters library ────────────────────────────
# allenai/specter2_base = base transformer
# allenai/specter2      = proximity adapter (retrieval tasks)
EMBEDDINGS_CACHE = os.path.join(OUTPUT_DIR, "specter2_embeddings_cache.npy")

if os.path.exists(EMBEDDINGS_CACHE):
    embeddings = np.load(EMBEDDINGS_CACHE)
    print(f"✅ Loaded cached embeddings from {EMBEDDINGS_CACHE}")
else:
    try:
        from transformers import AutoTokenizer
        from adapters import AutoAdapterModel

        SPECTER2_BASE    = "allenai/specter2_base"
        SPECTER2_ADAPTER = "allenai/specter2"   # proximity / retrieval adapter

        print(f"Loading tokenizer: {SPECTER2_BASE} …")
        tokenizer = AutoTokenizer.from_pretrained(SPECTER2_BASE)

        print(f"Loading base model: {SPECTER2_BASE} …")
        model = AutoAdapterModel.from_pretrained(SPECTER2_BASE)

        print(f"Loading adapter: {SPECTER2_ADAPTER} …")
        model.load_adapter(SPECTER2_ADAPTER, source="hf", load_as="proximity", set_active=True)

        # Choose device — fall back to CPU if MPS causes issues
        import torch
        _device = torch.device(DEVICE)
        model = model.to(_device)
        model.eval()

        print(f"Encoding {len(docs):,} documents (batch_size={BATCH_SIZE}, device={_device}) …")

        all_embeddings = []
        for i in range(0, len(docs), BATCH_SIZE):
            batch = docs[i : i + BATCH_SIZE]
            inputs = tokenizer(
                batch,
                padding=True,
                truncation=True,
                return_tensors="pt",
                return_token_type_ids=False,
                max_length=512,
            )
            inputs = {k: v.to(_device) for k, v in inputs.items()}
            with torch.no_grad():
                output = model(**inputs)
            # CLS token → embedding
            batch_emb = output.last_hidden_state[:, 0, :].cpu().float().numpy()
            all_embeddings.append(batch_emb)
            if (i // BATCH_SIZE) % 20 == 0:
                print(f"  {min(i + BATCH_SIZE, len(docs)):,}/{len(docs):,} docs encoded…")

        embeddings = np.vstack(all_embeddings)
        loaded_ok = True

    except Exception as e:
        print(f"⚠️  SPECTER2 (adapters) failed: {e}")
        print("Falling back to allenai/specter (original SPECTER via SentenceTransformer) …")
        loaded_ok = False

    if not loaded_ok:
        # ── Fallback: original SPECTER via SentenceTransformer ───────────────
        from sentence_transformers import SentenceTransformer
        FALLBACK_MODEL = "allenai/specter"
        encoder = SentenceTransformer(FALLBACK_MODEL, device=DEVICE)
        print(f"Encoding {len(docs):,} documents with {FALLBACK_MODEL} …")
        embeddings = encoder.encode(
            docs,
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True,
        )

    # ── 2.3 L2 normalise ────────────────────────────────────────────────────
    embeddings = normalize(embeddings, norm="l2")

    # ── 2.4 Cache to disk ───────────────────────────────────────────────────
    np.save(EMBEDDINGS_CACHE, embeddings)
    print(f"✅ Embeddings cached to {EMBEDDINGS_CACHE}")

print(f"Embeddings shape: {embeddings.shape}  (n_docs × dim)")


In [ ]:
# # Embedding stage with cache-first policy and automatic device fallback.

# from scripts.cluster_pipeline.embedding_utils import load_or_create_embeddings

# EMBEDDINGS_CACHE = os.path.join(OUTPUT_DIR, "specter2_embeddings_cache.npy")
# embeddings, cache_hit, used_device = load_or_create_embeddings(
#     docs=docs,
#     cache_path=EMBEDDINGS_CACHE,
#     model_name=SPECTER_MODEL,
#     device=DEVICE,
#     batch_size=BATCH_SIZE,
#     )

# if cache_hit:
#     print(f"Loaded cached embeddings from {EMBEDDINGS_CACHE}")
# else:
#     print(f"Computed and cached embeddings to {EMBEDDINGS_CACHE}")

# if used_device != DEVICE:
#     print(f"Device fallback applied: {DEVICE} -> {used_device}")
#     DEVICE = used_device

# print(f"Embeddings shape: {embeddings.shape} (n_docs x dim)")

# 3. Build BERTopic Pipeline (UMAP + AgglomerativeClustering + Vectorizer)
Assemble dimensionality reduction, hierarchical clustering, and c-TF-IDF components.  
This section also runs two-stage automatic parameter search and writes best params back to the main pipeline.

**【中文说明】**
- 本笔记本用 **层次聚类（AgglomerativeClustering）** 替代 BERTopic 默认的 HDBSCAN，使每篇文献都有明确主题（无默认噪声类）。
- 流程含两阶段超参搜索：粗网格在子样本上筛选，再在全量数据上精评；综合轮廓系数、DBI、主题规模均衡与中美覆盖等秩和指标。
- BERTopic 仍先基于 c-TF-IDF 生成候选词，但最终主题词由 `KeyBERTInspired` 表示层结合 specter2 语义相似度直接重排；`hdbscan_model` 参数名保留，但实际传入的是 sklearn 聚类器。


In [ ]:
# Agglomerative search + BERTopic assembly (modularized orchestration).
# Best UMAP+Agglo params are loaded from a prior grid-search export (no live grid search).

from pathlib import Path

from scripts.cluster_pipeline.agglom_pipeline import (
    _safe_cluster_scores,
    build_agglomerative_model,
    compute_topic_confidence,
    compute_topic_size_stats,
    plot_topic_centroid_dendrogram,
    # run_agglomerative_search,  # re-enable when running the grid search block below
)
from scripts.cluster_pipeline.specter2_backend import (
    get_specter2_representation_encoder as _get_specter2_representation_encoder,
)

# run_agglomerative_search writes both files with identical rows; prefer the canonical name.
for _p in (
    Path(HIER_DIR) / "agglom_search_results.csv",
    Path(HIER_DIR) / "results_agglomerative_grid.csv",
):
    if _p.is_file():
        AGGLOM_SEARCH_RESULTS_CSV = _p
        break
else:
    raise FileNotFoundError(
        f"Missing agglom search export under {HIER_DIR}. Expected "
        f'agglom_search_results.csv (or results_agglomerative_grid.csv). Run the commented grid-search block once.'
    )

agglom_search_df = pd.read_csv(AGGLOM_SEARCH_RESULTS_CSV)
_ok = agglom_search_df.loc[agglom_search_df["status"].astype(str).str.lower() == "ok"].copy()
if _ok.empty:
    raise RuntimeError(f"No status==ok rows in {AGGLOM_SEARCH_RESULTS_CSV}")
if "stage" in _ok.columns:
    _full = _ok.loc[_ok["stage"].astype(str).str.lower() == "full"]
    if not _full.empty:
        _ok = _full
_ok = _ok.sort_values("silhouette", ascending=False, na_position="last")
_param_keys = [
    "umap_n_neighbors",
    "umap_n_components",
    "umap_min_dist",
    "umap_metric",
    "agglom_n_clusters",
    "agglom_linkage",
    "agglom_metric",
]
_best_row = _ok.iloc[0]
best_agglom_params = {k: _best_row[k] for k in _param_keys}

# --- Grid search (disabled): uncomment imports + block below to re-run two-stage search ---
from scripts.cluster_pipeline.agglom_pipeline import run_agglomerative_search

_umap_search_cfg = _cfg.get("umap_search", {})
_umap_neighbors_grid = _umap_search_cfg.get("n_neighbors_grid", [15])
_umap_components_grid = _umap_search_cfg.get("n_components_grid", [5, 10])
_umap_min_dist_grid = _umap_search_cfg.get("min_dist_grid", [0.0, 0.1])
_cluster_counts = _cfg.get("agglomerative_search", {}).get(
    "cluster_counts", [15, 20, 25, 30, 40, 45, 50]
)
_eval_gap_fn = None
if "compute_gap_metrics" in globals() and callable(compute_gap_metrics):
    def _eval_gap_fn(labels_in):
        g = compute_gap_metrics(df, labels_in, topic_col_name="_topic_eval")
        return {
            "cn_coverage": g.get("cn_coverage", np.nan),
            "us_coverage": g.get("us_coverage", np.nan),
            "lead_lag_n_topics": g.get("lead_lag_n_topics", np.nan),
        }

agglom_search_df, best_agglom_params = run_agglomerative_search(
    embeddings_all=embeddings,
    seed=SEED,
    output_dir=HIER_DIR,
    umap_metric=UMAP_METRIC,
    search_subset_n=AGGLO_SEARCH_SUBSET_N,
    search_topk_full=AGGLO_SEARCH_TOPK_FULL,
    search_max_eval_points=AGGLO_SEARCH_MAX_EVAL_POINTS,
    linkage_metric_coarse=AGGLO_SEARCH_LINKAGE_METRIC_COARSE,
    linkage_metric_full=AGGLO_SEARCH_LINKAGE_METRIC_FULL,
    umap_neighbors_grid=_umap_neighbors_grid,
    umap_components_grid=_umap_components_grid,
    umap_min_dist_grid=_umap_min_dist_grid,
    cluster_counts=_cluster_counts,
    evaluate_gap_fn=_eval_gap_fn,
)
# --- end grid search ---

UMAP_N_NEIGHBORS = int(best_agglom_params["umap_n_neighbors"])
UMAP_N_COMPONENTS = int(best_agglom_params["umap_n_components"])
UMAP_MIN_DIST = float(best_agglom_params["umap_min_dist"])
UMAP_METRIC = str(best_agglom_params["umap_metric"])
AGGLO_N_CLUSTERS = int(best_agglom_params["agglom_n_clusters"])
AGGLO_LINKAGE = str(best_agglom_params["agglom_linkage"])
AGGLO_METRIC = str(best_agglom_params["agglom_metric"])

print(f"\nBest params loaded from {AGGLOM_SEARCH_RESULTS_CSV} (rank_score-min among ok rows; prefer stage=full).")
print(
    f"UMAP(n_neighbors={UMAP_N_NEIGHBORS}, n_components={UMAP_N_COMPONENTS}, "
    f"min_dist={UMAP_MIN_DIST}, metric='{UMAP_METRIC}')"
)
print(
    f"Agglomerative(n_clusters={AGGLO_N_CLUSTERS}, linkage='{AGGLO_LINKAGE}', metric='{AGGLO_METRIC}')"
)

umap_model = UMAP(
    n_neighbors=UMAP_N_NEIGHBORS,
    n_components=UMAP_N_COMPONENTS,
    min_dist=UMAP_MIN_DIST,
    metric=UMAP_METRIC,
    random_state=SEED,
    low_memory=True,
)

hdbscan_model = build_agglomerative_model(
    n_clusters=AGGLO_N_CLUSTERS,
    linkage_name=AGGLO_LINKAGE,
    metric_name=AGGLO_METRIC,
)

vectorizer_model = CountVectorizer(
    stop_words=CUSTOM_STOPWORDS,
    ngram_range=NGRAM_RANGE,
    max_features=MAX_FEATURES,
)

SPECTER2_REPR_ENCODER = globals().get("SPECTER2_REPR_ENCODER", None)


def get_specter2_representation_encoder(force_reload: bool = False):
    global SPECTER2_REPR_ENCODER
    if force_reload or SPECTER2_REPR_ENCODER is None:
        SPECTER2_REPR_ENCODER = _get_specter2_representation_encoder(
            base_model=SPECTER2_BASE_MODEL,
            adapter_model=SPECTER2_ADAPTER_MODEL,
            cache_root=SPECTER2_CACHE_ROOT,
            batch_size=SPECTER2_KEYBERT_BATCH_SIZE,
            device=DEVICE,
            force_reload=force_reload,
        )
    return SPECTER2_REPR_ENCODER


representation_embedding_model = get_specter2_representation_encoder()
representation_model = KeyBERTInspired()

topic_model = BERTopic(
    embedding_model=representation_embedding_model,
    representation_model=representation_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    top_n_words=TOP_N_KEYWORDS,
    calculate_probabilities=False,
    verbose=True,
)

print("BERTopic hierarchical pipeline assembled with specter2-backed KeyBERT representation.")

# 4. Fit & Transform
Run BERTopic with best UMAP + Agglomerative parameters, then create surrogate topic confidence and hierarchical diagnostics.

**【中文说明】**
- 在最优 UMAP+层次聚类参数下拟合 BERTopic，将主题编号写回 `df`，并计算“代理主题置信度”（样本与主题质心余弦相似度映射到 0–1）。
- 输出 UMAP 空间聚类质量、主题规模分布与主题质心树状图，用于诊断主题是否过度碎片化或失衡。


In [ ]:
# 【单元格中文说明】
# - fit_transform：以预计算 embeddings 训练 BERTopic；topics 写入 df['topic']；topic_prob 为主题级置信度代理。
# - 在固定 UMAP 与 Agglo linkage/metric 下扫描 n_clusters∈[25,60]，于 UMAP 空间计算 WCSS 并绘制肘部图，导出 CSV。
# - 计算最终轮廓系数与 DBI（在 UMAP 嵌入空间）；绘制最终二维 UMAP 散点、主题规模直方图与主题质心层次树状图。


# ── 4.1 Fit BERTopic with best params ────────────────────────────────────────
# 中文：fit_transform 使用预计算 embeddings；topics 为每篇文献的整数主题 ID。
topics, probs = topic_model.fit_transform(docs, embeddings)
topics = np.asarray(topics).astype(int)

# ── 4.2 Write results back to DataFrame ─────────────────────────────────────
df["topic"] = topics
# Surrogate confidence in [0,1], based on cosine(sample, topic centroid) in original embedding space.
df["topic_prob"] = compute_topic_confidence(embeddings, topics)  # 中文：非模型原生概率，仅作诊断用

# ── 4.3 Hierarchical clustering statistics ───────────────────────────────────
size_stats_main = compute_topic_size_stats(topics)
n_topics = int(size_stats_main["n_topics"])

outlier_ratio = 0.0  # hierarchical setting has no explicit noise class by design

# Final silhouette/DBI on model UMAP embeddings
reduced_for_eval = getattr(topic_model.umap_model, "embedding_", None)
if reduced_for_eval is None:
    reduced_for_eval = UMAP(
        n_neighbors=UMAP_N_NEIGHBORS,
        n_components=UMAP_N_COMPONENTS,
        min_dist=UMAP_MIN_DIST,
        metric=UMAP_METRIC,
        random_state=SEED,
        low_memory=True,
    ).fit_transform(embeddings)

# ── 4.3a Elbow sweep (n_clusters 25–60), same UMAP embedding + Agglo linkage/metric ──
def _within_cluster_sse_euclidean(X, labels):
    """Sum of squared Euclidean distances to cluster centroids (elbow criterion in UMAP space)."""
    X = np.asarray(X, dtype=float)
    labels = np.asarray(labels)
    total = 0.0
    for lab in np.unique(labels):
        pts = X[labels == lab]
        if pts.shape[0] == 0:
            continue
        c = pts.mean(axis=0)
        total += float(np.sum((pts - c) ** 2))
    return total


_Xr = np.asarray(reduced_for_eval, dtype=float)
_elbow_rows = []
for _nk in range(25, 70, 5):
    _m = build_agglomerative_model(
        n_clusters=_nk,
        linkage_name=AGGLO_LINKAGE,
        metric_name=AGGLO_METRIC,
    )
    _lab = _m.fit_predict(_Xr)
    _sse = _within_cluster_sse_euclidean(_Xr, _lab)
    _sil_k, _dbi_k = _safe_cluster_scores(
        _Xr, _lab, seed=SEED, max_eval_points=AGGLO_SEARCH_MAX_EVAL_POINTS
    )
    _elbow_rows.append(
        {"n_clusters": _nk, "within_cluster_sse": _sse, "silhouette": _sil_k, "dbi": _dbi_k}
    )
AGGLO_ELBOW_SWEEP_DF = pd.DataFrame(_elbow_rows)
HIER_DIR.mkdir(parents=True, exist_ok=True)
_elbow_csv = HIER_DIR / "agglom_elbow_n_clusters_sweep.csv"
AGGLO_ELBOW_SWEEP_DF.to_csv(_elbow_csv, index=False)
print(f"✅ Saved {_elbow_csv}")

fig_elbow, ax_elbow = plt.subplots(figsize=(9, 5.5))
ax_elbow.plot(
    AGGLO_ELBOW_SWEEP_DF["n_clusters"],
    AGGLO_ELBOW_SWEEP_DF["within_cluster_sse"],
    marker="o",
    ms=4,
    color="#3C6FE0",
)
ax_elbow.set_xlabel("n_clusters")
ax_elbow.set_ylabel("Within-cluster SSE (UMAP space, Euclidean)")
ax_elbow.set_title(
    "Elbow: Agglomerative n_clusters sweep (25–60)\n"
    f"Fixed UMAP({UMAP_N_NEIGHBORS},{UMAP_N_COMPONENTS},{UMAP_MIN_DIST},{UMAP_METRIC}) | "
    f"linkage={AGGLO_LINKAGE!r}, metric={AGGLO_METRIC!r}"
)
ax_elbow.grid(True, alpha=0.3)
plt.tight_layout()
_elbow_png = HIER_DIR / "agglom_elbow_n_clusters.png"
fig_elbow.savefig(_elbow_png, dpi=180)
plt.close(fig_elbow)
print(f"✅ Saved {_elbow_png}")

final_silhouette, final_dbi = _safe_cluster_scores(
    reduced_for_eval, topics, seed=SEED, max_eval_points=AGGLO_SEARCH_MAX_EVAL_POINTS
)

# Keep globally for downstream summary
FINAL_CLUSTER_STATS = {
    **size_stats_main,
    "outlier_ratio": outlier_ratio,
    "silhouette": final_silhouette,
    "dbi": final_dbi,
}

print(f"{'─'*60}")
print(f"Number of topics:               {n_topics}")
print(f"Outlier ratio:                  {outlier_ratio:.1%}")
print(f"Silhouette (UMAP space):        {final_silhouette:.4f}" if pd.notna(final_silhouette) else "Silhouette (UMAP space):        N/A")
print(f"Davies-Bouldin Index (UMAP):    {final_dbi:.4f}" if pd.notna(final_dbi) else "Davies-Bouldin Index (UMAP):    N/A")
print(f"Largest  topic size:            {size_stats_main['topic_size_max']:.0f}")
print(f"Smallest topic size:            {size_stats_main['topic_size_min']:.0f}")
print(f"Median   topic size:            {size_stats_main['topic_size_median']:.0f}")
print(f"Max topic share:                {size_stats_main['max_topic_share']:.2%}")
print(f"Topic size CV:                  {size_stats_main['topic_size_cv']:.4f}")
print(f"{'─'*60}")

# ── 4.4 Hierarchical visualizations and exports ─────────────────────────────
# 中文：导出最终 UMAP 散点、主题规模图与质心树状图。
HIER_DIR.mkdir(parents=True, exist_ok=True)

# (1) Final UMAP 2D scatter
rs = np.random.RandomState(SEED)
max_plot_n = 15000
if len(df) > max_plot_n:
    plot_idx = rs.choice(np.arange(len(df)), size=max_plot_n, replace=False)
    plot_idx = np.sort(plot_idx)
else:
    plot_idx = np.arange(len(df))

umap_2d_final = UMAP(
    n_neighbors=UMAP_N_NEIGHBORS,
    n_components=2,
    min_dist=0.1,
    metric=UMAP_METRIC,
    random_state=SEED,
    low_memory=True,
)
xy = umap_2d_final.fit_transform(np.asarray(embeddings)[plot_idx])
topic_plot = topics[plot_idx]

fig, ax = plt.subplots(figsize=(14, 10))
scatter = ax.scatter(
    xy[:, 0],
    xy[:, 1],
    c=topic_plot,
    cmap="tab20",
    s=4,
    alpha=0.55,
)
ax.set_title(
    "Final UMAP 2D — Topics from AgglomerativeClustering\n"
    f"best params: UMAP({UMAP_N_NEIGHBORS},{UMAP_N_COMPONENTS},{UMAP_MIN_DIST},{UMAP_METRIC}) | "
    f"Agglo({AGGLO_N_CLUSTERS},{AGGLO_LINKAGE},{AGGLO_METRIC})",
    fontsize=12,
)
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
cb = plt.colorbar(scatter, ax=ax, shrink=0.75)
cb.set_label("Topic ID")
plt.tight_layout()
umap_scatter_final_path = HIER_DIR / "umap_scatter_final.png"
fig.savefig(umap_scatter_final_path, dpi=200)
plt.close(fig)
print(f"✅ Saved {umap_scatter_final_path}")

try:
    import plotly.express as px

    plot_df_html = pd.DataFrame(
        {"x": xy[:, 0], "y": xy[:, 1], "topic": topic_plot.astype(str)}
    )
    fig_html = px.scatter(
        plot_df_html,
        x="x",
        y="y",
        color="topic",
        title="Final UMAP 2D (Agglomerative topics)",
        opacity=0.6,
    )
    fig_html.write_html(HIER_DIR / "umap_scatter_final.html")
except Exception:
    pass

# (2) Topic-size distribution
topic_size_series = pd.Series(topics).value_counts().sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(np.arange(len(topic_size_series)), topic_size_series.values, color="#3C6FE0", alpha=0.85)
axes[0].set_title("Topic Size (sorted)")
axes[0].set_xlabel("Topic rank")
axes[0].set_ylabel("Documents")
axes[0].grid(True, axis="y", alpha=0.3)
axes[1].hist(topic_size_series.values, bins=min(30, len(topic_size_series)), color="#E03C31", alpha=0.8, edgecolor="white")
axes[1].set_title("Topic Size Histogram")
axes[1].set_xlabel("Documents per topic")
axes[1].set_ylabel("Frequency")
axes[1].grid(True, axis="y", alpha=0.3)
plt.tight_layout()
topic_size_dist_path = HIER_DIR / "topic_size_distribution.png"
fig.savefig(topic_size_dist_path, dpi=180)
plt.close(fig)
print(f"✅ Saved {topic_size_dist_path}")

# (3) Topic centroid dendrogram
plot_topic_centroid_dendrogram(
    embeddings,
    topics,
    HIER_DIR / "topic_centroid_dendrogram.png",
)


# 5. Topic Interpretation
Read KeyBERT-inspired keywords directly from BERTopic's representation layer, build a summary table, and retrieve representative documents.

**【中文说明】**
- 直接读取 BERTopic 内置 `KeyBERTInspired` 表示层生成的主题关键词，并在此基础上构建统一的主题摘要表。
- 再调用 MiniMax 生成中文短标题与一句话摘要，使后续图表标签与解释文本保持一致。
- 代表性文档用于人工核对主题标签是否符合领域语义。


In [ ]:
# 【单元格中文说明】
# - get_topic_info() 返回每个主题的文档数、自动名称等；供后续导出与核对。

# ── 5.1 Topic info table ─────────────────────────────────────────────────────
topic_info = topic_model.get_topic_info()
print(f"topic_info rows: {len(topic_info)}")
topic_info.head(10)


# 5.2 KeyBERT 主题标签与 MiniMax 中文总结
直接读取 BERTopic 内置 `KeyBERTInspired` 表示层产出的主题词，不再运行独立的 post-hoc KeyBERT。  
随后基于 KeyBERT 关键词与代表文献片段，调用 MiniMax 生成中文短标题与一句话摘要。

**【中文说明】**
- 主题关键词统一来自 `topic_model.get_topic(...)`，确保后续图表、诊断和导出表使用同一套主题表示。
- MiniMax 结果会缓存到 `topic_summaries_minimax.json`，重复运行时若关键词未变则直接复用。


In [ ]:
# 【单元格中文说明】
# - 本单元不再额外运行独立 KeyBERT；而是直接读取 BERTopic 内置 KeyBERT 表示层产出的关键词。
# - 然后调用 MiniMax-M2.7-highspeed，在 KeyBERT 关键词与代表文献片段的基础上生成中文短标题与一句话摘要。
# - 结果缓存到 TOPIC_SUMMARY_CACHE；若关键词不变，则重复运行时不会再次请求 API。

# ── 5.2 Helpers: cache + MiniMax summarisation ─────────────────────────────
def _load_topic_summary_cache(path: str = TOPIC_SUMMARY_CACHE) -> Dict[str, Dict[str, Any]]:
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}


def _save_topic_summary_cache(cache: Dict[str, Dict[str, Any]], path: str = TOPIC_SUMMARY_CACHE) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)


def _summary_cache_key(topic_id: int, keywords: Sequence[str]) -> str:
    payload = f"{topic_id}::{' | '.join(keywords)}"
    return hashlib.md5(payload.encode("utf-8")).hexdigest()


def _fallback_topic_summary(topic_id: int, keywords: Sequence[str]) -> Dict[str, str]:
    clean_words = [str(word).strip() for word in keywords if str(word).strip()]
    short_title = " / ".join(clean_words[:3]) if clean_words else f"主题{topic_id}"
    summary = (
        f"该主题对应核领域中的 {', '.join(clean_words[:5])} 等细分方向，"
        if clean_words
        else f"该主题聚焦于核领域内部的细分研究方向 {topic_id}，"
    )
    return {
        "title": short_title[:30],
        "summary": (summary + "用于补充 BERTopic 的中文细分主题解释。")[:80],
    }


def _extract_json_object(text: str) -> Dict[str, Any]:
    text = (text or "").strip()
    if not text:
        raise ValueError("Empty MiniMax response")
    try:
        data = json.loads(text)
        if isinstance(data, dict):
            return data
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError(f"No JSON object found in response: {text[:200]}")
    data = json.loads(text[start : end + 1])
    if not isinstance(data, dict):
        raise ValueError("MiniMax response JSON is not an object")
    return data


def _build_topic_summary_messages(topic_id: int, keywords: Sequence[str], rep_docs: Sequence[str]) -> List[Dict[str, str]]:
    keywords_text = ", ".join([str(word).strip() for word in keywords if str(word).strip()])
    rep_docs_text = "\n\n".join([str(doc)[:400] for doc in rep_docs[:3] if str(doc).strip()])
    user_prompt = f"""
请根据以下 BERTopic 主题信息，生成中文主题标题和一句话摘要。

要求：
1. 仅输出 JSON，不要输出 Markdown 或额外解释。
2. JSON 格式必须为 {{\"title\": \"...\", \"summary\": \"...\"}}。
3. 所有文本都默认属于核领域，请不要再概括成“核能/核电/核技术”等大类，而要总结核领域内部的细分主题。
4. title 为中文短标题，不要写“主题”“研究”等空泛词，也不要把“核”“能源”“电力”作为泛化主词重复出现。
5. summary 为 1 句话中文摘要，尽量控制在 25-50 个中文字符。
6. 必须忠实反映关键词含义，不要编造未出现的方向。

主题ID：{topic_id}
KeyBERT关键词：{keywords_text}
代表文献片段：
{rep_docs_text}
""".strip()
    return [
        {
            "role": "system",
            "name": "MiniMax AI",
            "content": "你是一名科学计量与主题建模助手，擅长将英文主题关键词压缩成准确、简洁的中文主题标签与摘要。",
        },
        {
            "role": "user",
            "name": "用户",
            "content": user_prompt,
        },
    ]


def _summarize_topic_with_minimax(
    topic_id: int,
    keywords: Sequence[str],
    rep_docs: Sequence[str],
    cache: Dict[str, Dict[str, Any]],
) -> Dict[str, str]:
    cache_key = _summary_cache_key(topic_id, keywords)
    cached = cache.get(cache_key)
    if cached:
        return {
            "title": str(cached.get("title", "")).strip(),
            "summary": str(cached.get("summary", "")).strip(),
            "status": str(cached.get("status", "cached")),
        }

    fallback = _fallback_topic_summary(topic_id, keywords)
    if not MINIMAX_API_KEY:
        result = {**fallback, "status": "no_api_key"}
        cache[cache_key] = result
        return result

    url = f"{MINIMAX_API_BASE}{MINIMAX_CHAT_ENDPOINT}"
    payload = {
        "model": MINIMAX_MODEL,
        "messages": _build_topic_summary_messages(topic_id, keywords, rep_docs),
        "temperature": 0.2,
    }
    headers = {
        "Authorization": f"Bearer {MINIMAX_API_KEY}",
        "Content-Type": "application/json",
    }

    last_error = None
    for attempt in range(1, MINIMAX_MAX_RETRIES + 1):
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=MINIMAX_TIMEOUT)
            response.raise_for_status()
            data = response.json()
            content = data["choices"][0]["message"]["content"]
            parsed = _extract_json_object(content)
            result = {
                "title": str(parsed.get("title", fallback["title"])).strip() or fallback["title"],
                "summary": str(parsed.get("summary", fallback["summary"])).strip() or fallback["summary"],
                "status": "api_generated",
            }
            cache[cache_key] = {**result, "model": MINIMAX_MODEL}
            return result
        except Exception as exc:
            last_error = exc
            if attempt < MINIMAX_MAX_RETRIES:
                time.sleep(MINIMAX_RETRY_SLEEP * attempt)

    result = {**fallback, "status": f"fallback_after_error: {type(last_error).__name__}"}
    cache[cache_key] = {**result, "model": MINIMAX_MODEL}
    return result


# ── 5.3 Export KeyBERT topic table + MiniMax summaries ─────────────────────
summary_cache = _load_topic_summary_cache()
rows = []
custom_topic_labels = {}
cache_hit_count = 0
api_generated_count = 0
fallback_count = 0

for tid in sorted(topic_info["Topic"].unique()):
    topic_id = int(tid)
    count = int(topic_info.loc[topic_info["Topic"] == tid, "Count"].values[0])

    if topic_id == -1:
        rows.append(
            {
                "topic_id": topic_id,
                "count": count,
                "name": "-1_noise",
                "top_keywords": "",
                "summary_title": "-1_noise",
                "summary_text": "层次聚类设置下理论上无噪声类；此行仅用于兼容旧结果。",
                "summary_status": "noise",
                "keywords_source": "bertopic_keybert_specter2",
                "summary_model": MINIMAX_MODEL,
            }
        )
        continue

    topic_keywords = topic_model.get_topic(topic_id) or []
    words_only = [str(word).strip() for word, _ in topic_keywords[:TOP_N_KEYWORDS] if str(word).strip()]
    keyword_str = ", ".join(words_only)

    try:
        representative_docs = topic_model.get_representative_docs(topic_id) or []
    except Exception:
        representative_docs = []

    summary_result = _summarize_topic_with_minimax(topic_id, words_only, representative_docs, summary_cache)
    if summary_result["status"] == "cached":
        cache_hit_count += 1
    elif summary_result["status"] == "api_generated":
        api_generated_count += 1
    else:
        fallback_count += 1

    custom_topic_labels[topic_id] = f"T{topic_id}: {summary_result['title']}"
    rows.append(
        {
            "topic_id": topic_id,
            "count": count,
            "name": summary_result["title"],
            "top_keywords": keyword_str,
            "summary_title": summary_result["title"],
            "summary_text": summary_result["summary"],
            "summary_status": summary_result["status"],
            "keywords_source": "bertopic_keybert_specter2",
            "summary_model": MINIMAX_MODEL,
        }
    )

_save_topic_summary_cache(summary_cache)
topic_info_export = pd.DataFrame(rows).sort_values(["topic_id"]).reset_index(drop=True)

try:
    topic_model.set_topic_labels(custom_topic_labels)
    print(f"✅ Updated BERTopic custom labels for {len(custom_topic_labels)} topics")
except Exception as exc:
    print(f"⚠️ Failed to update BERTopic custom labels: {exc}")

topic_info_export.to_csv(os.path.join(OUTPUT_DIR, "topic_info.csv"), index=False)
print(f"✅ Exported topic_info.csv  ({len(topic_info_export)} topics)")
print(
    f"MiniMax summary status → cached: {cache_hit_count}, api_generated: {api_generated_count}, fallback: {fallback_count}"
)
topic_info_export.head(10)


In [ ]:
# 【单元格中文说明】
# - get_representative_docs：读取 BERTopic 内部保留的代表文献；保存为 JSON，供人工核对与 MiniMax 总结复核。

# ── 5.3 Representative documents per topic ───────────────────────────────────
repr_docs = {}
for tid in sorted(set(topics)):
    if tid == -1:
        continue
    try:
        rd = topic_model.get_representative_docs(tid)
        repr_docs[str(tid)] = rd[:5] if rd else []
    except Exception:
        repr_docs[str(tid)] = []

# Save as JSON
repr_path = os.path.join(OUTPUT_DIR, "representative_docs.json")
with open(repr_path, "w", encoding="utf-8") as f:
    json.dump(repr_docs, f, ensure_ascii=False, indent=2)
print(f"✅ Exported representative_docs.json  ({len(repr_docs)} topics)")

# Quick preview: show top 3 repr docs of the largest topic
largest_topic = topic_info_export.loc[topic_info_export["topic_id"] != -1].sort_values("count", ascending=False).iloc[0]
print(f"\n── Largest topic: {largest_topic['topic_id']}  ({largest_topic['count']} docs) ──")
print(f"Keywords: {largest_topic['top_keywords']}")
for i, doc in enumerate(repr_docs.get(str(int(largest_topic['topic_id'])), [])[:3], 1):
    print(f"  [{i}] {doc[:200]}…")

# 6. CN vs US Technology Structure
Compute topic × country count and share matrices, then visualise the comparative distribution with a stacked bar chart.

**【中文说明】**
- 在中美子样本上构建「主题×国家」计数矩阵，并计算行归一化（主题内中美占比）与列归一化（国家内主题分布）。
- 为后续“技术结构差异”与可视化（堆叠柱状图）提供统一输入。


In [ ]:
# 【单元格中文说明】
# - 仅保留中美样本（层次聚类下通常无 topic=-1，此处仍兼容旧模型噪声类）。
# - groupby 构建 topic×country 矩阵；导出 topic_country_matrix.csv 与 topic_share_country.csv（两种归一化）。


# ── 6.1 Filter to CN & US (hierarchical clustering keeps all samples) ───────
df_cn_us = df[df["country_code"].isin(["CN", "US"])].copy()
if (df_cn_us["topic"] == -1).any():
    # compatibility path for legacy models that might still emit -1
    df_cn_us_valid = df_cn_us[df_cn_us["topic"] != -1].copy()
else:
    df_cn_us_valid = df_cn_us.copy()
print(f"CN+US papers used for topic structure: {len(df_cn_us_valid):,}")

# ── 6.2 Topic × Country count matrix ────────────────────────────────────────
topic_country_matrix = (
    df_cn_us_valid
    .groupby(["topic", "country_code"])
    .size()
    .unstack(fill_value=0)
)
# Ensure both columns exist
for col in ["CN", "US"]:
    if col not in topic_country_matrix.columns:
        topic_country_matrix[col] = 0

topic_country_matrix["total"] = topic_country_matrix["CN"] + topic_country_matrix["US"]
topic_country_matrix = topic_country_matrix.sort_values("total", ascending=False)

topic_country_matrix.to_csv(os.path.join(OUTPUT_DIR, "topic_country_matrix.csv"))
print("✅ Exported topic_country_matrix.csv")

# ── 6.3 Share matrices (two normalisations) ─────────────────────────────────
# (a) Row-normalised: within each topic, what fraction is CN vs US?
topic_share_row = topic_country_matrix[["CN", "US"]].div(
    topic_country_matrix[["CN", "US"]].sum(axis=1), axis=0
)
topic_share_row.columns = ["CN_share", "US_share"]

# (b) Column-normalised: within each country, what fraction goes to each topic?
country_totals = topic_country_matrix[["CN", "US"]].sum(axis=0)
topic_share_col = topic_country_matrix[["CN", "US"]].div(country_totals, axis=1)
topic_share_col.columns = ["CN_topic_share", "US_topic_share"]

topic_share_country = pd.concat([topic_share_row, topic_share_col], axis=1)
topic_share_country.to_csv(os.path.join(OUTPUT_DIR, "topic_share_country.csv"))
print("✅ Exported topic_share_country.csv")
print("\n── Row-normalised share (top 10 topics by size) ──")
topic_share_country.head(10)


In [ ]:
# 【单元格中文说明】
# - 取规模最大的 TOP_K_TOPICS 个主题画堆叠柱：红=中国、蓝=美国，直观对比结构重心差异。

# ── 6.4 Stacked bar chart: CN vs US topic distribution ───────────────────────
top_k = topic_country_matrix.head(TOP_K_TOPICS).copy()

# Merge topic names for labels
name_map = dict(zip(topic_info_export["topic_id"], topic_info_export["name"]))
labels = [f"T{tid}: {name_map.get(tid, '')[:30]}" for tid in top_k.index]

fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(len(labels))
width = 0.6

bars_cn = ax.bar(x, top_k["CN"], width, label="China (CN)", color="#E03C31")
bars_us = ax.bar(x, top_k["US"], width, bottom=top_k["CN"], label="United States (US)", color="#3C6FE0")

ax.set_xlabel("Topic", fontsize=12)
ax.set_ylabel("Number of Papers", fontsize=12)
ax.set_title(f"CN vs US Paper Distribution — Top {TOP_K_TOPICS} Topics", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=60, ha="right", fontsize=8)
ax.legend(fontsize=11)
plt.tight_layout()

fig_path = os.path.join(OUTPUT_DIR, "cn_us_topic_distribution.png")
fig.savefig(fig_path, dpi=200)
print(f"✅ Saved {fig_path}")
plt.show()

# 7. Gap Metrics (basic)
Quantify the CN–US technology gap using **topic coverage** and **Jensen-Shannon divergence** of topic distributions.

**【中文说明】**
- **覆盖度**：中美各自覆盖了多少个主题，是否存在仅一国出现的主题。
- **Jensen–Shannon**：将两国在主题上的论文分布看作概率分布，衡量结构差异（0 为完全一致）。


In [ ]:
# 【单元格中文说明】
# - 覆盖度：主题集合差集；JS：将两国在全主题上的论文数转为分布后计算 jensenshannon。
# - metrics.json 汇总上述指标及中美论文量，供报告引用。

# ── 7.1 Coverage ─────────────────────────────────────────────────────────────
all_topics_set = set(df_cn_us_valid["topic"].unique())
cn_topics = set(df_cn_us_valid.loc[df_cn_us_valid["country_code"] == "CN", "topic"].unique())
us_topics = set(df_cn_us_valid.loc[df_cn_us_valid["country_code"] == "US", "topic"].unique())

cn_coverage = len(cn_topics) / len(all_topics_set) if all_topics_set else 0
us_coverage = len(us_topics) / len(all_topics_set) if all_topics_set else 0
cn_only = cn_topics - us_topics
us_only = us_topics - cn_topics

print("── Topic Coverage ──")
print(f"  Total topics (CN∪US, excl noise): {len(all_topics_set)}")
print(f"  CN covers: {len(cn_topics)} topics  ({cn_coverage:.2%})")
print(f"  US covers: {len(us_topics)} topics  ({us_coverage:.2%})")
print(f"  CN-only topics: {sorted(cn_only) if cn_only else '(none)'}")
print(f"  US-only topics: {sorted(us_only) if us_only else '(none)'}")

# ── 7.2 Jensen-Shannon Divergence ───────────────────────────────────────────
# Build probability distributions over all topics for CN and US
all_topic_ids = sorted(all_topics_set)
cn_counts = topic_country_matrix.reindex(all_topic_ids)["CN"].fillna(0).values.astype(float)
us_counts = topic_country_matrix.reindex(all_topic_ids)["US"].fillna(0).values.astype(float)

# Normalise to probability distributions
p_cn = cn_counts / cn_counts.sum() if cn_counts.sum() > 0 else cn_counts
p_us = us_counts / us_counts.sum() if us_counts.sum() > 0 else us_counts

js_div = jensenshannon(p_cn, p_us)   # returns the JS *distance* (sqrt of divergence)
js_divergence = js_div ** 2           # actual divergence

print(f"\n── Structure Divergence ──")
print(f"  JS distance  (√divergence): {js_div:.4f}")
print(f"  JS divergence:              {js_divergence:.4f}")
print(f"  (0 = identical distributions; 1 = maximally different)")

# ── 7.3 Export metrics ──────────────────────────────────────────────────────
metrics = {
    "n_topics_total": int(len(all_topics_set)),
    "cn_topic_count": int(len(cn_topics)),
    "us_topic_count": int(len(us_topics)),
    "cn_coverage": round(float(cn_coverage), 4),
    "us_coverage": round(float(us_coverage), 4),
    "cn_only_topics": [int(t) for t in sorted(cn_only)],
    "us_only_topics": [int(t) for t in sorted(us_only)],
    "js_distance": round(float(js_div), 6),
    "js_divergence": round(float(js_divergence), 6),
    "cn_total_papers": int(cn_counts.sum()),
    "us_total_papers": int(us_counts.sum()),
}

metrics_path = os.path.join(OUTPUT_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\n✅ Exported {metrics_path}")


# 8. Visualizations
Generate BERTopic interactive topic map (HTML), a UMAP 2-D scatter plot coloured by topic, and the topic hierarchy.

**【中文说明】**
- 生成 BERTopic 自带的交互式主题图、UMAP 二维散点与主题层次/条形图（HTML），便于探索与汇报。
- Matplotlib 散点作为静态备选或论文插图。


In [ ]:
# 【单元格中文说明】
# - BERTopic 内置 visualize_topics 生成交互 HTML，可在浏览器中缩放、框选主题。

# ── 8.1 BERTopic interactive topic map (HTML) ────────────────────────────────
try:
    fig_topics = topic_model.visualize_topics()
    html_path = os.path.join(OUTPUT_DIR, "topic_map.html")
    fig_topics.write_html(html_path)
    print(f"✅ Saved interactive topic map → {html_path}")
except Exception as e:
    print(f"⚠️  visualize_topics() failed ({e}); will use UMAP scatter fallback.")

In [ ]:
# 【单元格中文说明】
# - 额外做二维 UMAP 静态散点图（全量 embeddings），用于论文插图或与 HTML 互证。


# ── 8.2 UMAP 2-D scatter plot (matplotlib fallback / complementary) ──────────
print("Computing 2-D UMAP projection for scatter plot…")
umap_2d = UMAP(
    n_neighbors=UMAP_N_NEIGHBORS,
    n_components=2,
    min_dist=0.3,
    metric=UMAP_METRIC,
    random_state=SEED,
)
xy = umap_2d.fit_transform(embeddings)

fig, ax = plt.subplots(figsize=(14, 10))
scatter = ax.scatter(
    xy[:, 0],
    xy[:, 1],
    c=np.array(topics),
    cmap="tab20",
    s=4,
    alpha=0.5,
)

ax.set_title("UMAP 2-D Projection — Coloured by BERTopic Topic", fontsize=14)
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
plt.colorbar(scatter, ax=ax, label="Topic ID", shrink=0.7)
plt.tight_layout()

scatter_path = os.path.join(OUTPUT_DIR, "umap_scatter.png")
fig.savefig(scatter_path, dpi=200)
print(f"✅ Saved {scatter_path}")
plt.show()


In [ ]:
# 【单元格中文说明】
# - visualize_hierarchy / visualize_barchart：主题层级与关键词条形图 HTML；失败时仅打印警告不影响主流程。

# ── 8.3 Topic hierarchy (optional) ───────────────────────────────────────────
try:
    fig_hier = topic_model.visualize_hierarchy()
    hier_path = os.path.join(OUTPUT_DIR, "topic_hierarchy.html")
    fig_hier.write_html(hier_path)
    print(f"✅ Saved topic hierarchy → {hier_path}")
except Exception as e:
    print(f"⚠️  visualize_hierarchy() skipped: {e}")

# ── 8.4 Bar chart of top topic keywords ─────────────────────────────────────
try:
    fig_bar = topic_model.visualize_barchart(top_n_topics=TOP_K_TOPICS)
    bar_path = os.path.join(OUTPUT_DIR, "topic_barchart.html")
    fig_bar.write_html(bar_path)
    print(f"✅ Saved topic barchart → {bar_path}")
except Exception as e:
    print(f"⚠️  visualize_barchart() skipped: {e}")

# 9. Topic Reduction (optional)
If the initial model produces too many fine-grained topics, use `reduce_topics()` to merge similar ones.

**How to choose `nr_topics`:**
- Start with the hierarchy visualisation (Section 8.3) and pick a cut level
- Or set `nr_topics="auto"` to let BERTopic decide
- Common range for scientometrics: 20–50 macro topics

**【中文说明】**
- 若主题过细，可调用 `reduce_topics` 合并相似主题；需手动取消注释并按需求设定目标主题数或 `auto`。
- 合并后应重新导出主题信息与下游表格。


In [ ]:
# 【单元格中文说明】
# - 主题归并示例代码，默认注释掉；需要时设置 NR_TOPICS_REDUCED 并取消注释后运行。

# ── 9.1 Reduce topics (uncomment & run when needed) ──────────────────────────
# Set the desired number of topics:
NR_TOPICS_REDUCED = 30   # change as needed; or set to "auto"

# topic_model.reduce_topics(docs, nr_topics=NR_TOPICS_REDUCED)
# reduced_topics = topic_model.topics_

# print(f"Reduced to {len(set(reduced_topics)) - (1 if -1 in reduced_topics else 0)} topics (+ noise)")

# # Update the DataFrame
# df["topic_reduced"] = reduced_topics

# # Re-export reduced topic info
# reduced_info = topic_model.get_topic_info()
# reduced_rows = []
# for tid in sorted(reduced_info["Topic"].unique()):
#     kws = topic_model.get_topic(tid)
#     kw_str = ", ".join([w for w, _ in kws[:TOP_N_KEYWORDS]]) if kws else ""
#     reduced_rows.append({
#         "topic_id": tid,
#         "count": int(reduced_info.loc[reduced_info["Topic"] == tid, "Count"].values[0]),
#         "name": reduced_info.loc[reduced_info["Topic"] == tid, "Name"].values[0],
#         "top_keywords": kw_str,
#     })
# pd.DataFrame(reduced_rows).to_csv(os.path.join(OUTPUT_DIR, "topic_info_reduced.csv"), index=False)
# print("✅ Exported topic_info_reduced.csv")

# # Visualise comparison
# try:
#     fig_reduced = topic_model.visualize_barchart(top_n_topics=NR_TOPICS_REDUCED)
#     fig_reduced.write_html(os.path.join(OUTPUT_DIR, "topic_barchart_reduced.html"))
#     print("✅ Saved reduced topic barchart")
# except Exception as e:
#     print(f"⚠️  Reduced barchart skipped: {e}")

print("ℹ️  Uncomment this cell and run it to perform topic reduction.")

# 10. Save Artifacts
Persist the BERTopic model, the per-paper topic assignment table, and print a summary of all generated files.

**【中文说明】**
- 持久化训练好的 BERTopic 模型（含 safetensors 与 c-TF-IDF），并导出 `paper_topics.csv` 供其他笔记本或软件读取。
- 对 JSON 序列化做 numpy 类型补丁，避免保存时报错。


In [ ]:
# 【单元格中文说明】
# - save：序列化模型；paper_topics.csv 保留每篇文献的主题与置信度（及年份若可用）。
# - JSONEncoder 补丁：解决 numpy 标量无法直接写入 JSON 的问题，保存后恢复原实现。

# ── 10.1 Save BERTopic model ──────────────────────────────────────────────────
# Patch json's default encoder to handle numpy scalar types (BERTopic internal bug)
import json as _json

_orig_default = _json.JSONEncoder.default

def _numpy_safe_default(self, obj):
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return _orig_default(self, obj)

_json.JSONEncoder.default = _numpy_safe_default

model_dir = os.path.join(OUTPUT_DIR, "bertopic_model")
try:
    topic_model.save(model_dir, serialization="safetensors", save_ctfidf=False, save_embedding_model=SPECTER2_BASE_MODEL)
    print(f"✅ BERTopic model saved → {model_dir}")
finally:
    _json.JSONEncoder.default = _orig_default   # restore

# ── 10.2 Export paper_topics.csv ─────────────────────────────────────────────
export_cols = ["paper_id", "country", "country_code", "topic", "topic_prob"]
if HAS_YEAR:
    export_cols.insert(2, "year")

paper_topics = df[export_cols].copy()
paper_topics_path = os.path.join(OUTPUT_DIR, "paper_topics.csv")
paper_topics.to_csv(paper_topics_path, index=False)
print(f"✅ Exported paper_topics.csv  ({len(paper_topics):,} rows)")

# ── 10.3 Summary of all output files ────────────────────────────────────────
print(f"\n{'═'*60}")
print(f"  All outputs in: {os.path.abspath(OUTPUT_DIR)}")
print(f"{'═'*60}")
for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(OUTPUT_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = "  " * (level + 1)
    for f in sorted(files):
        fpath = os.path.join(root, f)
        size_mb = os.path.getsize(fpath) / 1024 / 1024
        print(f"{sub_indent}{f}  ({size_mb:.2f} MB)")

print(f"\n🎉 Pipeline complete!")


# 11. Capability Gap Analysis — Config & Variable Mapping
Define all parameters for the **Frontier gap** and **Impact gap** analyses.  
Auto-detect the best topic column; standardise country labels to `{CN, US, OTHER}`; create a dedicated output sub-directory.

**【中文说明】**
- **能力缺口分析**的前置配置：自动选择 `topic`/`topic_reduced` 等列；统一 `country2`；设定前沿窗口、分桶最小样本量等。
- 输出子目录 `capability_gap/` 存放归一化引用、前沿集合与各类缺口指标。


In [ ]:
# 【单元格中文说明】
# - 自动选择最终主题列；定义引用列、年份列、国家列与前沿/分桶参数；建立 country2 与 CAP_DIR。

# ══════════════════════════════════════════════════════════════════════════════
# Cell A — Config & Variable Mapping
# ══════════════════════════════════════════════════════════════════════════════

# ── Auto-detect the best topic column ────────────────────────────────────────
_topic_candidates = ["topic_reduced", "topic_reassign", "topic"]
TOPIC_COL = None
for _c in _topic_candidates:
    if _c in df.columns and df[_c].notna().any():
        TOPIC_COL = _c
        break
if TOPIC_COL is None:
    raise ValueError("No topic column found in df. Run BERTopic clustering first.")
print(f"✅ TOPIC_COL = '{TOPIC_COL}'")

# ── Column aliases ───────────────────────────────────────────────────────────
CIT_COL     = "citation"
YEAR_COL    = "year"
COUNTRY_COL = "country_code"   # populated in Section 1

# ── Analysis parameters ──────────────────────────────────────────────────────
WINDOW_YEARS         = 5      # recent-year window for Frontier_B
EXCLUDE_RECENT_YEARS = 1      # drop most recent N year(s) — citation window too short
MIN_BUCKET_SIZE      = 20     # min papers in (topic, year) bucket to compute q90
MIN_FRONTIER_US      = 20     # min US-frontier papers per topic for centroid calc
TOPK_REPORT          = 20     # top-K topics shown in charts / reports

# ── Country label lists (for safety, re-normalise) ──────────────────────────
COUNTRY_US = ["United States", "US", "USA", "United States of America", "united states"]
COUNTRY_CN = ["China", "CN", "PRC", "Peoples R China", "china", "CHINA"]

def _map_country2(c):
    """Map raw country / country_code to {CN, US, OTHER}."""
    if c in COUNTRY_CN or c == "CN":
        return "CN"
    if c in COUNTRY_US or c == "US":
        return "US"
    return "OTHER"

df["country2"] = df[COUNTRY_COL].apply(_map_country2)

# ── Output directory for capability-gap analysis ────────────────────────────
CAP_DIR = os.path.join(OUTPUT_DIR, "capability_gap")
os.makedirs(CAP_DIR, exist_ok=True)

# ── Report ───────────────────────────────────────────────────────────────────
print(f"TOPIC_COL            = {TOPIC_COL}")
print(f"CIT_COL              = {CIT_COL}")
print(f"YEAR_COL             = {YEAR_COL}")
print(f"COUNTRY_COL          = {COUNTRY_COL}")
print(f"WINDOW_YEARS         = {WINDOW_YEARS}")
print(f"EXCLUDE_RECENT_YEARS = {EXCLUDE_RECENT_YEARS}")
print(f"MIN_BUCKET_SIZE      = {MIN_BUCKET_SIZE}")
print(f"MIN_FRONTIER_US      = {MIN_FRONTIER_US}")
print(f"TOPK_REPORT          = {TOPK_REPORT}")
print(f"CAP_DIR              = {CAP_DIR}")
print(f"\ncountry2 distribution:")
print(df["country2"].value_counts().to_string())

## 11.1 Data Checks
Verify paper counts by country, year distribution, citation missingness, and topic structure statistics before proceeding.

**【中文说明】**
- 在进入引用与前沿计算前，核对中美样本量、年份与被引字段缺失、主题数与 outlier 等，避免后续静默偏差。


In [ ]:
# 【单元格中文说明】
# - 打印描述性统计：分国样本量、年份分布、被引缺失率、主题数与嵌入形状，作为能力缺口分析的前置质检。


# ══════════════════════════════════════════════════════════════════════════════
# Cell B — Data Checks
# ══════════════════════════════════════════════════════════════════════════════

print(f"Total papers (N):  {len(df):,}")
print(f"CN papers:         {(df['country2']=='CN').sum():,}")
print(f"US papers:         {(df['country2']=='US').sum():,}")
print(f"OTHER papers:      {(df['country2']=='OTHER').sum():,}")

print(f"\n── Year distribution ──")
if YEAR_COL in df.columns:
    print(df[YEAR_COL].value_counts().sort_index().to_string())
else:
    print("  (no year column)")

print(f"\n── Citation column: {CIT_COL} ──")
cit_missing = df[CIT_COL].isna().sum()
print(f"  Missing: {cit_missing:,} ({cit_missing/len(df):.2%})")
print(f"  Mean:    {df[CIT_COL].mean():.2f}")
print(f"  Median:  {df[CIT_COL].median():.1f}")

print(f"\n── Topic column: {TOPIC_COL} ──")
n_minus1 = int((df[TOPIC_COL] == -1).sum())
n_topics_data = int(df[df[TOPIC_COL].notna()][TOPIC_COL].nunique())
print(f"  Number of topics:  {n_topics_data:,}")
print(f"  Documents with topic=-1 (if any): {n_minus1:,}")
print(f"  Outlier ratio (hierarchical): 0.00%")
print(f"  Embeddings shape:   {embeddings.shape}")


## 11.2 Normalised Citations (MNCS-style)
For each paper $i$ in bucket $(topic, year)$:

$$nc_i = \frac{c_i + 1}{E[c \mid topic, year] + 1}$$

The $+1$ smoothing avoids zero-division and dampens low-citation noise.

**【中文说明】**
- **MNCS 风格归一化引用**：在 (主题, 年份) 桶内用期望被引平滑，得到可比强度指标 `nc`，削弱发表年份与主题热度差异。


In [ ]:
# 【单元格中文说明】
# - 在 (topic,year) 桶内计算期望被引，构造 nc；导出 paper_norm_citations.csv。


# ══════════════════════════════════════════════════════════════════════════════
# Cell C — Build Normalised Citations (MNCS-style)
# nc_i = (c_i + 1) / (E[c | topic, year] + 1)
# ══════════════════════════════════════════════════════════════════════════════

# ── Work on valid-topic papers with valid citations & years ──────────────────
_mask_valid = (
    df[TOPIC_COL].notna()
    & df[CIT_COL].notna()
    & df[YEAR_COL].notna()
)
# Legacy compatibility: if topic=-1 exists, keep excluding it from citation buckets.
if (df[TOPIC_COL] == -1).any():
    _mask_valid = _mask_valid & (df[TOPIC_COL] != -1)

df_valid = df[_mask_valid].copy()
print(f"Papers with valid topic + citation + year: {len(df_valid):,}")

# ── Bucket key: (topic, year) ────────────────────────────────────────────────
df_valid["_topic"] = df_valid[TOPIC_COL].astype(int)
df_valid["_year"] = df_valid[YEAR_COL].astype(int)
df_valid["bucket_key"] = list(zip(df_valid["_topic"], df_valid["_year"]))

# ── Expected citations per bucket ────────────────────────────────────────────
bucket_stats = (
    df_valid
    .groupby("bucket_key")[CIT_COL]
    .agg(["mean", "median", "count"])
    .rename(columns={"mean": "expected_mean", "median": "expected_median", "count": "bucket_size"})
)
df_valid = df_valid.join(bucket_stats, on="bucket_key")

# ── Normalised citation ──────────────────────────────────────────────────────
df_valid["nc"] = (df_valid[CIT_COL] + 1) / (df_valid["expected_mean"] + 1)

print(f"\n── Bucket statistics ──")
print(f"  Total buckets (topic, year):   {len(bucket_stats):,}")
print(f"  Small buckets (< {MIN_BUCKET_SIZE}):  {(bucket_stats['bucket_size'] < MIN_BUCKET_SIZE).sum():,}")
print(f"  nc — mean: {df_valid['nc'].mean():.3f},  median: {df_valid['nc'].median():.3f}")

# ── Export ────────────────────────────────────────────────────────────────────
export_nc = df_valid[
    ["paper_id", "country2", YEAR_COL, TOPIC_COL, CIT_COL, "nc", "expected_mean", "expected_median", "bucket_size"]
].copy()
export_nc.to_csv(os.path.join(CAP_DIR, "paper_norm_citations.csv"), index=False)
print(f"✅ Exported paper_norm_citations.csv  ({len(export_nc):,} rows)")


## 11.3 Top-10 % Flag (PP(top10 %)-style)
Within each $(topic, year)$ bucket, compute the 90th-percentile citation threshold $q_{90}$.  
A paper is **top-10 %** if $c_i \geq q_{90}$. Buckets with fewer than `MIN_BUCKET_SIZE` papers are skipped.

**【中文说明】**
- 在每个足够大的 (主题, 年份) 桶内定义 **前 10% 高被引** 阈值 `q90`，构造二值 `top10_flag`，近似顶刊常用的顶尖论文占比分析。


In [ ]:
# 【单元格中文说明】
# - 仅对足够大桶估计 q90；生成 top10_flag；分国输出顶尖论文占比；导出 paper_top10_flag.csv。

# ══════════════════════════════════════════════════════════════════════════════
# Cell D — Top-10 % Flag per Bucket
# ══════════════════════════════════════════════════════════════════════════════

# ── q90 threshold per bucket (only large-enough buckets) ─────────────────────
bucket_q90 = (
    df_valid[df_valid["bucket_size"] >= MIN_BUCKET_SIZE]
    .groupby("bucket_key")[CIT_COL]
    .quantile(0.9)
    .rename("q90")
)

# Track skipped buckets
skipped_buckets = set(df_valid["bucket_key"].unique()) - set(bucket_q90.index)
n_skipped_papers = df_valid[df_valid["bucket_key"].isin(skipped_buckets)].shape[0]
print(f"Skipped small buckets: {len(skipped_buckets):,}  "
      f"({n_skipped_papers:,} papers affected)")

# ── Merge q90 & assign flag ─────────────────────────────────────────────────
df_valid = df_valid.join(bucket_q90, on="bucket_key")
df_valid["top10_flag"] = False
_has_q90 = df_valid["q90"].notna()
df_valid.loc[_has_q90, "top10_flag"] = (
    df_valid.loc[_has_q90, CIT_COL] >= df_valid.loc[_has_q90, "q90"]
)

# ── Summary ──────────────────────────────────────────────────────────────────
print(f"\n── Top-10 % Summary (eligible buckets only) ──")
_eligible = df_valid[_has_q90]
print(f"  Eligible papers:  {len(_eligible):,}")
print(f"  Top-10 % papers:  {_eligible['top10_flag'].sum():,}  "
      f"({_eligible['top10_flag'].mean():.2%})")
for c in ["CN", "US"]:
    _sub = _eligible[_eligible["country2"] == c]
    if len(_sub) > 0:
        pct = _sub["top10_flag"].mean()
        print(f"  {c} top-10 %:  {_sub['top10_flag'].sum():,} / {len(_sub):,}  = {pct:.2%}")

# ── Export ────────────────────────────────────────────────────────────────────
export_t10 = df_valid[["paper_id", "country2", YEAR_COL, TOPIC_COL, CIT_COL,
                        "nc", "q90", "top10_flag", "bucket_size"]].copy()
export_t10.to_csv(os.path.join(CAP_DIR, "paper_top10_flag.csv"), index=False)
print(f"\n✅ Exported paper_top10_flag.csv  ({len(export_t10):,} rows)")

## 11.4 Define Frontier Sets
Two definitions:

| Frontier | Scope |
|----------|-------|
| **A** (all-period) | Every top-10 % paper up to `max_year − EXCLUDE_RECENT_YEARS` |
| **B** (recent, **default**) | Top-10 % papers within a `WINDOW_YEARS`-year sliding window ending at `max_year − EXCLUDE_RECENT_YEARS` |

**【中文说明】**
- **前沿 A**：截至排除近年后的所有 top10%；**前沿 B（默认）**：最近 `WINDOW_YEARS` 年窗口内的 top10%，更贴近“当前前沿”。
- 排除最近 `EXCLUDE_RECENT_YEARS` 年可减少引用尚未稳定的偏差。


In [ ]:
# 【单元格中文说明】
# - 基于 top10_flag 与时间窗口生成 frontier_A / frontier_B；导出 frontier_papers.csv 并标记仅 A、仅 B 或同时属于两者。

# ══════════════════════════════════════════════════════════════════════════════
# Cell E — Define Frontier Sets (A: all-period, B: recent window)
# ══════════════════════════════════════════════════════════════════════════════

max_year = int(df_valid[YEAR_COL].max())
frontier_year_end   = max_year - EXCLUDE_RECENT_YEARS
frontier_year_start = frontier_year_end - WINDOW_YEARS + 1

print(f"max year in data:          {max_year}")
print(f"EXCLUDE_RECENT_YEARS:      {EXCLUDE_RECENT_YEARS}")
print(f"Frontier_B window:         {frontier_year_start} – {frontier_year_end}")
print(f"Frontier_A window:         all years  ≤ {frontier_year_end}")

# ── Frontier_A: all-period top10 % (up to frontier_year_end) ─────────────────
mask_A = (df_valid["top10_flag"]) & (df_valid["_year"] <= frontier_year_end)
df_valid["frontier_A"] = mask_A

# ── Frontier_B: recent-window top10 % (DEFAULT) ─────────────────────────────
mask_B = (
    (df_valid["top10_flag"])
    & (df_valid["_year"] >= frontier_year_start)
    & (df_valid["_year"] <= frontier_year_end)
)
df_valid["frontier_B"] = mask_B

print(f"\nFrontier_A papers: {mask_A.sum():,}")
print(f"Frontier_B papers: {mask_B.sum():,}")
for c in ["CN", "US"]:
    _fa = ((df_valid["country2"] == c) & df_valid["frontier_A"]).sum()
    _fb = ((df_valid["country2"] == c) & df_valid["frontier_B"]).sum()
    print(f"  {c}  Frontier_A={_fa:,}   Frontier_B={_fb:,}")

# ── Export ────────────────────────────────────────────────────────────────────
frontier_cols = ["paper_id", "country2", YEAR_COL, TOPIC_COL, CIT_COL,
                 "nc", "top10_flag", "frontier_A", "frontier_B"]
export_fr = df_valid[df_valid["frontier_A"] | df_valid["frontier_B"]][frontier_cols].copy()
export_fr["frontier_type"] = "none"
export_fr.loc[
    df_valid.loc[export_fr.index, "frontier_A"] & ~df_valid.loc[export_fr.index, "frontier_B"],
    "frontier_type"
] = "A_only"
export_fr.loc[
    ~df_valid.loc[export_fr.index, "frontier_A"] & df_valid.loc[export_fr.index, "frontier_B"],
    "frontier_type"
] = "B_only"
export_fr.loc[
    df_valid.loc[export_fr.index, "frontier_A"] & df_valid.loc[export_fr.index, "frontier_B"],
    "frontier_type"
] = "A_and_B"
export_fr.to_csv(os.path.join(CAP_DIR, "frontier_papers.csv"), index=False)
print(f"\n✅ Exported frontier_papers.csv  ({len(export_fr):,} rows)")

## 11.5 Frontier Topic-Distribution Gap
Compare the topic distributions of **US frontier** vs **CN frontier** papers using:
- Jensen–Shannon distance $JS_d = \sqrt{JSD}$ and divergence $JSD = JS_d^2$
- Per-topic delta $\Delta_F(t) = P_{USF}(t) - P_{CNF}(t)$

**【中文说明】**
- 在中美各自的前沿论文集合上比较 **主题分布**；报告 JS 距离/散度与逐主题差异 Δ（美国前沿占比 − 中国前沿占比）。
- 用于刻画“前沿活动”在主题空间上的结构性偏移。


In [ ]:
# 【单元格中文说明】
# - 在中美前沿子集上估计主题分布 P_USF、P_CNF，计算 JS 与逐主题 delta_F；保存 CSV/JSON 与柱状图。


# ══════════════════════════════════════════════════════════════════════════════
# Cell F — Frontier Topic-Distribution Gap (uses Frontier_B by default)
# ══════════════════════════════════════════════════════════════════════════════
FRONTIER_COL = "frontier_B"   # switch to "frontier_A" for all-period

# ── Filter: CN & US frontier papers ─────────────────────────────────────────
df_fr = df_valid[df_valid[FRONTIER_COL]].copy()
if (df_fr[TOPIC_COL] == -1).any():
    df_fr = df_fr[df_fr[TOPIC_COL] != -1].copy()
df_fr_us = df_fr[df_fr["country2"] == "US"]
df_fr_cn = df_fr[df_fr["country2"] == "CN"]
print(f"US frontier papers: {len(df_fr_us):,}")
print(f"CN frontier papers: {len(df_fr_cn):,}")

# ── Topic distributions ──────────────────────────────────────────────────────
all_valid_topics = sorted(df_valid[df_valid[TOPIC_COL].notna()][TOPIC_COL].astype(int).unique())
if -1 in all_valid_topics:
    all_valid_topics = [t for t in all_valid_topics if t != -1]
us_fr_counts = df_fr_us[TOPIC_COL].value_counts().reindex(all_valid_topics, fill_value=0)
cn_fr_counts = df_fr_cn[TOPIC_COL].value_counts().reindex(all_valid_topics, fill_value=0)

p_usf = (us_fr_counts / us_fr_counts.sum()).values.astype(float)
p_cnf = (cn_fr_counts / cn_fr_counts.sum()).values.astype(float)

# ── JS distance & divergence (scipy jensenshannon returns √JSD) ─────────────
from scipy.spatial.distance import jensenshannon as _jsd
js_dist_frontier = float(_jsd(p_usf, p_cnf))
jsd_frontier = js_dist_frontier ** 2
print(f"\nFrontier JS distance  (√JSD): {js_dist_frontier:.4f}")
print(f"Frontier JS divergence (JSD): {jsd_frontier:.4f}")

# ── Topic-wise delta ─────────────────────────────────────────────────────────
frontier_topic_dist = pd.DataFrame({
    "topic": all_valid_topics,
    "P_USF": p_usf,
    "P_CNF": p_cnf,
    "delta_F": p_usf - p_cnf,   # positive → US frontier larger share
})
# Merge topic names
_name_map = dict(zip(topic_info_export["topic_id"], topic_info_export["name"]))
frontier_topic_dist["topic_name"] = frontier_topic_dist["topic"].map(_name_map)
frontier_topic_dist = frontier_topic_dist.sort_values("delta_F", key=abs, ascending=False)
frontier_topic_dist.to_csv(os.path.join(CAP_DIR, "frontier_topic_distribution.csv"), index=False)
print("✅ Exported frontier_topic_distribution.csv")

# ── Metrics JSON ─────────────────────────────────────────────────────────────
js_metrics = {
    "frontier_type": FRONTIER_COL,
    "frontier_window": f"{frontier_year_start}-{frontier_year_end}",
    "us_frontier_papers": int(len(df_fr_us)),
    "cn_frontier_papers": int(len(df_fr_cn)),
    "js_distance_frontier": round(js_dist_frontier, 6),
    "jsd_frontier": round(jsd_frontier, 6),
}
with open(os.path.join(CAP_DIR, "frontier_js_metrics.json"), "w") as f:
    json.dump(js_metrics, f, indent=2)
print("✅ Exported frontier_js_metrics.json")

# ── Bar chart: US vs CN frontier topic distribution (top K by |Δ|) ───────────
_top = frontier_topic_dist.head(TOPK_REPORT).copy()
_labels = [f"T{int(t)}: {str(n)[:25]}" for t, n in zip(_top["topic"], _top["topic_name"])]

fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(len(_labels))
w = 0.35
ax.bar(x - w / 2, _top["P_USF"], w, label="US Frontier", color="#3C6FE0")
ax.bar(x + w / 2, _top["P_CNF"], w, label="CN Frontier", color="#E03C31")
ax.set_xticks(x)
ax.set_xticklabels(_labels, rotation=60, ha="right", fontsize=8)
ax.set_ylabel("Topic Share in Frontier", fontsize=12)
ax.set_title(f"Frontier Topic Distribution — US vs CN (top {TOPK_REPORT} by |Δ|)", fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
_path = os.path.join(CAP_DIR, "frontier_topic_distribution.png")
fig.savefig(_path, dpi=200)
print(f"✅ Saved {_path}")
plt.show()


## 11.6 Frontier Semantic Distance Gap (Centre-based)
For each topic $t$ with enough US frontier papers:

1. Compute the US-frontier centroid $\mu_{USF,t}$ (mean of L2-normed SPECTER2 embeddings, then re-normalise).
2. $D_{CN}(t) = \text{mean}_{i \in CN,t}\bigl(1 - \cos(e_i, \mu_{USF,t})\bigr)$
3. $D_{US}(t) = \text{mean}_{i \in US,t}\bigl(1 - \cos(e_i, \mu_{USF,t})\bigr)$
4. $\text{Gap}(t) = D_{CN}(t) - D_{US}(t)$   *(positive → CN farther from frontier)*

**【中文说明】**
- 以美国前沿质心为参照，在原始嵌入空间计算中国与全美样本相对该质心的语义距离，并得到 per-topic 的 **Gap = D_CN − D_US**。
- 质心基于 L2 归一化向量，几何上兼容余弦相似度；样本不足的主题会跳过并记录原因。


In [ ]:
# 【单元格中文说明】
# - 对美国前沿质心（按主题）与中美全体样本做余弦距离 1−cos；得 gap_frontier_semantic；样本不足主题跳过。

# ══════════════════════════════════════════════════════════════════════════════
# Cell G — Frontier Semantic Distance Gap (Centre-based, per topic)
# ══════════════════════════════════════════════════════════════════════════════

# Embeddings are already L2-normed (Section 2) — verify / re-norm for safety
# emb_norm = normalize(embeddings, norm="l2")

# FRONTIER_COL_SEM = "frontier_B"

# results_sem = []
# skipped_topics_sem = []

# for t in sorted(df_valid[df_valid[TOPIC_COL] != -1][TOPIC_COL].unique()):
#     # US frontier in topic t
#     mask_us_fr = (
#         (df_valid["country2"] == "US")
#         & (df_valid[TOPIC_COL] == t)
#         & (df_valid[FRONTIER_COL_SEM])
#     )
#     idx_us_fr = df_valid[mask_us_fr].index.values

#     if len(idx_us_fr) < MIN_FRONTIER_US:
#         skipped_topics_sem.append(t)
#         results_sem.append({
#             "topic": int(t),
#             "n_us_frontier": int(len(idx_us_fr)),
#             "n_cn_topic": 0, "n_us_topic": 0,
#             "D_CN": np.nan, "D_US": np.nan,
#             "gap_frontier_semantic": np.nan,
#             "skipped": True,
#             "reason": f"US frontier < {MIN_FRONTIER_US}",
#         })
#         continue

#     # ── Centroid of US frontier (L2 re-normalise the mean) ───────────────
#     centroid_usf = emb_norm[idx_us_fr].mean(axis=0)
#     centroid_usf /= np.linalg.norm(centroid_usf) + 1e-10

#     # CN papers in topic t (all papers, not just frontier)
#     mask_cn_t = (df_valid["country2"] == "CN") & (df_valid[TOPIC_COL] == t)
#     idx_cn = df_valid[mask_cn_t].index.values

#     # US papers in topic t
#     mask_us_t = (df_valid["country2"] == "US") & (df_valid[TOPIC_COL] == t)
#     idx_us = df_valid[mask_us_t].index.values

#     # Distance = 1 − cosine_similarity
#     D_CN = float(1 - (emb_norm[idx_cn] @ centroid_usf).mean()) if len(idx_cn) > 0 else np.nan
#     D_US = float(1 - (emb_norm[idx_us] @ centroid_usf).mean()) if len(idx_us) > 0 else np.nan
#     gap  = (D_CN - D_US) if not (np.isnan(D_CN) or np.isnan(D_US)) else np.nan

#     results_sem.append({
#         "topic": int(t),
#         "n_us_frontier": int(len(idx_us_fr)),
#         "n_cn_topic": int(len(idx_cn)),
#         "n_us_topic": int(len(idx_us)),
#         "D_CN": round(D_CN, 6) if not np.isnan(D_CN) else np.nan,
#         "D_US": round(D_US, 6) if not np.isnan(D_US) else np.nan,
#         "gap_frontier_semantic": round(gap, 6) if not (isinstance(gap, float) and np.isnan(gap)) else np.nan,
#         "skipped": False,
#     })

# semantic_gap_df = pd.DataFrame(results_sem)
# semantic_gap_df["topic_name"] = semantic_gap_df["topic"].map(_name_map)

# print(f"Topics computed: {(~semantic_gap_df['skipped']).sum()}")
# print(f"Topics skipped:  {semantic_gap_df['skipped'].sum()}")
# if skipped_topics_sem:
#     print(f"  Skipped IDs: {skipped_topics_sem[:20]}{'…' if len(skipped_topics_sem) > 20 else ''}")

# semantic_gap_df.to_csv(
#     os.path.join(CAP_DIR, "frontier_semantic_gap_by_topic.csv"), index=False)
# print(f"\n✅ Exported frontier_semantic_gap_by_topic.csv")

# # ── Bar chart: Frontier semantic gap (top K topics) ──────────────────────────
# _plot = (
#     semantic_gap_df[~semantic_gap_df["skipped"]]
#     .dropna(subset=["gap_frontier_semantic"])
#     .sort_values("gap_frontier_semantic", key=abs, ascending=False)
#     .head(TOPK_REPORT)
# )
# _labels = [f"T{int(t)}: {str(n)[:25]}" for t, n in zip(_plot["topic"], _plot["topic_name"])]
# colors = ["#E03C31" if g > 0 else "#3C6FE0" for g in _plot["gap_frontier_semantic"]]

# fig, ax = plt.subplots(figsize=(14, 7))
# ax.barh(range(len(_labels)), _plot["gap_frontier_semantic"].values, color=colors)
# ax.set_yticks(range(len(_labels)))
# ax.set_yticklabels(_labels, fontsize=9)
# ax.set_xlabel("Frontier Semantic Gap  (D_CN − D_US)", fontsize=12)
# ax.set_title(
#     f"Frontier Semantic Distance Gap by Topic (top {TOPK_REPORT})\n"
#     "(red / positive = CN farther from US frontier)",
#     fontsize=13,
# )
# ax.axvline(0, color="black", linewidth=0.8)
# plt.tight_layout()
# _path = os.path.join(CAP_DIR, "frontier_semantic_gap_by_topic.png")
# fig.savefig(_path, dpi=200)
# print(f"✅ Saved {_path}")
# plt.show()

## 11.7 Impact Gap by Topic
Two complementary indicators per topic $t$:

| Indicator | Formula | Interpretation |
|-----------|---------|----------------|
| **MNCS** | $\overline{nc}_{country,t}$ | Mean normalised citations — overall impact |
| **PP(top 10 %)** | share of top-10 % papers | Excellence / breakthrough capacity |

$\text{Gap}(t) = \text{US}(t) - \text{CN}(t)$  *(positive → US leads)*

**【中文说明】**
- **影响缺口**：在每主题上比较 MNCS（归一化引用均值）与 PP(top10%)（顶尖论文比例），缺口定义为 US − CN。
- 与结构类指标互补：一个偏“整体影响力”，一个偏“顶尖突破比例”。


In [ ]:
# 【单元格中文说明】
# - 按主题聚合 MNCS 与 PP(top10%)，形成 US−CN 的缺口并绘制横向条形图。

# ══════════════════════════════════════════════════════════════════════════════
# Cell H — Impact Gap by Topic (MNCS + PP(top10%))
# ══════════════════════════════════════════════════════════════════════════════

# ── Filter to CN & US papers with valid topic ────────────────────────────────
_cn_us = df_valid[
    df_valid["country2"].isin(["CN", "US"]) & (df_valid[TOPIC_COL] != -1)
].copy()

# ── MNCS by (topic, country) ────────────────────────────────────────────────
mncs_by_ct = (
    _cn_us
    .groupby([TOPIC_COL, "country2"])["nc"]
    .mean()
    .unstack(fill_value=np.nan)
)
mncs_by_ct.columns = [f"MNCS_{c}" for c in mncs_by_ct.columns]
for _col in ["MNCS_CN", "MNCS_US"]:
    if _col not in mncs_by_ct.columns:
        mncs_by_ct[_col] = np.nan
mncs_by_ct["gap_MNCS"] = mncs_by_ct["MNCS_US"] - mncs_by_ct["MNCS_CN"]

# ── PP(top10%) by (topic, country) ──────────────────────────────────────────
pp10_by_ct = (
    _cn_us[_cn_us["q90"].notna()]          # only eligible buckets
    .groupby([TOPIC_COL, "country2"])["top10_flag"]
    .mean()
    .unstack(fill_value=np.nan)
)
pp10_by_ct.columns = [f"PP10_{c}" for c in pp10_by_ct.columns]
for _col in ["PP10_CN", "PP10_US"]:
    if _col not in pp10_by_ct.columns:
        pp10_by_ct[_col] = np.nan
pp10_by_ct["gap_PP10"] = pp10_by_ct["PP10_US"] - pp10_by_ct["PP10_CN"]

# ── Merge & export ───────────────────────────────────────────────────────────
impact_gap = mncs_by_ct.join(pp10_by_ct, how="outer")
impact_gap.index.name = "topic"
impact_gap["topic_name"] = impact_gap.index.map(_name_map)
impact_gap.to_csv(os.path.join(CAP_DIR, "topic_impact_gap.csv"))
print(f"✅ Exported topic_impact_gap.csv  ({len(impact_gap)} topics)")

# ══════════════════════════════════════════════════════════════════════════════
# Chart 1 — MNCS gap (top K topics by |gap|)
# ══════════════════════════════════════════════════════════════════════════════
_p1 = (impact_gap.dropna(subset=["gap_MNCS"])
       .sort_values("gap_MNCS", key=abs, ascending=False)
       .head(TOPK_REPORT))
_labels1 = [f"T{int(t)}: {str(n)[:25]}" for t, n in zip(_p1.index, _p1["topic_name"])]
colors1 = ["#3C6FE0" if g > 0 else "#E03C31" for g in _p1["gap_MNCS"]]

fig, ax = plt.subplots(figsize=(14, 7))
ax.barh(range(len(_labels1)), _p1["gap_MNCS"].values, color=colors1)
ax.set_yticks(range(len(_labels1)))
ax.set_yticklabels(_labels1, fontsize=9)
ax.set_xlabel("MNCS Gap  (US − CN)", fontsize=12)
ax.set_title(
    f"Impact Gap: MNCS by Topic (top {TOPK_REPORT} by |gap|)\n"
    "(blue = US leads, red = CN leads)", fontsize=13)
ax.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
fig.savefig(os.path.join(CAP_DIR, "impact_gap_mncs.png"), dpi=200)
print(f"✅ Saved impact_gap_mncs.png")
plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# Chart 2 — PP(top10%) gap (top K topics by |gap|)
# ══════════════════════════════════════════════════════════════════════════════
_p2 = (impact_gap.dropna(subset=["gap_PP10"])
       .sort_values("gap_PP10", key=abs, ascending=False)
       .head(TOPK_REPORT))
_labels2 = [f"T{int(t)}: {str(n)[:25]}" for t, n in zip(_p2.index, _p2["topic_name"])]
colors2 = ["#3C6FE0" if g > 0 else "#E03C31" for g in _p2["gap_PP10"]]

fig, ax = plt.subplots(figsize=(14, 7))
ax.barh(range(len(_labels2)), _p2["gap_PP10"].values, color=colors2)
ax.set_yticks(range(len(_labels2)))
ax.set_yticklabels(_labels2, fontsize=9)
ax.set_xlabel("PP(top10%) Gap  (US − CN)", fontsize=12)
ax.set_title(
    f"Impact Gap: PP(top10%) by Topic (top {TOPK_REPORT} by |gap|)\n"
    "(blue = US leads, red = CN leads)", fontsize=13)
ax.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
fig.savefig(os.path.join(CAP_DIR, "impact_gap_pp10.png"), dpi=200)
print(f"✅ Saved impact_gap_pp10.png")
plt.show()

## 11.8 Quantity–Quality Quadrant
Scatter plot of every topic $t$ in a four-quadrant space:

- **x-axis (Quantity):** Topic-share gap $P_{CN}(t) - P_{US}(t)$  — positive means CN publishes a larger share
- **y-axis (Quality):** Impact gap $MNCS_{CN}(t) - MNCS_{US}(t)$  — positive means CN has higher normalised impact

Quadrants: upper-right = CN leads both; lower-left = US leads both.

**【中文说明】**
- **四象限图**：横轴为两国在该主题上发文占比之差（量），纵轴为 MNCS 之差（质），用于同时观察“做大”与“做强”是否一致。


In [ ]:
# 【单元格中文说明】
# - 横轴：两国总发文中的主题占比差；纵轴：MNCS 差（注意与上一节 US−CN 定义通过取负对齐）。
# - 标注离原点较远的主题以突出“量质错位”象限。

# ══════════════════════════════════════════════════════════════════════════════
# Cell I — Quantity–Quality Quadrant
# ══════════════════════════════════════════════════════════════════════════════

# ── Quantity axis: topic share gap (CN − US) using ALL CN/US papers ──────────
_all_cn_us_valid = df_valid[
    df_valid["country2"].isin(["CN", "US"]) & (df_valid[TOPIC_COL] != -1)
]
cn_share = (
    _all_cn_us_valid[_all_cn_us_valid["country2"] == "CN"][TOPIC_COL]
    .value_counts(normalize=True)
)
us_share = (
    _all_cn_us_valid[_all_cn_us_valid["country2"] == "US"][TOPIC_COL]
    .value_counts(normalize=True)
)
_all_t = sorted(_all_cn_us_valid[TOPIC_COL].unique())
cn_share = cn_share.reindex(_all_t, fill_value=0)
us_share = us_share.reindex(_all_t, fill_value=0)
share_gap = cn_share - us_share      # positive → CN has larger share

# ── Quality axis: MNCS gap (CN − US), reversed from Cell H ──────────────────
quality_mncs = -(impact_gap["gap_MNCS"])  # Cell H: gap_MNCS = US − CN → negate
quality_pp10 = -(impact_gap["gap_PP10"])

# ── Build quadrant table ────────────────────────────────────────────────────
quadrant_df = pd.DataFrame({
    "topic": _all_t,
    "share_gap_CN_minus_US": share_gap.values,
    "mncs_gap_CN_minus_US":  quality_mncs.reindex(_all_t).values,
    "pp10_gap_CN_minus_US":  quality_pp10.reindex(_all_t).values,
})
quadrant_df["topic_name"] = quadrant_df["topic"].map(_name_map)
quadrant_df.to_csv(os.path.join(CAP_DIR, "topic_quadrant.csv"), index=False)
print(f"✅ Exported topic_quadrant.csv  ({len(quadrant_df)} topics)")

# ── Scatter: Quantity (x) vs Quality-MNCS (y) ───────────────────────────────
fig, ax = plt.subplots(figsize=(10, 10))
x = quadrant_df["share_gap_CN_minus_US"].values
y = quadrant_df["mncs_gap_CN_minus_US"].values

ax.scatter(x, y, s=40, alpha=0.7, c="#555555", edgecolors="white", linewidths=0.5)
ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")
ax.axvline(0, color="grey", linewidth=0.8, linestyle="--")

# Annotate outliers (top 8 by Euclidean distance from origin)
_d = np.sqrt(x ** 2 + y ** 2)
_idx_out = np.argsort(_d)[-8:]
for _i in _idx_out:
    _row = quadrant_df.iloc[_i]
    ax.annotate(
        f"T{int(_row['topic'])}",
        (x[_i], y[_i]),
        fontsize=8, ha="left", va="bottom",
        xytext=(4, 4), textcoords="offset points",
    )

ax.set_xlabel("Quantity Gap: Topic Share  (CN − US)", fontsize=12)
ax.set_ylabel("Quality Gap: MNCS  (CN − US)", fontsize=12)
ax.set_title(
    "Quantity–Quality Quadrant (CN relative to US)\n"
    "Upper-right = CN leads in both quantity & quality",
    fontsize=13,
)

# Quadrant labels
_xl = ax.get_xlim(); _yl = ax.get_ylim()
ax.text(_xl[1] * 0.70, _yl[1] * 0.85, "CN leads\nboth",
        ha="center", fontsize=10, color="#E03C31", alpha=0.5)
ax.text(_xl[0] * 0.70, _yl[0] * 0.85, "US leads\nboth",
        ha="center", fontsize=10, color="#3C6FE0", alpha=0.5)
ax.text(_xl[1] * 0.70, _yl[0] * 0.85, "CN: more\nUS: better",
        ha="center", fontsize=10, color="#888888", alpha=0.5)
ax.text(_xl[0] * 0.70, _yl[1] * 0.85, "US: more\nCN: better",
        ha="center", fontsize=10, color="#888888", alpha=0.5)

plt.tight_layout()
fig.savefig(os.path.join(CAP_DIR, "topic_quadrant.png"), dpi=200)
print(f"✅ Saved topic_quadrant.png")
plt.show()

## 11.9 Summary Exports
Generate a structured JSON and a human-readable Markdown summary of all capability-gap findings, then list the output directory.

**【中文说明】**
- 将前沿分布、语义缺口、影响缺口等结果汇总为 JSON 与 Markdown，便于插入报告并留痕分析配置。


In [ ]:
# # 【单元格中文说明】
# # - 汇总前沿/语义/影响缺口的关键主题清单，写入 capability_gap_summary.json 与 .md，并列目录。

# # ══════════════════════════════════════════════════════════════════════════════
# # Cell J — Summary Exports
# # ══════════════════════════════════════════════════════════════════════════════

# # ── Gather top findings ──────────────────────────────────────────────────────

# # Top frontier distribution deltas
# _top_fr_dist = (
#     frontier_topic_dist.head(5)[["topic", "topic_name", "delta_F"]]
#     .to_dict("records")
# )

# # Top frontier semantic gaps
# _sem_valid = (
#     semantic_gap_df[~semantic_gap_df["skipped"]]
#     .dropna(subset=["gap_frontier_semantic"])
# )
# _top_sem_gap = (
#     _sem_valid.sort_values("gap_frontier_semantic", ascending=False)
#     .head(5)[["topic", "topic_name", "gap_frontier_semantic"]]
#     .to_dict("records")
# )

# # Top MNCS impact gaps (US leads)
# _top_mncs = (
#     impact_gap.dropna(subset=["gap_MNCS"])
#     .sort_values("gap_MNCS", ascending=False).head(5)
# )
# _top_mncs_list = [
#     {"topic": int(t), "topic_name": str(n), "gap_MNCS": round(float(g), 4)}
#     for t, n, g in zip(_top_mncs.index, _top_mncs["topic_name"], _top_mncs["gap_MNCS"])
# ]

# # Top PP10 impact gaps (US leads)
# _top_pp10 = (
#     impact_gap.dropna(subset=["gap_PP10"])
#     .sort_values("gap_PP10", ascending=False).head(5)
# )
# _top_pp10_list = [
#     {"topic": int(t), "topic_name": str(n), "gap_PP10": round(float(g), 4)}
#     for t, n, g in zip(_top_pp10.index, _top_pp10["topic_name"], _top_pp10["gap_PP10"])
# ]

# # ── Build summary dict ───────────────────────────────────────────────────────
# summary = {
#     "analysis_date": pd.Timestamp.now().isoformat(),
#     "config": {
#         "TOPIC_COL": TOPIC_COL,
#         "WINDOW_YEARS": WINDOW_YEARS,
#         "EXCLUDE_RECENT_YEARS": EXCLUDE_RECENT_YEARS,
#         "MIN_BUCKET_SIZE": MIN_BUCKET_SIZE,
#         "MIN_FRONTIER_US": MIN_FRONTIER_US,
#         "FRONTIER_TYPE": FRONTIER_COL,
#         "frontier_window": f"{frontier_year_start}-{frontier_year_end}",
#     },
#     "frontier_distribution": {
#         "js_distance": round(float(js_dist_frontier), 6),
#         "jsd": round(float(jsd_frontier), 6),
#         "top_delta_topics": _top_fr_dist,
#     },
#     "frontier_semantic_gap": {
#         "topics_computed": int((~semantic_gap_df["skipped"]).sum()),
#         "topics_skipped": int(semantic_gap_df["skipped"].sum()),
#         "top_gap_topics_CN_farther": _top_sem_gap,
#     },
#     "impact_gap": {
#         "top_mncs_gap_US_leads": _top_mncs_list,
#         "top_pp10_gap_US_leads": _top_pp10_list,
#     },
# }

# # ── Write JSON ───────────────────────────────────────────────────────────────
# with open(os.path.join(CAP_DIR, "capability_gap_summary.json"), "w") as f:
#     json.dump(summary, f, indent=2, ensure_ascii=False, default=str)
# print("✅ Exported capability_gap_summary.json")

# # ── Write Markdown ───────────────────────────────────────────────────────────
# md = [
#     "# Capability Gap Summary\n",
#     f"**Analysis date:** {summary['analysis_date']}\n",
#     "## Configuration",
#     f"- Topic column: `{TOPIC_COL}`",
#     f"- Frontier type: `{FRONTIER_COL}` (window {frontier_year_start}–{frontier_year_end})",
#     f"- Exclude recent years: {EXCLUDE_RECENT_YEARS}",
#     f"- Min bucket size: {MIN_BUCKET_SIZE}; Min US frontier/topic: {MIN_FRONTIER_US}\n",
#     "## 1. Frontier Distribution Gap",
#     f"- JS distance: **{js_dist_frontier:.4f}**   JSD: **{jsd_frontier:.4f}**",
#     "- Top topics where US frontier share exceeds CN frontier (largest |Δ|):",
# ]
# for r in _top_fr_dist:
#     md.append(f"  - T{r['topic']} ({str(r.get('topic_name',''))[:40]}): "
#               f"Δ = {r['delta_F']:.4f}")

# md += [
#     "\n## 2. Frontier Semantic Gap (centre-based)",
#     f"- Topics computed: {summary['frontier_semantic_gap']['topics_computed']}",
#     f"- Topics skipped (insufficient US frontier): "
#     f"{summary['frontier_semantic_gap']['topics_skipped']}",
#     "- Top topics where CN is farthest from US frontier:",
# ]
# for r in _top_sem_gap:
#     md.append(f"  - T{r['topic']} ({str(r.get('topic_name',''))[:40]}): "
#               f"gap = {r['gap_frontier_semantic']:.4f}")

# md += ["\n## 3. Impact Gap — MNCS", "- Top topics where US leads:"]
# for r in _top_mncs_list:
#     md.append(f"  - T{r['topic']} ({str(r.get('topic_name',''))[:40]}): "
#               f"gap = {r['gap_MNCS']:.4f}")

# md += ["\n## 4. Impact Gap — PP(top 10 %)", "- Top topics where US leads:"]
# for r in _top_pp10_list:
#     md.append(f"  - T{r['topic']} ({str(r.get('topic_name',''))[:40]}): "
#               f"gap = {r['gap_PP10']:.4f}")

# with open(os.path.join(CAP_DIR, "capability_gap_summary.md"), "w") as f:
#     f.write("\n".join(md))
# print("✅ Exported capability_gap_summary.md")

# # ── Print output listing ─────────────────────────────────────────────────────
# print(f"\n{'═' * 60}")
# print(f"  Capability-gap outputs: {os.path.abspath(CAP_DIR)}")
# print(f"{'═' * 60}")
# for root, dirs, files in os.walk(CAP_DIR):
#     level = root.replace(CAP_DIR, "").count(os.sep)
#     indent = "  " * level
#     print(f"{indent}{os.path.basename(root)}/")
#     sub_indent = "  " * (level + 1)
#     for fn in sorted(files):
#         fpath = os.path.join(root, fn)
#         size_mb = os.path.getsize(fpath) / 1024 / 1024
#         print(f"{sub_indent}{fn}  ({size_mb:.2f} MB)")

# print(f"\n🎉 Capability-gap analysis complete!")

# 12. Topics over Time — Time-Evolution Evidence Chain
Compute per-topic share time-series for CN and US (yearly & 5-year rolling window), classify catching-up / pulling-away trends, and visualise the dynamics.

**【中文说明】**
- **主题随时间演化**：构建年度与滚动窗口下的中美主题份额序列，识别追赶/拉开/平稳等模式，并输出大量 per-topic 曲线图。
- 滚动窗口可平滑单年噪声，更适合观察趋势。


In [ ]:
# 【单元格中文说明】
# - 为时间演化准备：确定 topic 列与 country2；筛中美；清洗年份范围；输出到 time_evolution/。


# ══════════════════════════════════════════════════════════════════════════════
# 12.1  Data Cleaning & Standardisation for Time-Evolution Analysis
# ══════════════════════════════════════════════════════════════════════════════
from pathlib import Path
import warnings

# ── Output sub-directory ─────────────────────────────────────────────────────
TIME_DIR = Path(OUTPUT_DIR) / "time_evolution"
FIGS_DIR = TIME_DIR / "figs"
TIME_DIR.mkdir(parents=True, exist_ok=True)
FIGS_DIR.mkdir(parents=True, exist_ok=True)

# ── Select base DataFrame ───────────────────────────────────────────────────
base_df = df_valid.copy() if "df_valid" in dir() and df_valid is not None else df.copy()
print(f"Base DataFrame rows: {len(base_df):,}")

# ── Detect topic column (reuse TOPIC_COL from §11 or auto-detect) ───────────
_tc_candidates = ["topic_reduced", "topic_reassign", "topic"]
topic_col = None
if "TOPIC_COL" in dir() and TOPIC_COL in base_df.columns:
    topic_col = TOPIC_COL
else:
    for _c in _tc_candidates:
        if _c in base_df.columns and base_df[_c].notna().any():
            topic_col = _c
            break
if topic_col is None:
    raise RuntimeError("No topic column found – run BERTopic first.")
print(f"topic_col = '{topic_col}'")

# ── Ensure 'country2' exists ────────────────────────────────────────────────
if "country2" not in base_df.columns:
    if "country_code" in base_df.columns:
        base_df["country2"] = base_df["country_code"].map(
            lambda c: "CN" if c == "CN" else ("US" if c == "US" else "OTHER")
        )
    else:
        raise RuntimeError("No country column found.")

# ── Filter CN & US topic records ─────────────────────────────────────────────
te_df = base_df[base_df["country2"].isin(["CN", "US"])].copy()
te_df = te_df[te_df[topic_col].notna()].copy()
te_df[topic_col] = te_df[topic_col].astype(int)
if (te_df[topic_col] == -1).any():
    te_df = te_df[te_df[topic_col] != -1].copy()

# ── Year cleaning ────────────────────────────────────────────────────────────
YEAR_COL_TE = "year"   # may differ from YEAR_COL in §11
if YEAR_COL_TE not in te_df.columns:
    for _yc in ["year", "Year", "pub_year", "PY"]:
        if _yc in te_df.columns:
            YEAR_COL_TE = _yc
            break
te_df["year_int"] = pd.to_numeric(te_df[YEAR_COL_TE], errors="coerce")
te_df = te_df.dropna(subset=["year_int"]).copy()
te_df["year_int"] = te_df["year_int"].astype(int)

# Auto min/max
min_year = int(te_df["year_int"].min())
max_year_te = int(te_df["year_int"].max())
# Sanity guard
min_year = max(min_year, 1900)
max_year_te = min(max_year_te, 2100)
te_df = te_df[(te_df["year_int"] >= min_year) & (te_df["year_int"] <= max_year_te)].copy()

print(f"✅ Time-evolution base data: {len(te_df):,} papers")
print(f"   Year range: {min_year} – {max_year_te}")
print(f"   Topics: {te_df[topic_col].nunique()}")
print(f"   CN: {(te_df['country2']=='CN').sum():,}  US: {(te_df['country2']=='US').sum():,}")


In [ ]:
# 【单元格中文说明】
# - 按年计算各国总发文为分母，得到每主题年度份额及 delta=share_CN−share_US；导出 topic_share_yearly.csv。

# ══════════════════════════════════════════════════════════════════════════════
# 12.2  Build yearly topic share
# ══════════════════════════════════════════════════════════════════════════════

def build_topic_time_series(df_in, topic_col, year_col="year_int",
                            country_col="country2"):
    """
    Compute yearly topic share for CN and US.

    Returns
    -------
    yearly : pd.DataFrame with columns
        [topic, year, count_CN, count_US, share_CN, share_US, delta]
    """
    # 中文：按年、按国统计各主题发文数；用该国当年总发文作分母得份额；delta=CN 份额−US 份额。
    counts = (
        df_in.groupby([topic_col, year_col, country_col])
        .size()
        .reset_index(name="n")
    )

    # Total papers per country-year (denominator for share)
    country_year_total = (
        df_in.groupby([year_col, country_col])
        .size()
        .reset_index(name="n_total")
    )
    counts = counts.merge(country_year_total, on=[year_col, country_col], how="left")
    counts["share"] = counts["n"] / counts["n_total"]

    # Pivot to wide: one row per (topic, year)
    wide_count = counts.pivot_table(
        index=[topic_col, year_col], columns=country_col, values="n", fill_value=0
    ).reset_index()
    wide_share = counts.pivot_table(
        index=[topic_col, year_col], columns=country_col, values="share", fill_value=0
    ).reset_index()

    for c in ["CN", "US"]:
        if c not in wide_count.columns:
            wide_count[c] = 0
        if c not in wide_share.columns:
            wide_share[c] = 0.0

    result = pd.DataFrame({
        "topic":    wide_count[topic_col],
        "year":     wide_count[year_col],
        "count_CN": wide_count["CN"],
        "count_US": wide_count["US"],
        "share_CN": wide_share["CN"],
        "share_US": wide_share["US"],
    })
    result["delta"] = result["share_CN"] - result["share_US"]
    return result.sort_values(["topic", "year"]).reset_index(drop=True)

yearly = build_topic_time_series(te_df, topic_col)
yearly.to_csv(TIME_DIR / "topic_share_yearly.csv", index=False)
print(f"✅ Exported topic_share_yearly.csv  ({len(yearly):,} rows)")
yearly.head(10)

In [ ]:
# 【单元格中文说明】
# - 对年度计数做滚动求和再归一化，得到 roll5 份额序列，降低单年波动；导出 topic_share_roll5.csv。

# ══════════════════════════════════════════════════════════════════════════════
# 12.3  Build 5-year rolling-window topic share
# ══════════════════════════════════════════════════════════════════════════════

def build_rolling_share(df_in, topic_col, year_col="year_int",
                        country_col="country2", window=5, min_periods=3):
    """
    Compute 5-year rolling-window topic share for CN and US.

    Steps:
      1. Pivot counts to wide: index=[topic, year], columns=country, values=count
      2. Per topic-country: rolling(window).sum()
      3. Re-normalise within each year-country (denominator = sum of rolling counts)

    Returns
    -------
    roll_df : pd.DataFrame with columns
        [topic, year, roll_count_CN, roll_count_US, share_CN, share_US, delta]
    """
    # 中文：对年度计数做滚动求和再按国年归一化，得到平滑后的主题份额序列。
    counts = (
        df_in.groupby([topic_col, year_col, country_col])
        .size()
        .reset_index(name="n")
    )
    wide = counts.pivot_table(
        index=[topic_col, year_col], columns=country_col, values="n", fill_value=0
    ).reset_index()
    for c in ["CN", "US"]:
        if c not in wide.columns:
            wide[c] = 0

    all_years = sorted(df_in[year_col].unique())
    all_topics = sorted(df_in[topic_col].unique())

    # Full index so rolling works continuously
    full_idx = pd.MultiIndex.from_product(
        [all_topics, all_years], names=[topic_col, year_col]
    )
    wide = wide.set_index([topic_col, year_col]).reindex(full_idx, fill_value=0).reset_index()

    # Rolling sum per topic-country
    wide = wide.sort_values([topic_col, year_col])
    for c in ["CN", "US"]:
        wide[f"roll_{c}"] = (
            wide.groupby(topic_col)[c]
            .transform(lambda s: s.rolling(window=window, min_periods=min_periods).sum())
        )

    # Drop rows where rolling is NaN (not enough history)
    wide = wide.dropna(subset=["roll_CN", "roll_US"]).copy()

    # Denominator: total rolling counts per country-year
    for c in ["CN", "US"]:
        year_total = wide.groupby(year_col)[f"roll_{c}"].transform("sum")
        wide[f"share_{c}"] = wide[f"roll_{c}"] / year_total.replace(0, np.nan)
    wide["share_CN"] = wide["share_CN"].fillna(0)
    wide["share_US"] = wide["share_US"].fillna(0)
    wide["delta"] = wide["share_CN"] - wide["share_US"]

    result = wide[[topic_col, year_col, "roll_CN", "roll_US",
                    "share_CN", "share_US", "delta"]].copy()
    result.columns = ["topic", "year", "roll_count_CN", "roll_count_US",
                       "share_CN", "share_US", "delta"]
    return result.sort_values(["topic", "year"]).reset_index(drop=True)

roll5 = build_rolling_share(te_df, topic_col, window=5, min_periods=3)
roll5.to_csv(TIME_DIR / "topic_share_roll5.csv", index=False)
print(f"✅ Exported topic_share_roll5.csv  ({len(roll5):,} rows)")
roll5.head(10)

In [ ]:
# 【单元格中文说明】
# - 对每个主题的 delta(t) 用 Theil–Sen 估计稳健斜率，划分 catching_up / pulling_away / stable；导出 topic_trend_summary.csv。

# ══════════════════════════════════════════════════════════════════════════════
# 12.4  Classify trends: catching-up / pulling-away / stable
# ══════════════════════════════════════════════════════════════════════════════
from sklearn.linear_model import TheilSenRegressor

def classify_trends(roll_df, min_years=6, eps=1e-4):
    """
    For each topic, fit a robust slope on delta(t) from the roll5 series.

    Returns
    -------
    summary : pd.DataFrame with columns
        [topic, slope_delta, delta_start, delta_end, cross_year, trend_label, n_years]
    """
    # 中文：对每个主题的 delta(t) 用 Theil–Sen 稳健回归估斜率，并标记追赶/拉开/平稳。
    records = []
    for t, grp in roll_df.groupby("topic"):
        grp = grp.sort_values("year").dropna(subset=["delta"])
        n = len(grp)
        if n < min_years:
            records.append({
                "topic": int(t), "slope_delta": np.nan,
                "delta_start": np.nan, "delta_end": np.nan,
                "cross_year": np.nan, "trend_label": "insufficient",
                "n_years": n
            })
            continue

        years = grp["year"].values.reshape(-1, 1).astype(float)
        deltas = grp["delta"].values.astype(float)

        # Robust slope via Theil-Sen
        try:
            ts = TheilSenRegressor(random_state=SEED, max_subpopulation=5000)
            ts.fit(years, deltas)
            slope = float(ts.coef_[0])
        except Exception:
            # Fallback to simple polyfit
            slope = float(np.polyfit(years.ravel(), deltas, 1)[0])

        delta_start = float(deltas[0])
        delta_end   = float(deltas[-1])

        # Crossing: delta goes from <0 to >=0 (CN overtakes US in share)
        cross_year = np.nan
        for i in range(1, len(deltas)):
            if deltas[i - 1] < 0 and deltas[i] >= 0:
                cross_year = int(grp["year"].values[i])
                break

        # Label
        if slope > eps and (delta_end > delta_start + eps):
            label = "catching_up"
        elif slope < -eps:
            label = "pulling_away"
        else:
            label = "stable"

        records.append({
            "topic": int(t), "slope_delta": round(slope, 8),
            "delta_start": round(delta_start, 6),
            "delta_end": round(delta_end, 6),
            "cross_year": cross_year,
            "trend_label": label,
            "n_years": n,
        })
    return pd.DataFrame(records)

trend_summary = classify_trends(roll5, min_years=6, eps=1e-4)
trend_summary.to_csv(TIME_DIR / "topic_trend_summary.csv", index=False)
print(f"✅ Exported topic_trend_summary.csv  ({len(trend_summary)} topics)")
print(f"\nTrend distribution:")
print(trend_summary["trend_label"].value_counts().to_string())
trend_summary.sort_values("slope_delta", ascending=False).head(10)

In [ ]:
# 【单元格中文说明】
# - 选取 |斜率| 最大的若干主题绘制份额与 delta 曲线，并叠绘总览图；PNG 存于 time_evolution/figs/。

# ══════════════════════════════════════════════════════════════════════════════
# 12.5  Visualisation — TopK topic share & delta curves
# ══════════════════════════════════════════════════════════════════════════════
# TOPK_VIS = TOPK_REPORT if "TOPK_REPORT" in dir() else 60
TOPK_VIS = 60
# ── Helper: topic name map (reuse from §11 if available) ─────────────────────
_name_map_te = {}
if "topic_info_export" in dir():
    _name_map_te = dict(zip(topic_info_export["topic_id"], topic_info_export["name"]))
elif "_name_map" in dir():
    _name_map_te = _name_map

def _topic_label(tid):
    # 中文：组合主题编号与 BERTopic 名称，缩短用于图例。
    # 中文：组合主题编号与 BERTopic 名称，缩短用于图例。
    name = str(_name_map_te.get(tid, ""))[:30]
    return f"T{tid}: {name}" if name else f"T{tid}"

# ── Select TopK by |slope_delta| among classified topics ─────────────────────
_ranked = (
    trend_summary[trend_summary["trend_label"] != "insufficient"]
    .sort_values("slope_delta", key=abs, ascending=False)
    .head(TOPK_VIS)
)
topk_topics = _ranked["topic"].tolist()
print(f"Visualising {len(topk_topics)} topics (by |slope_delta|): {topk_topics}")


def plot_share_over_time(roll_df, tid, out_dir):
    """Plot CN vs US share (roll5) for a single topic."""
    # 中文：单主题 roll5 下中美份额随时间折线图。
    sub = roll_df[roll_df["topic"] == tid].sort_values("year")
    if sub.empty:
        return
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(sub["year"], sub["share_CN"], "o-", color="#E03C31", label="CN share", markersize=3)
    ax.plot(sub["year"], sub["share_US"], "s-", color="#3C6FE0", label="US share", markersize=3)
    ax.set_xlabel("Year")
    ax.set_ylabel("Topic share (roll5)")
    ax.set_title(f"Share over Time — {_topic_label(tid)}")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(out_dir / f"share_T{tid}.png", dpi=150)
    plt.close(fig)


def plot_delta_over_time(roll_df, tid, trend_row, out_dir):
    """Plot delta(t) = share_CN - share_US for a single topic."""
    # 中文：单主题 delta 随时间；可标出由负变正的年份（交叉年）。
    sub = roll_df[roll_df["topic"] == tid].sort_values("year")
    if sub.empty:
        return
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(sub["year"], sub["delta"], "o-", color="#6A0DAD", markersize=3)
    ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")
    ax.fill_between(sub["year"], sub["delta"], 0, alpha=0.15, color="#6A0DAD")

    # Mark cross_year if present
    cross = trend_row.get("cross_year", np.nan)
    if pd.notna(cross):
        ax.axvline(cross, color="green", linewidth=1.5, linestyle=":", label=f"cross year={int(cross)}")
        ax.legend(fontsize=9)

    label = trend_row.get("trend_label", "")
    slope = trend_row.get("slope_delta", np.nan)
    slope_str = f"{slope:.2e}" if pd.notna(slope) else "N/A"
    ax.set_xlabel("Year")
    ax.set_ylabel("Δ share (CN − US)")
    ax.set_title(f"Delta over Time — {_topic_label(tid)}  [{label}, slope={slope_str}]")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(out_dir / f"delta_T{tid}.png", dpi=150)
    plt.close(fig)


# ── Generate per-topic plots ─────────────────────────────────────────────────
trend_dict = trend_summary.set_index("topic").to_dict("index")
for tid in topk_topics:
    plot_share_over_time(roll5, tid, FIGS_DIR)
    tr = trend_dict.get(tid, {})
    plot_delta_over_time(roll5, tid, tr, FIGS_DIR)

print(f"✅ Saved {len(topk_topics)*2} per-topic plots to {FIGS_DIR}")


# # ── Overview: TopK delta curves overlay ──────────────────────────────────────
# fig, ax = plt.subplots(figsize=(14, 7))
# for tid in topk_topics:
#     sub = roll5[roll5["topic"] == tid].sort_values("year")
#     if sub.empty:
#         continue
#     ax.plot(sub["year"], sub["delta"], "-", linewidth=1.2,
#             label=_topic_label(tid), alpha=0.8)

# ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
# ax.set_xlabel("Year", fontsize=12)
# ax.set_ylabel("Δ share (CN − US)", fontsize=12)
# ax.set_title(f"Delta Trend Overview — Top {len(topk_topics)} Topics (roll5)", fontsize=13)
# ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7, ncol=1)
# ax.grid(True, alpha=0.3)
# plt.tight_layout()
# fig.savefig(FIGS_DIR / "delta_overview_topk.png", dpi=200, bbox_inches="tight")
# print(f"✅ Saved delta_overview_topk.png")
# plt.show()

# 13. Lead–Lag Analysis (Entry Period & Lag Years)
For each topic, determine when CN and US "entered" the field (sustained growth above baseline) and compute the lag in years. Validate with cross-correlation.

**【中文说明】**
- **进入年份**：当一国在某主题上的份额持续高于基线时记为“进入”该主题领域；中美进入年之差为 **滞后/领先年数**。
- 辅以交叉相关验证时间对齐关系，减少单次阈值带来的偶然性。


In [ ]:
# 【单元格中文说明】
# - detect_enter_year：用基线均值+kσ 阈值检测份额“跃升”进入年份；对中美分别求 enter_CN、enter_US。

# ══════════════════════════════════════════════════════════════════════════════
# 13.1  Detect enter_year for each topic-country
# ══════════════════════════════════════════════════════════════════════════════

def detect_enter_year(series_year, series_share, k=1.0, L=2, min_baseline=3):
    """
    Detect the 'entry year' for a topic-country share time-series.

    Parameters
    ----------
    series_year : array-like of int — sorted years
    series_share : array-like of float — corresponding share values
    k : float — number of sigma above baseline mean
    L : int — look-ahead window for sustained growth check
    min_baseline : int — minimum baseline years

    Returns
    -------
    enter_year : int or NaN
    flag : str — 'ok' or 'insufficient' or 'not_found'
    """
    # 中文：用早期年份份额作基线，高于 μ+kσ 且后续非跌视为「进入」该主题领域。
    years = np.array(series_year)
    shares = np.array(series_share, dtype=float)
    n = len(years)

    if n < min_baseline + L:
        return np.nan, "insufficient"

    # Baseline: first B years
    B = min(5, max(min_baseline, n // 3))
    mu = shares[:B].mean()
    sigma = shares[:B].std()
    if sigma == 0 or np.isnan(sigma):
        sigma = 1e-8

    threshold = mu + k * sigma

    for i in range(B, n):
        if shares[i] > threshold:
            # Check sustained growth: average change over next L years >= 0
            end_check = min(i + L, n - 1)
            if end_check > i:
                future_diffs = np.diff(shares[i:end_check + 1])
                if future_diffs.mean() >= 0:
                    return int(years[i]), "ok"
            else:
                # At the edge of data — accept if above threshold
                return int(years[i]), "ok"

    return np.nan, "not_found"


# ── Build per-topic-country share series from roll5 ──────────────────────────
lead_lag_records = []

for t in sorted(roll5["topic"].unique()):
    sub = roll5[roll5["topic"] == t].sort_values("year")
    if sub.empty:
        continue

    result = {"topic": int(t)}
    for country in ["CN", "US"]:
        col = f"share_{country}"
        valid = sub.dropna(subset=[col])
        if len(valid) < 5:
            result[f"enter_{country}"] = np.nan
            result[f"flag_{country}"] = "insufficient"
        else:
            ey, flag = detect_enter_year(valid["year"].values, valid[col].values)
            result[f"enter_{country}"] = ey
            result[f"flag_{country}"] = flag

    lead_lag_records.append(result)

lead_lag_df = pd.DataFrame(lead_lag_records)
print(f"✅ Entry-year detection done for {len(lead_lag_df)} topics")
print(f"   CN enter found: {lead_lag_df['enter_CN'].notna().sum()}")
print(f"   US enter found: {lead_lag_df['enter_US'].notna().sum()}")

In [ ]:
# 【单元格中文说明】
# - lag = enter_CN − enter_US；交叉相关扫描 ±max_lag 年对齐，记录最大相关的滞后 lag_ccf；合并导出 topic_lead_lag.csv。

# ══════════════════════════════════════════════════════════════════════════════
# 13.2  Compute lag + 13.3 Cross-correlation validation
# ══════════════════════════════════════════════════════════════════════════════

def compute_lead_lag(lead_lag_df):
    """Compute lag = enter_CN - enter_US and assign label."""
    # 中文：lag=enter_CN−enter_US；并扫描互相关得最优对齐滞后。
    df_ll = lead_lag_df.copy()
    both_valid = df_ll["enter_CN"].notna() & df_ll["enter_US"].notna()
    df_ll["lag"] = np.nan
    df_ll.loc[both_valid, "lag"] = (
        df_ll.loc[both_valid, "enter_CN"] - df_ll.loc[both_valid, "enter_US"]
    )

    def _label(row):
        if pd.isna(row["lag"]):
            return "NA"
        if row["lag"] > 0:
            return "CN_lagging"
        elif row["lag"] < 0:
            return "CN_leading"
        else:
            return "tie"
    df_ll["lag_label"] = df_ll.apply(_label, axis=1)
    return df_ll

lead_lag_df = compute_lead_lag(lead_lag_df)

# ── 13.3 Cross-correlation validation (no extra packages) ───────────────────
def cross_corr_lag(roll_df, topic, max_lag=10):
    """
    Compute cross-correlation between CN and US share series.

    Returns (best_lag, max_corr) or (NaN, NaN) if insufficient data.
    Positive lag means CN lags behind US by that many years.
    """
    # 中文：在 ±max_lag 内平移中美份额序列，取 Pearson 相关最大的滞后（正=CN 滞后）。
    sub = roll_df[roll_df["topic"] == topic].sort_values("year")
    cn = sub["share_CN"].values
    us = sub["share_US"].values
    if len(cn) < 6 or np.std(cn) < 1e-10 or np.std(us) < 1e-10:
        return np.nan, np.nan

    best_lag = 0
    best_corr = -2  # below any valid correlation

    for lag in range(-max_lag, max_lag + 1):
        if lag >= 0:
            cn_seg = cn[lag:]
            us_seg = us[:len(cn_seg)]
        else:
            us_seg = us[-lag:]
            cn_seg = cn[:len(us_seg)]
        min_len = min(len(cn_seg), len(us_seg))
        if min_len < 4:
            continue
        cn_seg = cn_seg[:min_len]
        us_seg = us_seg[:min_len]

        # Pearson correlation
        cn_demeaned = cn_seg - cn_seg.mean()
        us_demeaned = us_seg - us_seg.mean()
        denom = np.sqrt((cn_demeaned ** 2).sum() * (us_demeaned ** 2).sum())
        if denom < 1e-15:
            continue
        corr = (cn_demeaned * us_demeaned).sum() / denom
        if corr > best_corr:
            best_corr = corr
            best_lag = lag

    return best_lag, round(float(best_corr), 4) if best_corr > -2 else np.nan

# Apply cross-correlation
ccf_results = []
for t in lead_lag_df["topic"]:
    lag_ccf, max_corr = cross_corr_lag(roll5, t, max_lag=10)
    ccf_results.append({"topic": int(t), "lag_ccf": lag_ccf, "max_corr": max_corr})

ccf_df = pd.DataFrame(ccf_results)
lead_lag_df = lead_lag_df.merge(ccf_df, on="topic", how="left")

# ── Add topic names ──────────────────────────────────────────────────────────
lead_lag_df["topic_name"] = lead_lag_df["topic"].map(_name_map_te)

# ── Export ────────────────────────────────────────────────────────────────────
lead_lag_df.to_csv(TIME_DIR / "topic_lead_lag.csv", index=False)
print(f"✅ Exported topic_lead_lag.csv  ({len(lead_lag_df)} topics)")
print(f"\nLag label distribution:")
print(lead_lag_df["lag_label"].value_counts().to_string())
print(f"\nCross-correlation computed: {lead_lag_df['lag_ccf'].notna().sum()} topics")
lead_lag_df.sort_values("lag", ascending=False).head(10)

In [ ]:
# 【单元格中文说明】
# - 将滞后与 CCF 滞后以条形图展示，颜色区分中国相对美国更晚或更早进入。

# ══════════════════════════════════════════════════════════════════════════════
# 13.4  Visualisation — Lead-lag bar chart
# ══════════════════════════════════════════════════════════════════════════════

# ── Bar chart: lag by topic (sorted, TopK by |lag|) ──────────────────────────
_valid_lag = lead_lag_df.dropna(subset=["lag"]).copy()
_valid_lag["abs_lag"] = _valid_lag["lag"].abs()
_plot_lag = _valid_lag.sort_values("abs_lag", ascending=False).head(TOPK_VIS)

if len(_plot_lag) > 0:
    _labels = [_topic_label(int(t)) for t in _plot_lag["topic"]]
    _lags = _plot_lag["lag"].values
    colors = ["#E03C31" if lg > 0 else ("#3C6FE0" if lg < 0 else "#888888")
              for lg in _lags]

    fig, ax = plt.subplots(figsize=(14, 7))
    bars = ax.barh(range(len(_labels)), _lags, color=colors)
    ax.set_yticks(range(len(_labels)))
    ax.set_yticklabels(_labels, fontsize=9)
    ax.set_xlabel("Lag (years): CN enter − US enter", fontsize=12)
    ax.set_title(
        f"Lead–Lag: CN vs US Entry Year Gap — Top {len(_plot_lag)} Topics\n"
        "(red / positive = CN lags behind US; blue / negative = CN leads)",
        fontsize=13,
    )
    ax.axvline(0, color="black", linewidth=0.8)
    ax.grid(True, axis="x", alpha=0.3)
    plt.tight_layout()
    fig.savefig(FIGS_DIR / "lead_lag_bar.png", dpi=200)
    print(f"✅ Saved lead_lag_bar.png")
    plt.show()
else:
    print("⚠️  No valid lag values to plot — check data sufficiency.")

# ── Cross-correlation lag bar chart ──────────────────────────────────────────
_valid_ccf = lead_lag_df.dropna(subset=["lag_ccf"]).copy()
_valid_ccf["abs_ccf"] = _valid_ccf["lag_ccf"].abs()
_plot_ccf = _valid_ccf.sort_values("abs_ccf", ascending=False).head(TOPK_VIS)

if len(_plot_ccf) > 0:
    _labels_ccf = [_topic_label(int(t)) for t in _plot_ccf["topic"]]
    _ccf_vals = _plot_ccf["lag_ccf"].values
    colors_ccf = ["#E03C31" if lg > 0 else ("#3C6FE0" if lg < 0 else "#888888")
                  for lg in _ccf_vals]

    fig, ax = plt.subplots(figsize=(14, 7))
    ax.barh(range(len(_labels_ccf)), _ccf_vals, color=colors_ccf)
    ax.set_yticks(range(len(_labels_ccf)))
    ax.set_yticklabels(_labels_ccf, fontsize=9)
    ax.set_xlabel("Cross-correlation lag (years)", fontsize=12)
    ax.set_title(
        f"Cross-Correlation Lag — Top {len(_plot_ccf)} Topics\n"
        "(positive = CN signal lags US signal)",
        fontsize=13,
    )
    ax.axvline(0, color="black", linewidth=0.8)
    ax.grid(True, axis="x", alpha=0.3)
    plt.tight_layout()
    fig.savefig(FIGS_DIR / "lead_lag_ccf_bar.png", dpi=200)
    print(f"✅ Saved lead_lag_ccf_bar.png")
    plt.show()
else:
    print("⚠️  No valid cross-correlation lag values to plot.")

In [ ]:
# 【单元格中文说明】
# - 罗列 time_evolution/ 下生成的全部文件，便于与 overlap 结果对照。

# ══════════════════════════════════════════════════════════════════════════════
# §12-13  Summary — list all generated outputs
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'═' * 60}")
print(f"  Time-evolution outputs: {TIME_DIR.resolve()}")
print(f"{'═' * 60}")
for root, dirs, files in os.walk(TIME_DIR):
    level = str(root).replace(str(TIME_DIR), "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{Path(root).name}/")
    sub_indent = "  " * (level + 1)
    for fn in sorted(files):
        fpath = Path(root) / fn
        size_kb = fpath.stat().st_size / 1024
        print(f"{sub_indent}{fn}  ({size_kb:.1f} KB)")

print(f"\n🎉 Sections 12–13 (Time-evolution evidence chain) complete!")

# 14. BERTopic Overlap Lead–Lag Reproduction (Figure 11-style)
Reproduce the four-panel overlap-based lead–lag analysis between China (CN) and the United States (US).
Each panel compares country-year topic structures under different lag assumptions (0–15 years)
to study technology catch-up, gap-narrowing, and lead–lag dynamics.

**【中文说明】**
- **国家–年份主题重叠 lead–lag**：把每个 (国家, 年份) 的主题分布看作向量，在 0–15 年滞后下比较中美相似度，用于“技术周期”层面的同步与追赶诊断。


## 14.1  Config & variable checks

**【中文说明】**
- 检查国家、年份、主题列与最小样本阈值；配置 Jaccard/余弦等指标及滚动窗口；输出目录 `overlap_leadlag/`。


In [ ]:
# # 【单元格中文说明】
# # - §14 配置：国家/年份/主题列检测；LAG_RANGE 与最小单元格样本；选择 Jaccard/余弦开关与输出路径。


# # ══════════════════════════════════════════════════════════════════════════════
# # 14.1  Config & variable checks
# # ══════════════════════════════════════════════════════════════════════════════
# from pathlib import Path as _Path

# # ── Country column ────────────────────────────────────────────────────────────
# if "country2" in df.columns:
#     COUNTRY_COL_OL = "country2"
# elif "country_code" in df.columns:
#     if "country2" not in df.columns:
#         df["country2"] = df["country_code"].apply(
#             lambda x: "CN" if str(x).upper() in {"CN", "CHN", "CHINA"} else
#                       "US" if str(x).upper() in {"US", "USA", "UNITED STATES"} else "OTHER"
#         )
#     COUNTRY_COL_OL = "country2"
# else:
#     raise KeyError("Cannot find country column.  Expected 'country2' or 'country_code'.")

# # ── Year column ───────────────────────────────────────────────────────────────
# for _yc in ("year", "pub_year", "Year", "PY"):
#     if _yc in df.columns:
#         YEAR_COL_OL = _yc
#         break
# else:
#     raise KeyError("Cannot find year column.  Expected one of: year, pub_year, Year, PY.")

# # ── Topic column ──────────────────────────────────────────────────────────────
# for _tc in ("topic_reduced", "topic_reassign", "topic", TOPIC_COL):
#     if _tc in df.columns:
#         TOPIC_COL_OL = _tc
#         break
# else:
#     TOPIC_COL_OL = TOPIC_COL

# # ── Hyper-parameters ──────────────────────────────────────────────────────────
# LAG_RANGE                 = list(range(0, 16))   # 0–15 years
# MIN_DOCS_PER_COUNTRY_YEAR = 5                    # drop CY pairs with fewer docs
# PRESENCE_MIN_SHARE        = 0.01                 # topic must be ≥1 % of CY's corpus
# TOPN_SET                  = None                 # None = use PRESENCE_MIN_SHARE; int = top-N

# # ── Metric switches ───────────────────────────────────────────────────────────
# USE_WEIGHTED_JACCARD = True
# RUN_BINARY_JACCARD   = True
# RUN_COSINE           = True
# ROLLING_WINDOWS      = [1, 3, 5]
# FRONTIER_ONLY        = False  # also compute frontier-only robustness if flag col exists

# # ── Frontier flag column (from §11) ──────────────────────────────────────────
# _frontier_flag_col_ol = next(
#     (c for c in ("is_frontier", "is_top10", "top10_flag") if c in df.columns), None
# )

# # ── Output dirs ───────────────────────────────────────────────────────────────
# OVERLAP_DIR     = _Path(OUTPUT_DIR) / "overlap_leadlag"
# OVERLAP_FIG_DIR = OVERLAP_DIR / "figs"
# OVERLAP_TAB_DIR = OVERLAP_DIR / "tables"
# OVERLAP_DIR.mkdir(parents=True, exist_ok=True)
# OVERLAP_FIG_DIR.mkdir(parents=True, exist_ok=True)
# OVERLAP_TAB_DIR.mkdir(parents=True, exist_ok=True)

# # ── Subset ────────────────────────────────────────────────────────────────────
# _df_ol = df[df[COUNTRY_COL_OL].isin(["CN", "US"])].copy()
# _df_ol[YEAR_COL_OL] = _df_ol[YEAR_COL_OL].astype(int)

# print(f"§14 config OK")
# print(f"  country_col={COUNTRY_COL_OL!r}  year_col={YEAR_COL_OL!r}  topic_col={TOPIC_COL_OL!r}")
# print(f"  CN: {(_df_ol[COUNTRY_COL_OL]=='CN').sum():,}  US: {(_df_ol[COUNTRY_COL_OL]=='US').sum():,}")
# print(f"  lag range: {LAG_RANGE[0]}–{LAG_RANGE[-1]}  |  rolling windows: {ROLLING_WINDOWS}")
# print(f"  metrics: weighted_jaccard={USE_WEIGHTED_JACCARD}, binary={RUN_BINARY_JACCARD}, cosine={RUN_COSINE}")
# print(f"  OVERLAP_DIR → {OVERLAP_DIR}")


## 14.2  Build country-year topic matrices

**【中文说明】**
- 由文献级数据聚合为 (国家, 年份, 主题) 计数与份额，形成宽表矩阵，供后续向量相似度与滞后扫描使用。


In [ ]:
# # 【单元格中文说明】
# # - 聚合为 (国,年,主题) 长表与宽表，过滤极小样本单元；写出 cy_long / count_wide / share_wide。


# # ══════════════════════════════════════════════════════════════════════════════
# # 14.2  Build country-year topic matrices
# # ══════════════════════════════════════════════════════════════════════════════

# # ── Long table: (country, year, topic, n_docs, topic_share) ──────────────────
# _cy_long_ol = (
#     _df_ol.groupby([COUNTRY_COL_OL, YEAR_COL_OL, TOPIC_COL_OL])
#           .size().reset_index(name="n_docs")
# )
# _total_per_cy = (
#     _cy_long_ol.groupby([COUNTRY_COL_OL, YEAR_COL_OL])["n_docs"]
#                .transform("sum")
# )
# _cy_long_ol["topic_share"] = _cy_long_ol["n_docs"] / _total_per_cy.clip(lower=1)

# # Drop (country, year) with few total docs
# _cy_total_ol = _cy_long_ol.groupby([COUNTRY_COL_OL, YEAR_COL_OL])["n_docs"].transform("sum")
# _cy_long_ol  = _cy_long_ol[_cy_total_ol >= MIN_DOCS_PER_COUNTRY_YEAR].copy()
# print(f"cy_long_ol: {len(_cy_long_ol):,} rows  "
#       f"({_cy_long_ol[COUNTRY_COL_OL].nunique()} countries, "
#       f"{_cy_long_ol[YEAR_COL_OL].nunique()} years, "
#       f"{_cy_long_ol[TOPIC_COL_OL].nunique()} topics)")

# # ── Wide matrices ─────────────────────────────────────────────────────────────
# _count_wide_ol = (
#     _cy_long_ol.pivot_table(index=TOPIC_COL_OL,
#                              columns=[COUNTRY_COL_OL, YEAR_COL_OL],
#                              values="n_docs", aggfunc="sum", fill_value=0)
# )
# _share_wide_ol = (
#     _cy_long_ol.pivot_table(index=TOPIC_COL_OL,
#                              columns=[COUNTRY_COL_OL, YEAR_COL_OL],
#                              values="topic_share", aggfunc="mean", fill_value=0)
# )

# # ── Save raw tables ───────────────────────────────────────────────────────────
# _cy_long_ol.to_csv(OVERLAP_TAB_DIR / "cy_long.csv",         index=False)
# _count_wide_ol.to_csv(OVERLAP_TAB_DIR / "count_wide.csv")
# _share_wide_ol.to_csv(OVERLAP_TAB_DIR / "share_wide.csv")
# print("✔ cy_long.csv  |  count_wide.csv  |  share_wide.csv")

# # ── Per-country per-year summary ──────────────────────────────────────────────
# _cy_summary_ol = (
#     _cy_long_ol.groupby([COUNTRY_COL_OL, YEAR_COL_OL])
#                .agg(n_docs=("n_docs","sum"), n_topics=("n_docs","count"))
#                .reset_index()
# )
# _cy_summary_ol.to_csv(OVERLAP_TAB_DIR / "cy_summary.csv", index=False)
# print("✔ cy_summary.csv")
# print(_cy_summary_ol.pivot(index=YEAR_COL_OL, columns=COUNTRY_COL_OL,
#                              values="n_docs").fillna(0).astype(int).to_string())


## 14.3  Build topic sets / topic-share vectors

**【中文说明】**
- 将每年的主题份额转为集合或概率向量：可做二元 Jaccard、加权 Jaccard、余弦相似度等，以刻画结构重叠程度。


In [ ]:
# # 【单元格中文说明】
# # - 将长表转为每年每国的主题向量（集合或概率），供滞后扫描函数复用。


# # ══════════════════════════════════════════════════════════════════════════════
# # 14.3  Build topic sets / topic-share vectors
# # ══════════════════════════════════════════════════════════════════════════════

# def build_topic_set_from_row(row: dict, topic_col: str,
#                               share_col: str = "topic_share",
#                               presence_min_share: float = 0.01,
#                               topn: int | None = None) -> set:
#     """Extract the set of 'present' topics from a (country, year) aggregate row."""
#     # 中文：从 (国,年) 聚合行提取「出现」的主题集合；可按最小份额或 top-N 截断。
#     topics_series = row.get("topics", {})
#     if isinstance(topics_series, dict):
#         present = {t for t, s in topics_series.items() if s >= presence_min_share}
#     else:
#         present = set()
#     if topn is not None:
#         sorted_t = sorted(topics_series.items(), key=lambda x: x[1], reverse=True)
#         present = {t for t, _ in sorted_t[:topn]}
#     return present


# def safe_get_vector(vecs: dict, country: str, year: int) -> dict:
#     # 中文：安全读取嵌套字典 vecs[country][year]，缺省返回空。
#     # 中文：安全读取嵌套字典 vecs[country][year]，缺省返回空。
#     return vecs.get(country, {}).get(year, {})


# def has_enough_docs(cy_long, country_col, year_col, country, year,
#                     min_docs: int = 5) -> bool:
#     # 中文：判断该 (国,年) 单元格总发文是否达到最小阈值，过滤不可靠份额。
#     # 中文：判断该 (国,年) 单元格总发文是否达到最小阈值，过滤不可靠份额。
#     mask = (cy_long[country_col] == country) & (cy_long[year_col] == year)
#     return cy_long.loc[mask, "n_docs"].sum() >= min_docs


# # ── Build wide matrices (country × topic share per year) ─────────────────
# _count_wide = _count_wide_ol.copy()
# _share_wide = _share_wide_ol.copy()

# # ── Build vecs from the full long table ──────────────────────────────────
# topic_sets_ol: dict        = {}   # topic_sets_ol[country][year] = set
# topic_share_vectors_ol: dict = {}  # topic_share_vectors_ol[country][year] = {topic: share}
# topic_binary_vectors_ol: dict = {} # topic_binary_vectors_ol[country][year] = {topic: 0/1}

# for (country, year), grp in _cy_long_ol.groupby([COUNTRY_COL_OL, YEAR_COL_OL]):
#     share_dict = dict(zip(grp[TOPIC_COL_OL], grp["topic_share"]))
#     topic_share_vectors_ol.setdefault(country, {})[year] = share_dict
#     present = {t for t, s in share_dict.items() if s >= PRESENCE_MIN_SHARE}
#     topic_sets_ol.setdefault(country, {})[year] = present
#     topic_binary_vectors_ol.setdefault(country, {})[year] = {
#         t: 1 if s >= PRESENCE_MIN_SHARE else 0 for t, s in share_dict.items()
#     }

# _n_cn = len(topic_sets_ol.get("CN", {}))
# _n_us = len(topic_sets_ol.get("US", {}))
# print(f"topic_sets_ol: CN={_n_cn} years, US={_n_us} years")
# print(f"topic_share_vectors_ol: CN={len(topic_share_vectors_ol.get('CN', {}))}, "
#       f"US={len(topic_share_vectors_ol.get('US', {}))}")


## 14.4  Similarity metrics

**【中文说明】**
- 定义各类相似度度量及滞后对齐方式，使不同度量下的结论可对照。


In [ ]:
# # 【单元格中文说明】
# # - 实现加权 Jaccard、二元 Jaccard、余弦等相似度及与滞后对齐的例程（与后面曲线计算衔接）。


# # ══════════════════════════════════════════════════════════════════════════════
# # 14.4  Similarity metrics
# # ══════════════════════════════════════════════════════════════════════════════
# import numpy as np

# _TINY = 1e-12


# def binary_jaccard(v1: dict, v2: dict, topics: set,
#                    presence_min_share: float = 0.01) -> float:
#     """Binary Jaccard: fraction of topics present in *both* out of topics in *either*.

#     A topic is 'present' if its share >= presence_min_share in that country-year.
#     """
#     # 中文：二元 Jaccard：两集合交并比，强调「是否出现」而非权重。
#     s1 = {t for t in topics if v1.get(t, 0) >= presence_min_share}
#     s2 = {t for t in topics if v2.get(t, 0) >= presence_min_share}
#     union = s1 | s2
#     if not union:
#         return 0.0
#     return len(s1 & s2) / len(union)


# def weighted_jaccard(v1: dict, v2: dict, topics: set) -> float:
#     """Weighted Jaccard: sum(min(a,b)) / sum(max(a,b)) over shared topic universe."""
#     # 中文：加权 Jaccard：用最小份额之和归一，兼顾主题频率。
#     num = sum(min(v1.get(t, 0), v2.get(t, 0)) for t in topics)
#     den = sum(max(v1.get(t, 0), v2.get(t, 0)) for t in topics)
#     return num / (den + _TINY)


# def cosine_topic_similarity(v1: dict, v2: dict, topics: set) -> float:
#     """Cosine similarity between two topic-share vectors over the shared topic universe."""
#     # 中文：将主题份额视为向量，在共同主题子空间上算余弦相似度。
#     a = np.array([v1.get(t, 0) for t in topics])
#     b = np.array([v2.get(t, 0) for t in topics])
#     denom = (np.linalg.norm(a) * np.linalg.norm(b)) + _TINY
#     return float(np.dot(a, b) / denom)


# # ── Self-tests ─────────────────────────────────────────────────────────────
# _v1 = {"A": 0.5, "B": 0.3, "C": 0.2}
# _v2 = {"A": 0.5, "B": 0.3, "C": 0.2}
# assert abs(binary_jaccard(_v1, _v2, {"A","B","C"}) - 1.0) < 1e-9, "binary_jaccard identical"
# assert abs(weighted_jaccard(_v1, _v2, {"A","B","C"}) - 1.0) < 1e-9, "weighted_jaccard identical"
# assert abs(cosine_topic_similarity(_v1, _v2, {"A","B","C"}) - 1.0) < 1e-9, "cosine identical"

# _v3 = {"A": 1.0}; _v4 = {"B": 1.0}
# assert binary_jaccard(_v3, _v4, {"A","B"}) == 0.0, "binary_jaccard disjoint"
# assert weighted_jaccard(_v3, _v4, {"A","B"}) < 1e-9, "weighted_jaccard disjoint"
# print("✔ similarity metric self-tests passed")


## 14.5  Compute four-panel lead–lag curves

**【中文说明】**
- 对每个 CN 年、每个候选滞后 Δt，计算与对应 US 年的相似度曲线，是四宫格图的核心输入。


In [ ]:
# # 【单元格中文说明】
# # - 对每个 CN 年、每个候选 lag，将 CN 向量与“US 在 t+lag 年”向量比对，形成 similarity–lag 曲线表 overlap_curves_all。


# # ══════════════════════════════════════════════════════════════════════════════
# # 14.5  Compute four-panel lead–lag curves
# # ══════════════════════════════════════════════════════════════════════════════

# def _build_country_year_long(df_in, country_col, year_col, topic_col,
#                               min_docs: int = 5) -> "pd.DataFrame":
#     """Aggregate to (country, year, topic) with doc counts and share."""
#     # 中文：由文献级表聚合为 (国,年,主题) 长表，并附 n_docs、topic_share。
#     grp = (df_in.groupby([country_col, year_col, topic_col])
#                 .size().reset_index(name="n_docs"))
#     total = grp.groupby([country_col, year_col])["n_docs"].transform("sum")
#     grp["topic_share"] = grp["n_docs"] / total.clip(lower=1)
#     # Drop (country, year) pairs with too few docs
#     cy_total = grp.groupby([country_col, year_col])["n_docs"].transform("sum")
#     grp = grp[cy_total >= min_docs].copy()
#     return grp


# def _roll_country_year_long(cy_long, window: int) -> "pd.DataFrame":
#     """Rolling-average the topic_share over `window` years per (country, topic)."""
#     # 中文：对长表按国、年做滚动窗口平滑，再归一化得 rolling 份额。
#     if window <= 1:
#         return cy_long.copy()
#     out_rows = []
#     for (country, topic), sub in cy_long.groupby([cy_long.columns[0],
#                                                    cy_long.columns[2]]):
#         sub_s = sub.sort_values(cy_long.columns[1])
#         rolled = sub_s["topic_share"].rolling(window, min_periods=1).mean()
#         sub_s = sub_s.copy()
#         sub_s["topic_share"] = rolled.values
#         out_rows.append(sub_s)
#     return pd.concat(out_rows, ignore_index=True) if out_rows else cy_long.copy()


# def _build_vectors_from_long(cy_long, country_col, year_col, topic_col) -> dict:
#     """Return nested dict vecs[country][year] = {topic: share}."""
#     # 中文：将长表折叠为 topic_sets / share_vectors / binary_vectors 嵌套字典。
#     vecs: dict = {}
#     for (country, year), grp in cy_long.groupby([country_col, year_col]):
#         vecs.setdefault(country, {})[year] = dict(zip(grp[topic_col],
#                                                        grp["topic_share"]))
#     return vecs


# def _get_common_topics(v1: dict, v2: dict, presence_min_share: float) -> set:
#     # 中文：取两向量在足够份额下的主题并集或交集支撑，用于相似度计算。
#     # 中文：取两向量在足够份额下的主题并集或交集支撑，用于相似度计算。
#     t1 = {t for t, s in v1.items() if s >= presence_min_share}
#     t2 = {t for t, s in v2.items() if s >= presence_min_share}
#     return t1 | t2


# def compute_overlap_panel_curves(vecs: dict, lag_range, metrics: list,
#                                   presence_min_share: float,
#                                   min_docs_per_cy: int = 5) -> "pd.DataFrame":
#     """Compute similarity(cn_year, US_year=cn_year+lag) for every lag, metric."""
#     # 中文：对 lag 网格与多种 metric 扫描，输出面板曲线长表。
#     if "CN" not in vecs or "US" not in vecs:
#         print("⚠  'CN' and/or 'US' not found in vecs; check country labels.")
#         return pd.DataFrame()
#     cn_vecs = vecs["CN"]
#     us_vecs = vecs["US"]
#     rows = []
#     for cn_y in sorted(cn_vecs):
#         cn_v = cn_vecs[cn_y]
#         for lag in lag_range:
#             us_y = cn_y + lag
#             if us_y not in us_vecs:
#                 continue
#             us_v = us_vecs[us_y]
#             topics = _get_common_topics(cn_v, us_v, presence_min_share)
#             if not topics:
#                 continue
#             for m in metrics:
#                 if m == "weighted_jaccard":
#                     sim = weighted_jaccard(cn_v, us_v, topics)
#                 elif m == "binary_jaccard":
#                     sim = binary_jaccard(cn_v, us_v, topics, presence_min_share)
#                 elif m == "cosine":
#                     sim = cosine_topic_similarity(cn_v, us_v, topics)
#                 else:
#                     continue
#                 rows.append(dict(cn_year=cn_y, us_year=us_y, lag=lag,
#                                  metric=m, similarity=sim))
#     return pd.DataFrame(rows)


# # ── Build the curves across all metric × rolling_window combinations ──────────
# _cy_long_full = _build_country_year_long(
#     _df_ol, COUNTRY_COL_OL, YEAR_COL_OL, TOPIC_COL_OL, MIN_DOCS_PER_COUNTRY_YEAR
# )
# print(f"cy_long: {len(_cy_long_full):,} rows  "
#       f"({_cy_long_full[COUNTRY_COL_OL].nunique()} countries, "
#       f"{_cy_long_full[YEAR_COL_OL].nunique()} years)")

# _all_metrics = (
#     (["weighted_jaccard"] if USE_WEIGHTED_JACCARD else [])
#   + (["binary_jaccard"]   if RUN_BINARY_JACCARD   else [])
#   + (["cosine"]           if RUN_COSINE            else [])
# )

# _curve_chunks = []
# for _rw in ROLLING_WINDOWS:
#     _cy_r = _roll_country_year_long(_cy_long_full, _rw)
#     _vecs = _build_vectors_from_long(_cy_r, COUNTRY_COL_OL, YEAR_COL_OL, TOPIC_COL_OL)
#     _chunk = compute_overlap_panel_curves(
#         _vecs, LAG_RANGE, _all_metrics, PRESENCE_MIN_SHARE, MIN_DOCS_PER_COUNTRY_YEAR
#     )
#     if not _chunk.empty:
#         _chunk["rolling_window"] = _rw
#         _curve_chunks.append(_chunk)
#     print(f"  roll={_rw}  →  {len(_chunk):,} rows")

# overlap_curves_all = pd.concat(_curve_chunks, ignore_index=True) if _curve_chunks else pd.DataFrame()
# print(f"\noverlap_curves_all: {len(overlap_curves_all):,} rows  "
#       f"(metrics={overlap_curves_all['metric'].unique() if not overlap_curves_all.empty else []}, "
#       f"windows={sorted(overlap_curves_all['rolling_window'].unique()) if not overlap_curves_all.empty else []})")

# overlap_curves_all.to_csv(OVERLAP_TAB_DIR / "overlap_curves_all.csv", index=False)
# print("✔ overlap_curves_all.csv")


## 14.6  Plot four-panel figure

**【中文说明】**
- 将相似度–滞后曲线画成四面板：多 lag 曲线、最佳滞后随时间、最佳相似度随时间等，对应论文中常见 Figure 11 风格。


In [ ]:
# # 【单元格中文说明】
# # - plot_overlap_four_panel：绘制四宫格综述图（多 lag 曲线、最佳滞后趋势等），并保存 PNG/PDF。


# # ══════════════════════════════════════════════════════════════════════════════
# # 14.6  Plot four-panel figure
# # ══════════════════════════════════════════════════════════════════════════════
# import matplotlib.pyplot as plt
# import matplotlib.cm as _cm
# import numpy as np

# _PANEL_TITLES = {
#     "a": "Panel a — Topic-set overlap (Jaccard)",
#     "b": "Panel b — Cosine similarity",
#     "c": "Panel c — Best lag over time",
#     "d": "Panel d — Best similarity over time",
# }
# _XAXIS_LABELS = {
#     "a": "Lag Δt (years, positive = CN follows US)",
#     "b": "Lag Δt (years, positive = CN follows US)",
#     "c": "CN publication year",
#     "d": "CN publication year",
# }
# _METRIC_LABELS = {
#     "weighted_jaccard": "Weighted Jaccard",
#     "binary_jaccard":   "Binary Jaccard",
#     "cosine":           "Cosine similarity",
# }


# def plot_overlap_four_panel(curves_df, best_df, metric: str,
#                              rolling_window: int, save_dir, tag: str = ""):
#     """Produce the four-panel lead–lag figure for one (metric, rolling_window) spec."""
#     # 中文：四宫格：相似度-滞后曲线、最佳滞后/最佳相似度随 CN 年变化。
#     _sub = curves_df.query("metric == @metric and rolling_window == @rolling_window")
#     if {"metric", "rolling_window"}.issubset(best_df.columns):
#         _best = best_df.query("metric == @metric and rolling_window == @rolling_window")
#     else:
#         _best = pd.DataFrame(columns=["cn_year", "best_lag", "best_sim"])
#     if _sub.empty:
#         print(f"  ⚠  No data for {metric}, roll={rolling_window}")
#         return

#     fig, axes = plt.subplots(2, 2, figsize=(14, 9))
#     axes = axes.flatten()

#     years_sorted = sorted(_sub["cn_year"].unique())
#     _cmap = _cm.viridis
#     _colours = {y: _cmap(i / max(len(years_sorted) - 1, 1))
#                 for i, y in enumerate(years_sorted)}

#     # ── Panels a & b: similarity curve per year across lags ──────────────────
#     for _panel_idx, _metric_key in enumerate([metric, metric]):
#         ax = axes[_panel_idx]
#         for cy in years_sorted:
#             _row = _sub[_sub["cn_year"] == cy].sort_values("lag")
#             if _row.empty:
#                 continue
#             ax.plot(_row["lag"], _row["similarity"], color=_colours[cy],
#                     linewidth=0.9, alpha=0.8, label=str(cy))
#         ax.axvline(0, color="black", linestyle="--", linewidth=0.8)
#         ax.set_xlabel(_XAXIS_LABELS["a"])
#         ax.set_ylabel(_METRIC_LABELS.get(metric, metric))
#         ax.set_title((_PANEL_TITLES["a"] if _panel_idx == 0
#                       else _PANEL_TITLES["b"])
#                      + f"\n[{_METRIC_LABELS.get(metric, metric)}, roll={rolling_window}]",
#                      fontsize=9)
#         # Colour bar proxy
#         sm = plt.cm.ScalarMappable(cmap=_cmap,
#                                    norm=plt.Normalize(vmin=min(years_sorted),
#                                                       vmax=max(years_sorted)))
#         sm.set_array([])
#         plt.colorbar(sm, ax=ax, label="CN year")
#         if _panel_idx == 1:
#             break  # only fill panel b once — duplicate removed below

#     # ── Panel c: best lag trend ───────────────────────────────────────────────
#     _bs = _best.sort_values("cn_year") if not _best.empty else _best
#     if _bs.empty:
#         axes[2].text(0.5, 0.5, "No best-lag data", ha="center", va="center",
#                     transform=axes[2].transAxes, fontsize=9, color="gray")
#     else:
#         axes[2].plot(_bs["cn_year"], _bs["best_lag"], marker="o", color="steelblue")
#     axes[2].axhline(0, color="gray", linestyle="--", linewidth=0.8)
#     axes[2].set_xlabel(_XAXIS_LABELS["c"])
#     axes[2].set_ylabel("Best lag Δt* (years)")
#     axes[2].set_title(_PANEL_TITLES["c"]
#                       + f"\n[{_METRIC_LABELS.get(metric, metric)}, roll={rolling_window}]",
#                       fontsize=9)

#     # ── Panel d: best similarity trend ───────────────────────────────────────
#     if _bs.empty:
#         axes[3].text(0.5, 0.5, "No best-sim data", ha="center", va="center",
#                     transform=axes[3].transAxes, fontsize=9, color="gray")
#     else:
#         axes[3].plot(_bs["cn_year"], _bs["best_sim"], marker="s", color="darkorange")
#     axes[3].set_xlabel(_XAXIS_LABELS["d"])
#     axes[3].set_ylabel("Similarity at best lag")
#     axes[3].set_title(_PANEL_TITLES["d"]
#                       + f"\n[{_METRIC_LABELS.get(metric, metric)}, roll={rolling_window}]",
#                       fontsize=9)

#     fig.suptitle(
#         f"BERTopic-overlap lead–lag CN vs US  |  "
#         f"{_METRIC_LABELS.get(metric, metric)}  |  rolling={rolling_window}y",
#         fontsize=11, fontweight="bold"
#     )
#     fig.tight_layout(rect=[0, 0, 1, 0.96])

#     _stem = f"overlap_leadlag_{metric}_roll{rolling_window}{tag}"
#     _png  = save_dir / f"{_stem}.png"
#     _pdf  = save_dir / f"{_stem}.pdf"
#     fig.savefig(_png, dpi=150, bbox_inches="tight")
#     fig.savefig(_pdf,           bbox_inches="tight")
#     plt.show()
#     print(f"✔ {_png.name}  |  {_pdf.name}")


# # ── Generate figures ──────────────────────────────────────────────────────────
# _fig_specs = (
#     [("weighted_jaccard", w) for w in ROLLING_WINDOWS if USE_WEIGHTED_JACCARD]
#   + [("binary_jaccard",   w) for w in ROLLING_WINDOWS if RUN_BINARY_JACCARD]
#   + [("cosine",           w) for w in ROLLING_WINDOWS if RUN_COSINE]
# )

# for _metric, _win in _fig_specs:
#     plot_overlap_four_panel(overlap_curves_all, overlap_best
#                             if "overlap_best" in dir() else
#                             _extract_best_lag(overlap_curves_all)
#                             if "_extract_best_lag" in dir() else
#                             pd.DataFrame(),
#                             metric=_metric, rolling_window=_win,
#                             save_dir=OVERLAP_FIG_DIR)


## 14.7  Best-lag extraction and trend summary

**【中文说明】**
- 从曲线中提取每个 CN 年的 **argmax 滞后** 与对应最高相似度，并保存为表，用于画趋势与写结论。


In [ ]:
# # 【单元格中文说明】
# # - 从曲线中提取 argmax lag 作为 best_lag，并绘制随 CN 年变化的最佳滞后与最佳相似度。


# # ══════════════════════════════════════════════════════════════════════════════
# # 14.7  Best-lag extraction and trend summary
# # ══════════════════════════════════════════════════════════════════════════════

# def _extract_best_lag(curves_df: "pd.DataFrame") -> "pd.DataFrame":
#     """Return, per (metric, rolling_window, cn_year), the lag with highest similarity."""
#     # 中文：对每个 (metric, rolling_window, cn_year) 取 similarity 最大的 lag 作为 best_lag。
#     best = (
#         curves_df
#         .loc[curves_df.groupby(["metric", "rolling_window", "cn_year"])["similarity"].idxmax()]
#         .rename(columns={"lag": "best_lag", "similarity": "best_sim"})
#         .reset_index(drop=True)
#     )
#     return best

# overlap_best = _extract_best_lag(overlap_curves_all)
# print(f"best-lag table: {len(overlap_best)} rows  "
#       f"(metrics={overlap_best['metric'].unique()}, "
#       f"windows={sorted(overlap_best['rolling_window'].unique())})")

# # ── Save summaries ─────────────────────────────────────────────────────────
# overlap_best.to_csv(OVERLAP_TAB_DIR / "best_lag_summary.csv",       index=False)
# overlap_best[["metric","rolling_window","cn_year","best_sim"]].to_csv(
#     OVERLAP_TAB_DIR / "best_similarity_summary.csv", index=False)
# overlap_curves_all.to_csv(OVERLAP_TAB_DIR / "overlap_panel_summary.csv", index=False)
# print("✔ best_lag_summary.csv | best_similarity_summary.csv | overlap_panel_summary.csv")

# # ── Trend plots ────────────────────────────────────────────────────────────
# import matplotlib.pyplot as plt

# for _m, _grp in overlap_best.groupby("metric"):
#     fig, axes = plt.subplots(1, 2, figsize=(12, 4))
#     for _w, _sg in _grp.groupby("rolling_window"):
#         _sg_s = _sg.sort_values("cn_year")
#         axes[0].plot(_sg_s["cn_year"], _sg_s["best_lag"],  marker="o", label=f"roll={_w}")
#         axes[1].plot(_sg_s["cn_year"], _sg_s["best_sim"],  marker="s", label=f"roll={_w}")
#     axes[0].axhline(0, color="gray", linestyle="--", linewidth=0.8)
#     axes[0].set_title(f"Best lag over time  [{_m}]")
#     axes[0].set_xlabel("CN publication year")
#     axes[0].set_ylabel("Best lag Δt* (years)")
#     axes[0].legend(fontsize=8)
#     axes[1].set_title(f"Best similarity over time  [{_m}]")
#     axes[1].set_xlabel("CN publication year")
#     axes[1].set_ylabel("Similarity at best lag")
#     axes[1].legend(fontsize=8)
#     fig.tight_layout()
#     _fn = OVERLAP_FIG_DIR / f"best_lag_over_time_{_m}.png"
#     fig.savefig(_fn, dpi=150, bbox_inches="tight")
#     plt.show()
#     print(f"✔ {_fn.name}")


## 14.8  Robustness: rolling window / weighted overlap / frontier-only

**【中文说明】**
- 更换滚动窗口、仅前沿子样本等设定，检验 lead–lag 结论是否依赖特定平滑或样本定义。


In [ ]:
# # 【单元格中文说明】
# # - 对多种 metric×window×样本（全样本/前沿）重复计算 best_lag，汇总为 robustness_overlap_summary.csv。


# # ══════════════════════════════════════════════════════════════════════════════
# # 14.8  Robustness: rolling window / weighted overlap / frontier-only
# # ══════════════════════════════════════════════════════════════════════════════

# _rob_specs = []
# for _m in (["weighted_jaccard"] * RUN_BINARY_JACCARD.__class__  # keep linter happy
#            if False else
#            (["weighted_jaccard"] if USE_WEIGHTED_JACCARD else [])
#            + (["binary_jaccard"]  if RUN_BINARY_JACCARD   else [])
#            + (["cosine"]          if RUN_COSINE            else [])):
#     for _w in ROLLING_WINDOWS:
#         _rob_specs.append(dict(metric=_m, rolling_window=_w, sample_type="all"))

# # Optional frontier-only spec
# if FRONTIER_ONLY and _frontier_flag_col_ol in df.columns:
#     for _m in (["weighted_jaccard"] if USE_WEIGHTED_JACCARD else []):
#         _rob_specs.append(dict(metric=_m, rolling_window=3, sample_type="frontier"))

# _robustness_rows = []
# for _spec in _rob_specs:
#     _m, _w, _st = _spec["metric"], _spec["rolling_window"], _spec["sample_type"]
#     _df_sample = (
#         _df_ol[_df_ol[_frontier_flag_col_ol] == 1]
#         if _st == "frontier" and _frontier_flag_col_ol in _df_ol.columns
#         else _df_ol
#     )
#     _cy_l = _build_country_year_long(_df_sample, COUNTRY_COL_OL, YEAR_COL_OL,
#                                       TOPIC_COL_OL, MIN_DOCS_PER_COUNTRY_YEAR)
#     _cy_r = _roll_country_year_long(_cy_l, _w)
#     _vecs = _build_vectors_from_long(_cy_r, COUNTRY_COL_OL, YEAR_COL_OL, TOPIC_COL_OL)
#     _rob_c = compute_overlap_panel_curves(_vecs, LAG_RANGE, [_m], PRESENCE_MIN_SHARE,
#                                            MIN_DOCS_PER_COUNTRY_YEAR)
#     if _rob_c.empty:
#         continue
#     _rb = _rob_c.copy()
#     _rb["rolling_window"] = _w
#     _rb["sample_type"]    = _st
#     _best = _rb.loc[_rb.groupby(["cn_year"])["similarity"].idxmax()].copy()
#     _best.rename(columns={"lag": "best_lag", "similarity": "best_sim"}, inplace=True)
#     for _, row in _best.iterrows():
#         _robustness_rows.append({
#             "metric": _m, "rolling_window": _w, "sample_type": _st,
#             "cn_year": row["cn_year"], "best_lag": row["best_lag"],
#             "best_sim": row["best_sim"],
#         })
#     print(f"  robustness spec metric={_m}, roll={_w}, sample={_st}  → {len(_best)} years")

# robustness_ol = pd.DataFrame(_robustness_rows)
# if not robustness_ol.empty:
#     robustness_ol.to_csv(OVERLAP_TAB_DIR / "robustness_overlap_summary.csv", index=False)
#     print(f"✔ robustness_overlap_summary.csv  ({len(robustness_ol)} rows, "
#           f"{robustness_ol['metric'].nunique()} metrics × "
#           f"{robustness_ol['rolling_window'].nunique()} windows)")
# else:
#     print("⚠  No robustness rows — check ROLLING_WINDOWS / metric flags.")


## 14.9  Export tables and manifest

**【中文说明】**
- 导出首选设定下的曲线与 best-lag 表，并打印文件清单，便于与 `time_evolution/` 结果一并归档。


In [ ]:
# # 【单元格中文说明】
# # - 导出“首选”设定下的曲线与 best-lag 子表；打印 overlap_leadlag 目录文件清单。


# # ══════════════════════════════════════════════════════════════════════════════
# # 14.9  Export tables and manifest
# # ══════════════════════════════════════════════════════════════════════════════

# # ── Preferred spec subsets ────────────────────────────────────────────────────
# _pref_metric   = "weighted_jaccard" if USE_WEIGHTED_JACCARD else (
#                   "binary_jaccard"  if RUN_BINARY_JACCARD   else "cosine")
# _pref_window   = 3  # moderate smoothing default

# _pref_curves = overlap_curves_all.query(
#     "metric == @_pref_metric and rolling_window == @_pref_window"
# ).copy()
# _pref_curves.to_csv(OVERLAP_TAB_DIR / "overlap_curves_preferred.csv", index=False)
# print(f"✔ overlap_curves_preferred.csv  ({len(_pref_curves)} rows)")

# _pref_best = overlap_best.query(
#     "metric == @_pref_metric and rolling_window == @_pref_window"
# ).copy()
# _pref_best.to_csv(OVERLAP_TAB_DIR / "best_lag_preferred.csv", index=False)
# print(f"✔ best_lag_preferred.csv  ({len(_pref_best)} rows)")

# if "year" not in _pref_best.columns and "cn_year" in _pref_best.columns:
#     _pref_best = _pref_best.copy()
#     _pref_best["year"] = _pref_best["cn_year"]
# if "us_year" not in _pref_best.columns and {"cn_year", "best_lag"}.issubset(_pref_best.columns):
#     _pref_best = _pref_best.copy()
#     _pref_best["us_year"] = _pref_best["cn_year"] + _pref_best["best_lag"]
# _pref_sim_cols = [c for c in ["year", "cn_year", "us_year", "best_sim"] if c in _pref_best.columns]
# _pref_sim = _pref_best[_pref_sim_cols].copy()
# _pref_sim.to_csv(OVERLAP_TAB_DIR / "best_similarity_preferred.csv", index=False)
# print(f"✔ best_similarity_preferred.csv  ({len(_pref_sim)} rows)")

# # ── Print manifest ────────────────────────────────────────────────────────────
# print(f"\n📁 Files under {OVERLAP_DIR}:")
# _indent = "  "
# for _p in sorted(OVERLAP_DIR.rglob("*")):
#     if _p.is_file():
#         _rel = _p.relative_to(OVERLAP_DIR)
#         _size_kb = _p.stat().st_size / 1024
#         print(f"{_indent}{_rel}  ({_size_kb:.1f} KB)")

# print("\n🎉 §14 complete — BERTopic-overlap lead–lag analysis done!")


## 14.10  How to read the figures

| Panel | x-axis | y-axis | What "positive lag" means |
|-------|--------|--------|---------------------------|
| **a** Topic-set overlap (Jaccard) vs lag | Δt (years, CN relative to US) | Weighted/binary Jaccard ∈ [0,1] | CN research at year t is compared to US research at year t+Δt; positive Δt → CN follows US |
| **b** Cosine similarity vs lag | Δt (years) | Cosine similarity ∈ [0,1] | Same interpretation; cosine weights topic frequencies |
| **c** Best lag over time | Publication year | Best-lag Δt* (argmax similarity) | Trend toward 0 / negative = catch-up / leapfrog |
| **d** Best similarity over time | Publication year | Similarity at best lag | Trend up = increasing alignment, down = divergence |

**Robustness handles**
- `rolling_window` — 1 (raw), 3 (moderate smoothing), 5 (heavy smoothing)  
- `metric` — `weighted_jaccard` (main), `binary_jaccard`, `cosine`  
- `sample_type` — `all` (all papers), `frontier` (top-10 % cited only, if flag column available)

**【中文说明】**
- 解读四宫格各子图坐标与“正滞后”含义：正值多表示中国该年分布更接近美国 **之后** 若干年的分布，即相对滞后。


# Ⅳ. 技术版图 + 结构洞察

## §15 Topic Map — CN/US Dominance Overlay
Generate a custom Plotly interactive scatter where each dot is a topic, coloured by CN share and sized by paper count.

## §16 US-only Topics Deep Dive
For topics exclusively claimed by US (default: 21, 43), produce a diagnostic material pack to distinguish **true capability gaps** from **cluster split effects / local boundary artefacts**.

**【中文说明】**
- **技术版图**：在主题嵌入的 UMAP 平面上用颜色表示中美主导程度、点大小表示论文规模，将结构与国别结合起来看。
- **仅美国出现的主题**：生成诊断包，区分真实空白与聚类边界/拆分伪影。


In [ ]:
# 【单元格中文说明】
# - build_topic_stats：每主题统计中美篇数、占比、dominance=2*CN_share−1；make_topic_map：主题嵌入 UMAP 二维 + Plotly 交互散点。
# - 颜色表示中国在该主题内占比，点大小表示全球该主题论文数；粗边框表示中美一侧主导（>60%）。


# ══════════════════════════════════════════════════════════════════════════════
# §15  Topic Map — CN/US Dominance Overlay
# ══════════════════════════════════════════════════════════════════════════════
import plotly.graph_objects as go
from sklearn.metrics.pairwise import cosine_similarity as _cosine_sim

# ── 14.0 Fallback loaders — ensure key variables exist ───────────────────────
if "TOPIC_COL" not in dir() or TOPIC_COL is None:
    for _c in ["topic_reduced", "topic_reassign", "topic"]:
        if _c in df.columns and df[_c].notna().any():
            TOPIC_COL = _c
            break
print(f"TOPIC_COL = '{TOPIC_COL}'")

COUNTRY_COL_14 = "country2" if "country2" in df.columns else "country_code"
if COUNTRY_COL_14 == "country_code" and "country2" not in df.columns:
    df["country2"] = df["country_code"].map(
        lambda c: "CN" if c == "CN" else ("US" if c == "US" else "OTHER")
    )
    COUNTRY_COL_14 = "country2"

if "topic_model" not in dir() or topic_model is None:
    from bertopic import BERTopic as _BT
    topic_model = _BT.load(os.path.join(OUTPUT_DIR, "bertopic_model"))
    print("✅ Loaded BERTopic model from disk")

_repr_path_14 = os.path.join(OUTPUT_DIR, "representative_docs.json")
if "repr_docs" not in dir() or repr_docs is None:
    if os.path.exists(_repr_path_14):
        with open(_repr_path_14, "r", encoding="utf-8") as f:
            repr_docs = json.load(f)

_trend_path_14 = os.path.join(OUTPUT_DIR, "time_evolution", "topic_trend_summary.csv")
_trend_14 = None
if "trend_summary" in dir() and trend_summary is not None:
    _trend_14 = trend_summary.copy()
elif os.path.exists(_trend_path_14):
    _trend_14 = pd.read_csv(_trend_path_14)
    print("✅ Loaded topic_trend_summary.csv")

# ── 14.1 build_topic_stats() ────────────────────────────────────────────────
def build_topic_stats(df, topic_col, country_col, topic_model_obj):
    """Per-topic metrics: CN/US counts, shares, dominance, keywords."""
    # 中文：按主题统计中美篇数、占比、dominance，并拉取关键词用于悬停。
    valid_mask = df[topic_col].notna()
    if (df[topic_col] == -1).any():
        valid_mask = valid_mask & (df[topic_col] != -1)
    df_work = df[valid_mask & df[country_col].isin(["CN", "US"])].copy()
    ct = df_work.groupby([topic_col, country_col]).size().unstack(fill_value=0)
    for c in ["CN", "US"]:
        if c not in ct.columns:
            ct[c] = 0
    ct = ct[["CN", "US"]].copy()
    ct.columns = ["count_CN", "count_US"]
    ct["count_total_CNUS"] = ct["count_CN"] + ct["count_US"]
    ct["CN_share_in_topic"] = ct["count_CN"] / ct["count_total_CNUS"].replace(0, np.nan)
    ct["US_share_in_topic"] = ct["count_US"] / ct["count_total_CNUS"].replace(0, np.nan)
    ct["dominance"] = 2 * ct["CN_share_in_topic"] - 1

    # Total count across ALL countries (for dot size)
    valid_all = df[topic_col].notna()
    if (df[topic_col] == -1).any():
        valid_all = valid_all & (df[topic_col] != -1)
    total_all = df[valid_all].groupby(topic_col).size().rename("topic_total_count")
    ct = ct.join(total_all, how="left")
    ct["topic_total_count"] = ct["topic_total_count"].fillna(ct["count_total_CNUS"])

    # Keywords (top 15)
    kw_list = []
    for tid in ct.index:
        try:
            kws = topic_model_obj.get_topic(tid)
            kw_list.append(", ".join([w for w, _ in kws[:15]]) if kws else "")
        except Exception:
            kw_list.append("")
    ct["keywords"] = kw_list
    ct.index.name = "topic_id"
    return ct.reset_index()


# ── 14.2 make_topic_map() ───────────────────────────────────────────────────
def make_topic_map(topic_stats, topic_model_obj, trend_df=None, seed=42, output_dir=OUTPUT_DIR, top_label_n=10):
    """Interactive Plotly topic map coloured by CN/US dominance."""
    # 中文：用 topic_embeddings_ 做 UMAP 二维散点，颜色=CN 占比，大小=全球发文量。
    # ── Topic embeddings ─────────────────────────────────────────────────
    te = getattr(topic_model_obj, "topic_embeddings_", None)
    if te is None:
        raise RuntimeError("topic_model.topic_embeddings_ not found; reload model with embedding support.")
    ti = topic_model_obj.get_topic_info()
    tid_list = ti["Topic"].tolist()

    valid_tids = set(topic_stats["topic_id"].tolist())
    mask_te = [t in valid_tids for t in tid_list]
    te_filt = te[mask_te]
    tids_filt = [t for t, m in zip(tid_list, mask_te) if m]

    # ── UMAP 2-D on topic embeddings ────────────────────────────────────
    n_nbrs = min(15, max(2, len(tids_filt) - 1))
    umap_te = UMAP(n_components=2, random_state=seed, n_neighbors=n_nbrs, min_dist=0.1)
    coords = umap_te.fit_transform(te_filt)

    plot_df = pd.DataFrame({"topic_id": tids_filt, "x": coords[:, 0], "y": coords[:, 1]}).merge(
        topic_stats, on="topic_id", how="left"
    )

    # ── Merge trend info (optional) ─────────────────────────────────────
    if trend_df is not None:
        _tc = []
        for c in ["slope_delta", "trend_label", "cross_year"]:
            if c in trend_df.columns:
                _tc.append(c)
        if _tc:
            _merge = trend_df[["topic"] + _tc].rename(columns={"topic": "topic_id"})
            plot_df = plot_df.merge(_merge, on="topic_id", how="left")

    # ── Visual encodings ────────────────────────────────────────────────
    plot_df["size_scaled"] = np.sqrt(plot_df["topic_total_count"].fillna(1)) * 2.5

    def _bw(cn_s):
        if pd.isna(cn_s):
            return 1
        return 3 if (cn_s > 0.6 or cn_s < 0.4) else 1

    plot_df["border_width"] = plot_df["CN_share_in_topic"].apply(_bw)

    # ── Hover text ──────────────────────────────────────────────────────
    hover_texts = []
    for _, r in plot_df.iterrows():
        parts = [
            f"<b>Topic {int(r['topic_id'])}</b>",
            f"Keywords: {str(r.get('keywords', ''))[:100]}",
            f"count_CN: {int(r.get('count_CN', 0))}",
            f"count_US: {int(r.get('count_US', 0))}",
            f"CN_share: {r.get('CN_share_in_topic', 0):.3f}",
            f"US_share: {r.get('US_share_in_topic', 0):.3f}",
            f"dominance: {r.get('dominance', 0):.3f}",
        ]
        if "slope_delta" in r.index and pd.notna(r.get("slope_delta")):
            parts.append(f"slope_delta: {r['slope_delta']:.2e}")
        if "trend_label" in r.index and pd.notna(r.get("trend_label")):
            parts.append(f"trend: {r['trend_label']}")
        if "cross_year" in r.index and pd.notna(r.get("cross_year")):
            parts.append(f"cross_year: {int(r['cross_year'])}")
        hover_texts.append("<br>".join(parts))

    # ── Labels for top-N or extreme dominance ───────────────────────────
    top_size = set(plot_df.nlargest(top_label_n, "topic_total_count")["topic_id"])
    top_dom = set(plot_df.nlargest(5, "dominance")["topic_id"])
    bot_dom = set(plot_df.nsmallest(5, "dominance")["topic_id"])
    label_set = top_size | top_dom | bot_dom
    text_labels = [f"T{int(r['topic_id'])}" if r["topic_id"] in label_set else "" for _, r in plot_df.iterrows()]

    # ── Main Plotly figure ──────────────────────────────────────────────
    fig = go.Figure(
        go.Scatter(
            x=plot_df["x"],
            y=plot_df["y"],
            mode="markers+text",
            marker=dict(
                size=plot_df["size_scaled"],
                color=plot_df["CN_share_in_topic"],
                colorscale="RdBu_r",
                cmin=0,
                cmax=1,
                colorbar=dict(title="CN share<br>CN/(CN+US)"),
                line=dict(width=plot_df["border_width"], color="black"),
            ),
            text=text_labels,
            textposition="top center",
            textfont=dict(size=9),
            hovertext=hover_texts,
            hoverinfo="text",
        )
    )
    fig.update_layout(
        title="Topic Map — CN/US Dominance Overlay<br><sub>Color: CN share (red=CN, blue=US) | Size: total papers | Thick border: dominant (>0.6 or <0.4)</sub>",
        xaxis_title="UMAP-1",
        yaxis_title="UMAP-2",
        width=1200,
        height=900,
        template="plotly_white",
    )

    html_p = os.path.join(output_dir, "topic_map_country.html")
    fig.write_html(html_p)
    print(f"✅ Saved {html_p}")

    try:
        png_p = os.path.join(output_dir, "topic_map_country.png")
        fig.write_image(png_p, scale=2)
        print(f"✅ Saved {png_p}")
    except Exception as e:
        print(f"⚠️  PNG export failed ({e}). Try: pip install -U kaleido")

    # ── Dominant-only static version ────────────────────────────────────
    dom_mask = (plot_df["CN_share_in_topic"] > 0.6) | (plot_df["CN_share_in_topic"] < 0.4)
    dom_df = plot_df[dom_mask].copy()
    if len(dom_df) > 0:
        fig_dom = go.Figure(
            go.Scatter(
                x=dom_df["x"],
                y=dom_df["y"],
                mode="markers+text",
                marker=dict(
                    size=np.sqrt(dom_df["topic_total_count"].fillna(1)) * 2.5,
                    color=dom_df["CN_share_in_topic"],
                    colorscale="RdBu_r",
                    cmin=0,
                    cmax=1,
                    colorbar=dict(title="CN share"),
                    line=dict(width=2, color="black"),
                ),
                text=[f"T{int(t)}" for t in dom_df["topic_id"]],
                textposition="top center",
                textfont=dict(size=9),
                hovertext=[hover_texts[i] for i in dom_df.index],
                hoverinfo="text",
            )
        )
        fig_dom.update_layout(
            title="Topic Map — Dominant Topics Only (CN>0.6 or US>0.6)",
            xaxis_title="UMAP-1",
            yaxis_title="UMAP-2",
            width=1200,
            height=900,
            template="plotly_white",
        )
        try:
            dom_png = os.path.join(output_dir, "topic_map_country_dominant.png")
            fig_dom.write_image(dom_png, scale=2)
            print(f"✅ Saved {dom_png}")
        except Exception as e:
            print(f"⚠️  Dominant PNG failed ({e})")

    try:
        fig.show()
    except Exception as _e:
        print(
            f"⚠️  fig.show() failed ({_e})\n"
            "    Fix: pip install -U nbformat\n"
            "    Falling back to IFrame display of saved HTML …"
        )
        try:
            from IPython.display import IFrame, display
            _html_p = os.path.join(output_dir, "topic_map_country.html")
            display(IFrame(src=_html_p, width="100%", height=900))
        except Exception:
            print("    Open topic_map_country.html in a browser to view the chart.")
    return plot_df


# ── 14.3 Execute ─────────────────────────────────────────────────────────────
topic_stats_14 = build_topic_stats(df, TOPIC_COL, COUNTRY_COL_14, topic_model)
topic_stats_14.to_csv(os.path.join(OUTPUT_DIR, "topic_stats_country.csv"), index=False)
print(f"✅ Exported topic_stats_country.csv ({len(topic_stats_14)} topics)")

plot_df_14 = make_topic_map(
    topic_stats_14,
    topic_model,
    trend_df=_trend_14,
    seed=SEED,
    output_dir=OUTPUT_DIR,
    top_label_n=10,
)
print("\n🎉 §15 complete — topic map with country overlay generated.")


## §15 US-only Topics — Deep Dive & Diagnostic

For each US-only topic `tid`:
1. **Material pack** (Markdown): keywords, representative docs, core sources, time trend, gap strength
2. **Charts**: yearly CN/US counts, roll5 share curves
3. **Diagnostics**: adjacency (cosine-sim to neighbours), confidence (surrogate `topic_prob`), robustness (Agglomerative local re-clustering)

**【中文说明】**
- 对指定 `US_ONLY_TIDS` 主题导出 Markdown 报告：关键词、代表文献、期刊机构、时序与邻域相似度、主题置信度与局部重聚类稳定性。
- 用于质性解释与稳健性说明，避免仅凭“无 CN 论文”下结论。


In [ ]:
# # 【单元格中文说明】
# # - export_us_only_topic_pack：对 US_ONLY_TIDS 逐个生成 Markdown+图表；邻域高相似主题提示“拆分伪影”风险；UMAP 重聚类检查局部稳定性。


# # ══════════════════════════════════════════════════════════════════════════════
# # §16  US-only Topics — Deep Dive & Diagnostic
# # ══════════════════════════════════════════════════════════════════════════════

# US_ONLY_TIDS = [21, 43]  # ← edit as needed
# US_DIR = os.path.join(OUTPUT_DIR, "us_only_topics")
# os.makedirs(US_DIR, exist_ok=True)

# # ── 15.0 Ensure auxiliary data ───────────────────────────────────────────────
# _yearly_path_15 = os.path.join(OUTPUT_DIR, "time_evolution", "topic_share_yearly.csv")
# _roll5_path_15 = os.path.join(OUTPUT_DIR, "time_evolution", "topic_share_roll5.csv")
# _ll_path_15 = os.path.join(OUTPUT_DIR, "time_evolution", "topic_lead_lag.csv")
# _trend_path_15 = os.path.join(OUTPUT_DIR, "time_evolution", "topic_trend_summary.csv")
# _repr_path_15 = os.path.join(OUTPUT_DIR, "representative_docs.json")

# _yearly_15 = pd.read_csv(_yearly_path_15) if os.path.exists(_yearly_path_15) else (
#     yearly.copy() if "yearly" in dir() and yearly is not None else None
# )
# _roll5_15 = pd.read_csv(_roll5_path_15) if os.path.exists(_roll5_path_15) else (
#     roll5.copy() if "roll5" in dir() and roll5 is not None else None
# )
# _ll_15 = pd.read_csv(_ll_path_15) if os.path.exists(_ll_path_15) else (
#     lead_lag_df.copy() if "lead_lag_df" in dir() and lead_lag_df is not None else None
# )
# _trend_15 = pd.read_csv(_trend_path_15) if os.path.exists(_trend_path_15) else (
#     trend_summary.copy() if "trend_summary" in dir() and trend_summary is not None else None
# )
# _repr_15 = None
# if "repr_docs" in dir() and repr_docs is not None:
#     _repr_15 = repr_docs
# elif os.path.exists(_repr_path_15):
#     with open(_repr_path_15, "r", encoding="utf-8") as f:
#         _repr_15 = json.load(f)

# _has_prob = "topic_prob" in df.columns and df["topic_prob"].notna().any()

# # ── Topic embeddings & index mapping ─────────────────────────────────────────
# _te_15 = getattr(topic_model, "topic_embeddings_", None)
# _ti_15 = topic_model.get_topic_info()
# _tid_to_idx_15 = {t: i for i, t in enumerate(_ti_15["Topic"].tolist())}

# # ── UMAP reduced embeddings for local re-clustering ──────────────────────────
# _umap_emb_15 = None
# if hasattr(topic_model, "umap_model") and hasattr(topic_model.umap_model, "embedding_"):
#     _umap_emb_15 = topic_model.umap_model.embedding_
#     print(f"✅ UMAP embeddings from model: shape={_umap_emb_15.shape}")
# else:
#     try:
#         _umap_tmp = UMAP(
#             n_neighbors=UMAP_N_NEIGHBORS,
#             n_components=UMAP_N_COMPONENTS,
#             min_dist=UMAP_MIN_DIST,
#             metric=UMAP_METRIC,
#             random_state=SEED,
#             low_memory=True,
#         )
#         _umap_emb_15 = _umap_tmp.fit_transform(embeddings)
#         print(f"✅ Recomputed UMAP-{UMAP_N_COMPONENTS}D: shape={_umap_emb_15.shape}")
#     except Exception as _e:
#         print(f"⚠️  UMAP re-computation failed: {_e}; local robustness check will be skipped")


# def export_us_only_topic_pack(
#     tids,
#     df,
#     topic_col,
#     country_col,
#     topic_model_obj,
#     output_dir,
#     repr_docs_dict=None,
#     yearly_df=None,
#     roll5_df=None,
#     lead_lag_df_in=None,
#     trend_df=None,
#     te=None,
#     tid_to_idx=None,
#     umap_emb=None,
#     seed=42,
# ):
#     """Generate deep-dive material pack for a list of US-only topics."""
#     # 中文：为「仅美国出现」的主题生成 Markdown 报告、图表与邻域/重聚类诊断。
#     us_dir = os.path.join(output_dir, "us_only_topics")
#     os.makedirs(us_dir, exist_ok=True)

#     has_prob = "topic_prob" in df.columns and df["topic_prob"].notna().any()
#     valid_topic_mask = df[topic_col].notna()
#     if (df[topic_col] == -1).any():
#         valid_topic_mask = valid_topic_mask & (df[topic_col] != -1)

#     for tid in tids:
#         print(f"\n{'─'*60}")
#         print(f"  Topic {tid} — generating material pack …")
#         print(f"{'─'*60}")
#         md = [f"# US-only Topic {tid} — Deep Dive\n"]

#         # ── A1) Keywords ─────────────────────────────────────────────────
#         try:
#             kws = topic_model_obj.get_topic(tid)
#             kw_str = ", ".join([f"{w} ({s:.3f})" for w, s in kws[:15]]) if kws else "N/A"
#             kw_set_orig = set([w for w, _ in kws[:15]]) if kws else set()
#         except Exception:
#             kw_str = "N/A"
#             kw_set_orig = set()
#         md.append(f"## 1. Keywords (Top 15)\n\n{kw_str}\n")

#         # ── A2) Representative documents ─────────────────────────────────
#         md.append("## 2. Representative Documents\n")
#         reps_printed = False
#         if repr_docs_dict and str(tid) in repr_docs_dict:
#             reps = repr_docs_dict[str(tid)][:10]
#             if reps:
#                 for i, doc in enumerate(reps, 1):
#                     md.append(f"{i}. {str(doc)[:300]}\n")
#                 reps_printed = True
#         if not reps_printed:
#             df_tid = df[df[topic_col] == tid].copy()
#             if has_prob:
#                 df_tid = df_tid.sort_values("topic_prob", ascending=False)
#             elif "citation" in df_tid.columns:
#                 df_tid = df_tid.sort_values("citation", ascending=False)
#             for i, (_, row) in enumerate(df_tid.head(10).iterrows(), 1):
#                 _ti = str(row.get("title", ""))[:200]
#                 _ab = str(row.get("abstract", ""))[:200]
#                 _yr = row.get("year", "")
#                 _ct = row.get("citation", "")
#                 _co = row.get(country_col, "")
#                 md.append(f"{i}. **{_ti}** (year={_yr}, cit={_ct}, country={_co})\n   {_ab}…\n")
#             if len(df_tid) == 0:
#                 md.append("*(No documents found)*\n")

#         # ── A3) Core sources ─────────────────────────────────────────────
#         md.append("## 3. Core Sources\n")
#         df_tid_all = df[df[topic_col] == tid]
#         for col_name, label in [("SO", "Journals"), ("C1", "Institutions"), ("AU", "Authors")]:
#             if col_name in df.columns:
#                 vc = df_tid_all[col_name].dropna().value_counts().head(10)
#                 if len(vc) > 0:
#                     md.append(f"### {label} (Top 10)\n")
#                     for val, cnt in vc.items():
#                         md.append(f"- {val}: {cnt}")
#                     md.append("")

#         # ── A4) Time trend ───────────────────────────────────────────────
#         md.append("## 4. Time Trend\n")
#         if yearly_df is not None and "topic" in yearly_df.columns:
#             yt = yearly_df[yearly_df["topic"] == tid].sort_values("year")
#             if not yt.empty:
#                 md.append("### Yearly Count (CN vs US)\n")
#                 md.append("| Year | CN | US |")
#                 md.append("|------|-----|-----|")
#                 for _, r in yt.iterrows():
#                     cn_v = int(r.get("count_CN", r.get("CN", 0)))
#                     us_v = int(r.get("count_US", r.get("US", 0)))
#                     md.append(f"| {int(r['year'])} | {cn_v} | {us_v} |")
#                 md.append("")

#         if roll5_df is not None and "topic" in roll5_df.columns:
#             rt = roll5_df[roll5_df["topic"] == tid].sort_values("year")
#             if not rt.empty:
#                 md.append("### Roll5 Share\n")
#                 md.append("| Year | share_CN | share_US |")
#                 md.append("|------|----------|----------|")
#                 for _, r in rt.iterrows():
#                     md.append(f"| {int(r['year'])} | {r.get('share_CN', 0):.4f} | {r.get('share_US', 0):.4f} |")
#                 md.append("")

#         # ── A5) Gap strength indicators ──────────────────────────────────
#         md.append("## 5. Gap Strength Indicators\n")
#         df_cn = df_tid_all[df_tid_all[country_col] == "CN"]
#         df_us = df_tid_all[df_tid_all[country_col] == "US"]
#         cn_n, us_n = len(df_cn), len(df_us)
#         cn_share_val = cn_n / (cn_n + us_n) if (cn_n + us_n) > 0 else 0
#         md.append(f"- count_CN: {cn_n}")
#         md.append(f"- count_US: {us_n}")
#         md.append(f"- CN_share_in_topic: {cn_share_val:.4f}")
#         md.append(f"- CN coverage: {'Yes' if cn_n > 0 else '**No (zero)**'}")

#         if lead_lag_df_in is not None:
#             ll = lead_lag_df_in[lead_lag_df_in["topic"] == tid]
#             if len(ll) > 0:
#                 r_ll = ll.iloc[0]
#                 for c in ["enter_CN", "enter_US", "lag"]:
#                     if c in r_ll.index and pd.notna(r_ll[c]):
#                         md.append(f"- {c}: {r_ll[c]}")
#         md.append("")

#         # ── C) Diagnostics ───────────────────────────────────────────────
#         md.append("## 6. Diagnostic — True Gap vs Split / Boundary\n")

#         # C1) Adjacency check
#         md.append("### 6.1 Adjacency Check (Cosine Similarity)\n")
#         if te is not None and tid in tid_to_idx:
#             idx = tid_to_idx[tid]
#             emb_tid = te[idx].reshape(1, -1)
#             sims = _cosine_sim(emb_tid, te)[0]
#             sim_recs = sorted(
#                 [(ot, sims[oi]) for ot, oi in tid_to_idx.items() if ot != tid and ot != -1],
#                 key=lambda x: -x[1],
#             )[:5]
#             md.append("| Rank | Topic | Similarity | Keywords |")
#             md.append("|------|-------|------------|----------|")
#             strong_split = False
#             for rank, (ot, sv) in enumerate(sim_recs, 1):
#                 try:
#                     ok = topic_model_obj.get_topic(ot)
#                     ok_str = ", ".join([w for w, _ in ok[:8]]) if ok else ""
#                 except Exception:
#                     ok_str = ""
#                 flag = " SPLIT?" if sv > 0.9 else ""
#                 if sv > 0.9:
#                     strong_split = True
#                 md.append(f"| {rank} | T{ot} | {sv:.4f}{flag} | {ok_str} |")
#             md.append("")
#             if strong_split:
#                 md.append("**Warning: strong split signal (sim > 0.9)**\n")
#             else:
#                 md.append("**No strong split signal (all sim <= 0.9)**\n")
#         else:
#             md.append("*(Topic embeddings unavailable)*\n")

#         # C2) Confidence check
#         md.append("### 6.2 Confidence Check (surrogate topic_prob)\n")
#         if has_prob:
#             prob_t = df[df[topic_col] == tid]["topic_prob"].dropna()
#             prob_all = df[valid_topic_mask]["topic_prob"].dropna()
#             if len(prob_t) > 0 and len(prob_all) > 0:
#                 md.append(f"- **Topic {tid}**: mean={prob_t.mean():.4f}, median={prob_t.median():.4f}, p10={prob_t.quantile(0.1):.4f}")
#                 md.append(f"- **All topics**: mean={prob_all.mean():.4f}, median={prob_all.median():.4f}, p10={prob_all.quantile(0.1):.4f}")
#                 if prob_t.mean() < prob_all.quantile(0.25):
#                     md.append("\n**Warning: low confidence — topic_prob mean below global 25th percentile**\n")
#                 else:
#                     md.append("\n**Confidence acceptable**\n")
#             else:
#                 md.append("*(Insufficient data)*\n")
#         else:
#             md.append("*(topic_prob not available)*\n")

#         # C3) Local hierarchical re-clustering
#         md.append("### 6.3 Robustness Check (Agglomerative local re-clustering)\n")
#         if umap_emb is not None:
#             orig_mask = (df[topic_col] == tid).values
#             orig_indices = set(np.where(orig_mask)[0])
#             local_grid = [
#                 (max(12, AGGLO_N_CLUSTERS // 2), "average", "cosine"),
#                 (AGGLO_N_CLUSTERS, "complete", "cosine"),
#                 (min(120, AGGLO_N_CLUSTERS + 20), "ward", "euclidean"),
#             ]
#             stability_rows = []
#             for ncl, lnk, met in local_grid:
#                 try:
#                     n_neighbors_conn = max(2, min(30, len(df) - 1))
#                     conn = kneighbors_graph(umap_emb, n_neighbors=n_neighbors_conn, include_self=False)
#                     _clu = build_agglomerative_model(n_clusters=ncl, linkage_name=lnk, metric_name=met, connectivity=conn)
#                     _labels = _clu.fit_predict(umap_emb)
#                 except Exception as e:
#                     stability_rows.append(
#                         {
#                             "n_clusters": ncl,
#                             "linkage": lnk,
#                             "metric": met,
#                             "exists": False,
#                             "count_CN": 0,
#                             "count_US": 0,
#                             "jaccard_docs": 0.0,
#                             "jaccard_keywords": 0.0,
#                             "error_msg": f"{type(e).__name__}: {e}",
#                         }
#                     )
#                     continue

#                 best_jac, best_cid = 0.0, -1
#                 for cid in np.unique(_labels):
#                     cl_set = set(np.where(_labels == cid)[0])
#                     inter = len(orig_indices & cl_set)
#                     union = len(orig_indices | cl_set)
#                     jac = inter / union if union > 0 else 0.0
#                     if jac > best_jac:
#                         best_jac = jac
#                         best_cid = cid

#                 exists = best_jac > 0.3
#                 c_cn, c_us, jac_kw = 0, 0, 0.0
#                 if exists and best_cid >= 0:
#                     cl_idx = np.where(_labels == best_cid)[0]
#                     cl_countries = df.iloc[cl_idx][country_col].value_counts()
#                     c_cn = int(cl_countries.get("CN", 0))
#                     c_us = int(cl_countries.get("US", 0))
#                     cl_texts = df.iloc[cl_idx]["text"].dropna().astype(str).tolist()
#                     from collections import Counter
#                     word_freq = Counter()
#                     for _txt in cl_texts[:500]:
#                         word_freq.update(_txt.lower().split())
#                     for sw in [
#                         "the", "of", "and", "in", "to", "a", "is", "for", "that", "with", "on", "was", "are",
#                         "by", "this", "an", "be", "as", "from", "it", "or", "at", "were", "which", "has", "been",
#                         "have", "not", "its", "we", "our", "can", "their", "also", "these", "than", "between",
#                         "using", "used", "based", "such", "more", "two", "most", "other", "may", "but", "all",
#                         "both", "results", "however", "paper", "study", "research",
#                     ]:
#                         word_freq.pop(sw, None)
#                     new_kw_set = set([w for w, _ in word_freq.most_common(15)])
#                     jac_kw = (
#                         len(kw_set_orig & new_kw_set) / len(kw_set_orig | new_kw_set)
#                         if len(kw_set_orig | new_kw_set) > 0
#                         else 0.0
#                     )

#                 stability_rows.append(
#                     {
#                         "n_clusters": ncl,
#                         "linkage": lnk,
#                         "metric": met,
#                         "exists": exists,
#                         "count_CN": c_cn,
#                         "count_US": c_us,
#                         "jaccard_docs": round(best_jac, 4),
#                         "jaccard_keywords": round(jac_kw, 4),
#                     }
#                 )

#             stab_df = pd.DataFrame(stability_rows)
#             md.append("| n_clusters | linkage | metric | exists | count_CN | count_US | Jaccard(docs) | Jaccard(keywords) |")
#             md.append("|-----------:|---------|--------|:------:|---------:|---------:|--------------:|------------------:|")
#             for _, sr in stab_df.iterrows():
#                 md.append(
#                     f"| {int(sr['n_clusters'])} | {sr['linkage']} | {sr['metric']} | "
#                     f"{'Y' if bool(sr['exists']) else 'N'} | {int(sr['count_CN'])} | {int(sr['count_US'])} | "
#                     f"{float(sr['jaccard_docs']):.4f} | {float(sr['jaccard_keywords']):.4f} |"
#                 )
#             md.append("")

#             if all(bool(sr["exists"]) and int(sr["count_CN"]) == 0 for _, sr in stab_df.iterrows()):
#                 md.append("**Robust US-only: local clusters persist and stay CN-absent across Agglomerative settings.**\n")
#             elif not all(bool(sr["exists"]) for _, sr in stab_df.iterrows()):
#                 md.append("**Potential instability: local cluster disappears under some Agglomerative settings.**\n")
#             else:
#                 md.append("**Cluster persists but CN representation changes across settings.**\n")

#             stab_df.to_csv(os.path.join(us_dir, f"stability_topic_{tid}.csv"), index=False)
#         else:
#             md.append("*(UMAP embeddings unavailable — local robustness check skipped)*\n")

#         # ── Write Markdown ───────────────────────────────────────────────
#         md_path = os.path.join(us_dir, f"us_only_topic_{tid}.md")
#         with open(md_path, "w", encoding="utf-8") as f:
#             f.write("\n".join(md))
#         print(f"  ✅ Markdown: {md_path}")

#         # ── Charts ───────────────────────────────────────────────────────
#         if yearly_df is not None and "topic" in yearly_df.columns:
#             yt = yearly_df[yearly_df["topic"] == tid].sort_values("year")
#             if not yt.empty:
#                 fig_b1, ax_b1 = plt.subplots(figsize=(9, 5))
#                 _cn_col = "count_CN" if "count_CN" in yt.columns else "CN"
#                 _us_col = "count_US" if "count_US" in yt.columns else "US"
#                 w = 0.35
#                 x_pos = np.arange(len(yt))
#                 ax_b1.bar(x_pos - w / 2, yt[_cn_col].values, w, label="CN", color="#E03C31")
#                 ax_b1.bar(x_pos + w / 2, yt[_us_col].values, w, label="US", color="#3C6FE0")
#                 ax_b1.set_xticks(x_pos)
#                 ax_b1.set_xticklabels(yt["year"].astype(int).values, rotation=45, ha="right", fontsize=8)
#                 ax_b1.set_xlabel("Year")
#                 ax_b1.set_ylabel("Paper Count")
#                 ax_b1.set_title(f"Topic {tid} — Annual Paper Count (CN vs US)")
#                 ax_b1.legend()
#                 plt.tight_layout()
#                 p_b1 = os.path.join(us_dir, f"topic_{tid}_counts_by_year.png")
#                 fig_b1.savefig(p_b1, dpi=150)
#                 plt.close(fig_b1)
#                 print(f"  ✅ Chart: {p_b1}")

#         if roll5_df is not None and "topic" in roll5_df.columns:
#             rt = roll5_df[roll5_df["topic"] == tid].sort_values("year")
#             if not rt.empty:
#                 fig_b2, ax_b2 = plt.subplots(figsize=(9, 5))
#                 ax_b2.plot(rt["year"], rt["share_CN"], "o-", color="#E03C31", label="CN share", markersize=3)
#                 ax_b2.plot(rt["year"], rt["share_US"], "s-", color="#3C6FE0", label="US share", markersize=3)
#                 ax_b2.set_xlabel("Year")
#                 ax_b2.set_ylabel("Topic Share (roll5)")
#                 ax_b2.set_title(f"Topic {tid} — 5-year Rolling Share (CN vs US)")
#                 ax_b2.legend()
#                 ax_b2.grid(True, alpha=0.3)
#                 plt.tight_layout()
#                 p_b2 = os.path.join(us_dir, f"topic_{tid}_roll5_share.png")
#                 fig_b2.savefig(p_b2, dpi=150)
#                 plt.close(fig_b2)
#                 print(f"  ✅ Chart: {p_b2}")

#     print(f"\n✅ All US-only topic packs written to {us_dir}")


# # ── 15.1 Execute ─────────────────────────────────────────────────────────────
# export_us_only_topic_pack(
#     tids=US_ONLY_TIDS,
#     df=df,
#     topic_col=TOPIC_COL,
#     country_col=COUNTRY_COL_14,
#     topic_model_obj=topic_model,
#     output_dir=OUTPUT_DIR,
#     repr_docs_dict=_repr_15,
#     yearly_df=_yearly_15,
#     roll5_df=_roll5_15,
#     lead_lag_df_in=_ll_15,
#     trend_df=_trend_15,
#     te=_te_15,
#     tid_to_idx=_tid_to_idx_15,
#     umap_emb=_umap_emb_15,
#     seed=SEED,
# )

# print("\n🎉 §16 complete — US-only topic deep dives generated.")


In [ ]:
# # 【单元格中文说明】
# # - 列出 §15–§16 期望产物清单并检查文件是否存在，作为交付清单。


# # ══════════════════════════════════════════════════════════════════════════════
# # §15-16  Product Manifest
# # ══════════════════════════════════════════════════════════════════════════════
# from pathlib import Path as _P

# _section_files = {
#     "§15 Topic Map Overlay": [
#         "topic_stats_country.csv",
#         "topic_map_country.html",
#         "topic_map_country.png",
#         "topic_map_country_dominant.png",
#     ],
#     "§16 US-only Deep Dives": [],
#     "Hierarchical Clustering Upgrade": [
#         "hierarchical_clustering/agglom_search_results.csv",
#         "hierarchical_clustering/agglom_search_top10.csv",
#         "hierarchical_clustering/best_agglom_params.json",
#         "hierarchical_clustering/umap_scatter_final.png",
#         "hierarchical_clustering/topic_size_distribution.png",
#         "hierarchical_clustering/topic_centroid_dendrogram.png",
#         "hierarchical_clustering/agglom_search_silhouette.png",
#         "hierarchical_clustering/agglom_search_dbi.png",
#         "hierarchical_clustering/agglom_search_topk.png",
#     ],
# }

# # Collect §16 files dynamically
# _us_dir = _P(OUTPUT_DIR) / "us_only_topics"
# if _us_dir.exists():
#     for f in sorted(_us_dir.iterdir()):
#         _section_files["§16 US-only Deep Dives"].append(f"us_only_topics/{f.name}")

# print(f"\n{'═'*65}")
# print("  Ⅳ. 技术版图 + 结构洞察 — Product Manifest")
# print(f"  Base directory: {os.path.abspath(OUTPUT_DIR)}")
# print(f"{'═'*65}")

# for section, files in _section_files.items():
#     print(f"\n  [{section}]")
#     for fn in files:
#         fp = _P(OUTPUT_DIR) / fn
#         if fp.exists():
#             sz = fp.stat().st_size
#             unit = "KB" if sz < 1_048_576 else "MB"
#             val = sz / 1024 if unit == "KB" else sz / 1_048_576
#             print(f"    ✅ {fn}  ({val:.1f} {unit})")
#         else:
#             print(f"    ❌ {fn}  (not found — check export)")

# print(f"\n{'═'*65}")
# print("🎉 Sections §15–§16 (技术版图 + 结构洞察) complete!")


# §17 Robustness & Uncertainty / 稳健性与不确定性

This section adds low-cost robustness checks for the CN–US technology gap metrics under UMAP + AgglomerativeClustering.

- Outputs are written to `output/cluster_results/robustness/`.
- We report both Jensen-Shannon **distance** and **divergence**.
- In SciPy, `jensenshannon(p, q)` returns **distance**; divergence is `distance**2`.

**【中文说明】**
- 对核心缺口指标做 **随机种子重复、超参网格、分层自助法**，报告区间与方差，提醒读者注意估计不确定性。
- 明确 SciPy 返回的是 JS **距离**，散度为其平方。


In [ ]:
# # 【单元格中文说明】
# # - 稳健性开关：重复随机种子、超参网格、分层自助法；ROBUST_METRIC_COLS 列出要汇总的不确定性指标。


# # ══════════════════════════════════════════════════════════════════════════════
# # 16.1 Config & switches
# # ══════════════════════════════════════════════════════════════════════════════
# from pathlib import Path
# import itertools
# import time
# import traceback

# ROBUST_DIR = Path(OUTPUT_DIR) / "robustness"
# ROBUST_DIR.mkdir(parents=True, exist_ok=True)

# RUN_SEED_REPEAT = True
# RUN_GRID = True
# RUN_BOOTSTRAP = True

# SEED_LIST = [SEED + i for i in range(5)]
# GRID_MAX_COMBOS = 18
# GRID_MULTI_SEED = False
# GRID_SEED_LIST = [SEED] if not GRID_MULTI_SEED else [SEED, SEED + 1, SEED + 2]

# BOOTSTRAP_B = 200
# BOOTSTRAP_SEED = 2026
# BOOTSTRAP_USE_YEAR_STRATA = True

# CALC_PROBS_IN_ROBUST = False

# ROBUST_BASE_UMAP = {
#     "n_neighbors": UMAP_N_NEIGHBORS,
#     "n_components": UMAP_N_COMPONENTS,
#     "min_dist": UMAP_MIN_DIST,
#     "metric": UMAP_METRIC,
# }
# ROBUST_BASE_CLUSTER = {
#     "n_clusters": AGGLO_N_CLUSTERS,
#     "linkage": AGGLO_LINKAGE,
#     "metric": AGGLO_METRIC,
# }
# ROBUST_BASE_VECTORIZER = {
#     "stop_words": CUSTOM_STOPWORDS,
#     "ngram_range": NGRAM_RANGE,
#     "max_features": MAX_FEATURES,
# }

# ROBUST_METRIC_COLS = [
#     "js_distance",
#     "js_divergence",
#     "cn_coverage",
#     "us_coverage",
#     "frontier_js_distance",
#     "frontier_js_divergence",
#     "impact_gap_mncs_us_minus_cn",
#     "impact_gap_pp10_us_minus_cn",
#     "lead_lag_mean",
#     "lead_lag_median",
#     "lead_lag_n_topics",
#     "n_topics",
#     "outlier_ratio",
#     "topic_size_min",
#     "topic_size_median",
#     "topic_size_max",
#     "max_topic_share",
#     "topic_size_cv",
# ]

# print(f"ROBUST_DIR = {ROBUST_DIR.resolve()}")
# print(f"RUN_SEED_REPEAT={RUN_SEED_REPEAT}, RUN_GRID={RUN_GRID}, RUN_BOOTSTRAP={RUN_BOOTSTRAP}")
# print(f"SEED_LIST={SEED_LIST}")
# print(f"GRID_MAX_COMBOS={GRID_MAX_COMBOS}, GRID_SEED_LIST={GRID_SEED_LIST}")
# print(f"BOOTSTRAP_B={BOOTSTRAP_B}, BOOTSTRAP_SEED={BOOTSTRAP_SEED}")


In [ ]:
# # 【单元格中文说明】
# # - 封装一次性 BERTopic 拟合、gap 指标聚合、自助抽样索引等，供后续多实验循环调用。


# # ══════════════════════════════════════════════════════════════════════════════
# # 16.2 Helper functions
# # ══════════════════════════════════════════════════════════════════════════════
# def _map_country_cn_us(v):
#     # 中文：将多种写法统一映射到 CN/US/OTHER。
#     # 中文：将多种写法统一映射到 CN/US/OTHER。
#     if pd.isna(v):
#         return "OTHER"
#     x = str(v).strip()
#     if x in {"CN", "China", "CHINA", "china", "PRC", "Peoples R China"}:
#         return "CN"
#     if x in {"US", "USA", "United States", "United States of America", "united states"}:
#         return "US"
#     return "OTHER"


# def _ensure_country2(df_in):
#     # 中文：若无 country2 则从 country / country_code 推导。
#     # 中文：若无 country2 则从 country / country_code 推导。
#     df_out = df_in.copy()
#     if "country2" in df_out.columns and df_out["country2"].notna().any():
#         return df_out
#     if "country_code" in df_out.columns:
#         df_out["country2"] = df_out["country_code"].map(
#             lambda c: "CN" if c == "CN" else ("US" if c == "US" else "OTHER")
#         )
#     elif "country" in df_out.columns:
#         df_out["country2"] = df_out["country"].apply(_map_country_cn_us)
#     else:
#         df_out["country2"] = "OTHER"
#     return df_out


# def _detect_topic_col(df_in):
#     # 中文：优先使用全局 TOPIC_COL，否则在候选列中自动检测。
#     # 中文：优先使用全局 TOPIC_COL，否则在候选列中自动检测。
#     if "TOPIC_COL" in globals() and TOPIC_COL in df_in.columns:
#         return TOPIC_COL
#     for c in ["topic_reduced", "topic_reassign", "topic"]:
#         if c in df_in.columns:
#             return c
#     raise ValueError("No topic column found.")


# def fit_topics_once(docs, embeddings, seed, umap_params, cluster_params, vectorizer_params, calc_probs=False):
#     # 中文：用给定 UMAP/聚类/向量化参数训练一次 BERTopic 并返回主题标签。
#     # 中文：这里显式复用 specter2 adapter backend + KeyBERTInspired，保证鲁棒性实验与主流程的主题表示一致。
#     umap_model = UMAP(
#         n_neighbors=umap_params["n_neighbors"],
#         n_components=umap_params["n_components"],
#         min_dist=umap_params.get("min_dist", 0.1),
#         metric=umap_params.get("metric", "cosine"),
#         random_state=seed,
#         low_memory=True,
#     )
#     cluster_model = build_agglomerative_model(
#         n_clusters=cluster_params["n_clusters"],
#         linkage_name=cluster_params.get("linkage", "average"),
#         metric_name=cluster_params.get("metric", "cosine"),
#     )
#     vectorizer_model = CountVectorizer(
#         stop_words=vectorizer_params.get("stop_words", CUSTOM_STOPWORDS),
#         ngram_range=vectorizer_params.get("ngram_range", (1, 2)),
#         max_features=vectorizer_params.get("max_features", None),
#     )
#     model = BERTopic(
#         embedding_model=get_specter2_representation_encoder(),
#         representation_model=KeyBERTInspired(),
#         umap_model=umap_model,
#         hdbscan_model=cluster_model,
#         vectorizer_model=vectorizer_model,
#         top_n_words=TOP_N_KEYWORDS,
#         calculate_probabilities=calc_probs,
#         verbose=False,
#     )
#     topics, _ = model.fit_transform(docs, embeddings)
#     return np.asarray(topics).astype(int), model


# def _compute_lead_lag_summary(df_eval, topic_col, year_col):
#     # 中文：基于 roll5 与 detect_enter_year 汇总 lead–lag 的均值/中位数/有效主题数。
#     # 中文：基于 roll5 与 detect_enter_year 汇总 lead–lag 的均值/中位数/有效主题数。
#     if year_col not in df_eval.columns:
#         return np.nan, np.nan, 0

#     te = df_eval[df_eval["country2"].isin(["CN", "US"])].copy()
#     if (te[topic_col] == -1).any():
#         te = te[te[topic_col] != -1].copy()
#     te["year_int"] = pd.to_numeric(te[year_col], errors="coerce")
#     te = te.dropna(subset=["year_int"]).copy()
#     if te.empty:
#         return np.nan, np.nan, 0
#     te["year_int"] = te["year_int"].astype(int)

#     if "build_rolling_share" not in globals() or "detect_enter_year" not in globals():
#         return np.nan, np.nan, 0

#     tmp = te.rename(columns={topic_col: "_topic_eval_ll"})
#     roll = build_rolling_share(tmp, topic_col="_topic_eval_ll", year_col="year_int", country_col="country2", window=5, min_periods=3)
#     if roll.empty:
#         return np.nan, np.nan, 0

#     recs = []
#     for t, grp in roll.groupby("topic"):
#         grp = grp.sort_values("year")
#         r = {"topic": int(t)}
#         for cc in ["CN", "US"]:
#             col = f"share_{cc}"
#             valid = grp.dropna(subset=[col])
#             if len(valid) < 5:
#                 r[f"enter_{cc}"] = np.nan
#             else:
#                 ey, _flag = detect_enter_year(valid["year"].values, valid[col].values)
#                 r[f"enter_{cc}"] = ey
#         recs.append(r)

#     ll = pd.DataFrame(recs)
#     if ll.empty:
#         return np.nan, np.nan, 0

#     if "compute_lead_lag" in globals():
#         ll = compute_lead_lag(ll)
#     else:
#         both = ll["enter_CN"].notna() & ll["enter_US"].notna()
#         ll["lag"] = np.nan
#         ll.loc[both, "lag"] = ll.loc[both, "enter_CN"] - ll.loc[both, "enter_US"]

#     lag = pd.to_numeric(ll.get("lag", pd.Series(dtype=float)), errors="coerce").dropna()
#     if lag.empty:
#         return np.nan, np.nan, 0
#     return float(lag.mean()), float(lag.median()), int(lag.shape[0])


# def compute_gap_metrics(df_base, topic_assignments, topic_col_name="_topic_eval"):
#     # 中文：在给定主题赋值下重算 JS、覆盖、前沿、影响与 lead–lag 等缺口指标。
#     # 中文：在给定主题赋值下重算 JS、覆盖、前沿、影响与 lead–lag 等缺口指标。
#     res = {
#         "js_distance": np.nan,
#         "js_divergence": np.nan,
#         "cn_coverage": np.nan,
#         "us_coverage": np.nan,
#         "frontier_js_distance": np.nan,
#         "frontier_js_divergence": np.nan,
#         "impact_gap_mncs_us_minus_cn": np.nan,
#         "impact_gap_pp10_us_minus_cn": np.nan,
#         "lead_lag_mean": np.nan,
#         "lead_lag_median": np.nan,
#         "lead_lag_n_topics": 0,
#         "n_topics": np.nan,
#         "outlier_ratio": 0.0,
#         "topic_size_min": np.nan,
#         "topic_size_median": np.nan,
#         "topic_size_max": np.nan,
#         "max_topic_share": np.nan,
#         "topic_size_cv": np.nan,
#         "status": "ok",
#         "error_msg": "",
#     }

#     try:
#         df_eval = df_base.reset_index(drop=True).copy()
#         if len(df_eval) != len(topic_assignments):
#             raise ValueError(f"Length mismatch: df={len(df_eval)}, topics={len(topic_assignments)}")

#         df_eval = _ensure_country2(df_eval)
#         topics_s = pd.Series(topic_assignments)
#         topics_s = pd.to_numeric(topics_s, errors="coerce")
#         if topics_s.isna().any():
#             topics_s = topics_s.fillna(-1)
#         df_eval[topic_col_name] = topics_s.astype(int)

#         valid_topic_mask = df_eval[topic_col_name].notna()
#         if (df_eval[topic_col_name] == -1).any():
#             valid_topic_mask = valid_topic_mask & (df_eval[topic_col_name] != -1)
#         topic_sizes = df_eval.loc[valid_topic_mask, topic_col_name].value_counts()
#         res["n_topics"] = int(topic_sizes.shape[0])
#         if topic_sizes.shape[0] > 0:
#             res["topic_size_min"] = float(topic_sizes.min())
#             res["topic_size_median"] = float(topic_sizes.median())
#             res["topic_size_max"] = float(topic_sizes.max())
#             res["max_topic_share"] = float(topic_sizes.max() / topic_sizes.sum())
#             res["topic_size_cv"] = float(topic_sizes.std(ddof=0) / (topic_sizes.mean() + 1e-12))

#         cnus = df_eval[df_eval["country2"].isin(["CN", "US"]) & valid_topic_mask].copy()
#         all_topics = sorted(cnus[topic_col_name].unique())

#         cn_topics = set(cnus.loc[cnus["country2"] == "CN", topic_col_name].unique())
#         us_topics = set(cnus.loc[cnus["country2"] == "US", topic_col_name].unique())
#         topic_union = set(all_topics)
#         if len(topic_union) > 0:
#             res["cn_coverage"] = float(len(cn_topics) / len(topic_union))
#             res["us_coverage"] = float(len(us_topics) / len(topic_union))

#         if len(all_topics) > 0:
#             cn_counts = cnus.loc[cnus["country2"] == "CN", topic_col_name].value_counts().reindex(all_topics, fill_value=0).astype(float)
#             us_counts = cnus.loc[cnus["country2"] == "US", topic_col_name].value_counts().reindex(all_topics, fill_value=0).astype(float)
#             if cn_counts.sum() > 0 and us_counts.sum() > 0:
#                 p_cn = (cn_counts / cn_counts.sum()).values
#                 p_us = (us_counts / us_counts.sum()).values
#                 js_d = float(jensenshannon(p_cn, p_us))
#                 res["js_distance"] = js_d
#                 res["js_divergence"] = float(js_d ** 2)

#         CIT_COL_LOCAL = CIT_COL if "CIT_COL" in globals() else ("citation" if "citation" in df_eval.columns else None)
#         YEAR_COL_LOCAL = YEAR_COL if "YEAR_COL" in globals() else ("year" if "year" in df_eval.columns else None)
#         MIN_BUCKET_LOCAL = MIN_BUCKET_SIZE if "MIN_BUCKET_SIZE" in globals() else 20
#         WINDOW_LOCAL = WINDOW_YEARS if "WINDOW_YEARS" in globals() else 5
#         EXCL_LOCAL = EXCLUDE_RECENT_YEARS if "EXCLUDE_RECENT_YEARS" in globals() else 1

#         if CIT_COL_LOCAL is not None and YEAR_COL_LOCAL is not None:
#             work = cnus.copy()
#             work["_cit"] = pd.to_numeric(work[CIT_COL_LOCAL], errors="coerce")
#             work["_year"] = pd.to_numeric(work[YEAR_COL_LOCAL], errors="coerce")
#             work = work.dropna(subset=["_cit", "_year"]).copy()
#             if not work.empty:
#                 work["_year"] = work["_year"].astype(int)
#                 work["_bucket"] = list(zip(work[topic_col_name].astype(int), work["_year"]))

#                 bucket_stats = work.groupby("_bucket")["_cit"].agg(["mean", "count"]).rename(
#                     columns={"mean": "expected_mean", "count": "bucket_size"}
#                 )
#                 work = work.join(bucket_stats, on="_bucket")
#                 work["nc"] = (work["_cit"] + 1) / (work["expected_mean"] + 1)

#                 bucket_q90 = work[work["bucket_size"] >= MIN_BUCKET_LOCAL].groupby("_bucket")["_cit"].quantile(0.9).rename("q90")
#                 work = work.join(bucket_q90, on="_bucket")
#                 work["top10_flag"] = False
#                 has_q = work["q90"].notna()
#                 work.loc[has_q, "top10_flag"] = work.loc[has_q, "_cit"] >= work.loc[has_q, "q90"]

#                 max_year_local = int(work["_year"].max())
#                 frontier_end = max_year_local - EXCL_LOCAL
#                 frontier_start = frontier_end - WINDOW_LOCAL + 1
#                 mask_B = work["top10_flag"] & (work["_year"] >= frontier_start) & (work["_year"] <= frontier_end)
#                 fr = work[mask_B].copy()

#                 if not fr.empty:
#                     all_t_fr = sorted(work[topic_col_name].unique())
#                     us_fr = fr.loc[fr["country2"] == "US", topic_col_name].value_counts().reindex(all_t_fr, fill_value=0).astype(float)
#                     cn_fr = fr.loc[fr["country2"] == "CN", topic_col_name].value_counts().reindex(all_t_fr, fill_value=0).astype(float)
#                     if us_fr.sum() > 0 and cn_fr.sum() > 0:
#                         p_usf = (us_fr / us_fr.sum()).values
#                         p_cnf = (cn_fr / cn_fr.sum()).values
#                         js_fr = float(jensenshannon(p_usf, p_cnf))
#                         res["frontier_js_distance"] = js_fr
#                         res["frontier_js_divergence"] = float(js_fr ** 2)

#                 eligible = work[work["q90"].notna()].copy()
#                 if not eligible.empty:
#                     mncs = eligible.groupby([topic_col_name, "country2"])["nc"].mean().unstack(fill_value=np.nan)
#                     if "US" not in mncs.columns:
#                         mncs["US"] = np.nan
#                     if "CN" not in mncs.columns:
#                         mncs["CN"] = np.nan
#                     gap_mncs = (mncs["US"] - mncs["CN"]).dropna()
#                     if not gap_mncs.empty:
#                         res["impact_gap_mncs_us_minus_cn"] = float(gap_mncs.mean())

#                     pp10 = eligible.groupby([topic_col_name, "country2"])["top10_flag"].mean().unstack(fill_value=np.nan)
#                     if "US" not in pp10.columns:
#                         pp10["US"] = np.nan
#                     if "CN" not in pp10.columns:
#                         pp10["CN"] = np.nan
#                     gap_pp10 = (pp10["US"] - pp10["CN"]).dropna()
#                     if not gap_pp10.empty:
#                         res["impact_gap_pp10_us_minus_cn"] = float(gap_pp10.mean())

#                 try:
#                     ll_mean, ll_med, ll_n = _compute_lead_lag_summary(work, topic_col_name, "_year")
#                     res["lead_lag_mean"] = ll_mean
#                     res["lead_lag_median"] = ll_med
#                     res["lead_lag_n_topics"] = ll_n
#                 except Exception:
#                     pass

#     except Exception as e:
#         res["status"] = "error"
#         res["error_msg"] = f"{type(e).__name__}: {e}"

#     return res


# def summarize_runs(df_runs, metric_cols):
#     # 中文：对多次实验的指标做均值、标准差与分位数汇总。
#     # 中文：对多次实验的指标做均值、标准差与分位数汇总。
#     rows = []
#     for m in metric_cols:
#         if m not in df_runs.columns:
#             continue
#         s = pd.to_numeric(df_runs[m], errors="coerce").dropna()
#         if s.empty:
#             rows.append(
#                 {
#                     "metric": m,
#                     "mean": np.nan,
#                     "std": np.nan,
#                     "q2_5": np.nan,
#                     "q50": np.nan,
#                     "q97_5": np.nan,
#                     "n_valid": 0,
#                 }
#             )
#         else:
#             rows.append(
#                 {
#                     "metric": m,
#                     "mean": float(s.mean()),
#                     "std": float(s.std(ddof=1)) if len(s) > 1 else 0.0,
#                     "q2_5": float(s.quantile(0.025)),
#                     "q50": float(s.quantile(0.5)),
#                     "q97_5": float(s.quantile(0.975)),
#                     "n_valid": int(len(s)),
#                 }
#             )
#     return pd.DataFrame(rows)


# def stratified_bootstrap_indices(df_in, strata_cols, B, seed):
#     # 中文：按分层（如年×国）有放回抽样索引，用于自助法。
#     # 中文：按分层（如年×国）有放回抽样索引，用于自助法。
#     rs = np.random.RandomState(seed)
#     work = df_in.reset_index(drop=True)
#     strata_cols = [c for c in (strata_cols or []) if c in work.columns]

#     if len(work) == 0:
#         return []

#     if not strata_cols:
#         base_idx = np.arange(len(work))
#         return [rs.choice(base_idx, size=len(base_idx), replace=True) for _ in range(B)]

#     groups = []
#     for _key, grp in work.groupby(strata_cols, dropna=False, sort=False):
#         idx = grp.index.to_numpy()
#         if idx.size > 0:
#             groups.append(idx)

#     if not groups:
#         base_idx = np.arange(len(work))
#         return [rs.choice(base_idx, size=len(base_idx), replace=True) for _ in range(B)]

#     out = []
#     for _ in range(B):
#         sampled = [rs.choice(g, size=len(g), replace=True) for g in groups]
#         idx_all = np.concatenate(sampled)
#         rs.shuffle(idx_all)
#         out.append(idx_all)
#     return out


# print("Helper functions ready: fit_topics_once, compute_gap_metrics, summarize_runs, stratified_bootstrap_indices")


In [ ]:
# # 【单元格中文说明】
# # - 在 SEED_LIST 上重复训练并计算 gap 指标，输出箱线图/误差条，观察种子敏感性。


# # ══════════════════════════════════════════════════════════════════════════════
# # 16.3 Seed repeat experiment
# # ══════════════════════════════════════════════════════════════════════════════
# results_seed_df = pd.DataFrame()
# results_seed_summary_df = pd.DataFrame()

# if RUN_SEED_REPEAT:
#     seed_records = []
#     for i, run_seed in enumerate(SEED_LIST, start=1):
#         t0 = time.time()
#         rec = {
#             "experiment": "seed_repeat",
#             "seed": int(run_seed),
#             "umap_n_neighbors": ROBUST_BASE_UMAP["n_neighbors"],
#             "umap_n_components": ROBUST_BASE_UMAP["n_components"],
#             "agglom_n_clusters": ROBUST_BASE_CLUSTER["n_clusters"],
#             "agglom_linkage": ROBUST_BASE_CLUSTER["linkage"],
#             "agglom_metric": ROBUST_BASE_CLUSTER["metric"],
#             "status": "ok",
#             "error_msg": "",
#         }
#         try:
#             topics_run, _model_run = fit_topics_once(
#                 docs=docs,
#                 embeddings=embeddings,
#                 seed=run_seed,
#                 umap_params=ROBUST_BASE_UMAP,
#                 cluster_params=ROBUST_BASE_CLUSTER,
#                 vectorizer_params=ROBUST_BASE_VECTORIZER,
#                 calc_probs=CALC_PROBS_IN_ROBUST,
#             )
#             m = compute_gap_metrics(df, topics_run, topic_col_name="_topic_eval")
#             rec.update(m)
#         except Exception as e:
#             rec.update({k: np.nan for k in ROBUST_METRIC_COLS})
#             rec["status"] = "error"
#             rec["error_msg"] = f"{type(e).__name__}: {e}"
#             print(f"[Seed {run_seed}] failed: {e}")
#             print(traceback.format_exc().splitlines()[-1])

#         rec["runtime_sec"] = round(time.time() - t0, 2)
#         seed_records.append(rec)
#         print(f"[{i}/{len(SEED_LIST)}] seed={run_seed}, status={rec['status']}, runtime={rec['runtime_sec']}s")

#     results_seed_df = pd.DataFrame(seed_records)
#     results_seed_path = ROBUST_DIR / "results_seed.csv"
#     results_seed_df.to_csv(results_seed_path, index=False)

#     results_seed_summary_df = summarize_runs(results_seed_df, ROBUST_METRIC_COLS)
#     results_seed_summary_path = ROBUST_DIR / "results_seed_summary.csv"
#     results_seed_summary_df.to_csv(results_seed_summary_path, index=False)

#     print(f"Saved: {results_seed_path}")
#     print(f"Saved: {results_seed_summary_path}")

#     core_cols = [
#         c
#         for c in ["js_distance", "frontier_js_distance", "impact_gap_mncs_us_minus_cn"]
#         if c in results_seed_df.columns and pd.to_numeric(results_seed_df[c], errors="coerce").notna().any()
#     ]

#     if core_cols:
#         fig, ax = plt.subplots(figsize=(9, 5))
#         box_data = [pd.to_numeric(results_seed_df[c], errors="coerce").dropna().values for c in core_cols]
#         ax.boxplot(box_data, labels=core_cols, showmeans=True)
#         ax.set_title("Seed Repeatability: Core Metrics (boxplot)")
#         ax.set_ylabel("Metric value")
#         ax.grid(True, axis="y", alpha=0.3)
#         plt.xticks(rotation=20, ha="right")
#         plt.tight_layout()
#         p_box = ROBUST_DIR / "seed_core_metrics_box.png"
#         fig.savefig(p_box, dpi=180)
#         print(f"Saved: {p_box}")
#         plt.show()

#         ss = results_seed_summary_df.set_index("metric")
#         x = np.arange(len(core_cols))
#         means = [ss.loc[c, "mean"] if c in ss.index else np.nan for c in core_cols]
#         stds = [ss.loc[c, "std"] if c in ss.index else np.nan for c in core_cols]

#         fig, ax = plt.subplots(figsize=(9, 5))
#         ax.errorbar(x, means, yerr=stds, fmt="o", capsize=4)
#         ax.set_xticks(x)
#         ax.set_xticklabels(core_cols, rotation=20, ha="right")
#         ax.set_title("Seed Repeatability: mean ± std")
#         ax.set_ylabel("Metric value")
#         ax.grid(True, axis="y", alpha=0.3)
#         plt.tight_layout()
#         p_err = ROBUST_DIR / "seed_errorbar_mean_std.png"
#         fig.savefig(p_err, dpi=180)
#         print(f"Saved: {p_err}")
#         plt.show()
# else:
#     print("RUN_SEED_REPEAT=False, skipped.")


In [ ]:
# # 【单元格中文说明】
# # - 在 UMAP×Agglo 网格上系统扰动超参，观察 JS 距离、前沿 JS 与主题规模离散度之间的关系。


# # ══════════════════════════════════════════════════════════════════════════════
# # 16.4 Hyperparam grid robustness (UMAP + Agglomerative)
# # ══════════════════════════════════════════════════════════════════════════════
# results_grid_df = pd.DataFrame()

# if RUN_GRID:
#     neighbors_cand = [15, 30, 50]
#     components_cand = [5, 10]
#     min_dist_cand = [0.0, 0.1]
#     linkage_metric_cand = [
#         ("average", "cosine"),
#         ("complete", "cosine"),
#         ("ward", "euclidean"),
#     ]
#     n_clusters_cand = [25, 40, 60, 80]

#     grid_all = []
#     for nn, nc, mdist, ncl, (lnk, met) in itertools.product(
#         neighbors_cand, components_cand, min_dist_cand, n_clusters_cand, linkage_metric_cand
#     ):
#         score = (
#             abs(nn - UMAP_N_NEIGHBORS) / 10.0
#             + abs(nc - UMAP_N_COMPONENTS)
#             + abs(mdist - UMAP_MIN_DIST) * 10.0
#             + abs(ncl - AGGLO_N_CLUSTERS) / 20.0
#             + (0.0 if lnk == AGGLO_LINKAGE else 0.5)
#             + (0.0 if met == AGGLO_METRIC else 0.5)
#         )
#         grid_all.append(
#             {
#                 "n_neighbors": nn,
#                 "n_components": nc,
#                 "min_dist": mdist,
#                 "n_clusters": ncl,
#                 "linkage": lnk,
#                 "metric": met,
#                 "_score": score,
#             }
#         )

#     grid_selected = sorted(grid_all, key=lambda x: x["_score"])[: int(GRID_MAX_COMBOS)]
#     run_seed_list = GRID_SEED_LIST if GRID_MULTI_SEED else [SEED]

#     print(f"Grid total={len(grid_all)}, selected={len(grid_selected)}, seeds={run_seed_list}")

#     grid_records = []
#     total_runs = len(grid_selected) * len(run_seed_list)
#     done = 0

#     for gi, g in enumerate(grid_selected, start=1):
#         for run_seed in run_seed_list:
#             done += 1
#             t0 = time.time()
#             rec = {
#                 "experiment": "grid",
#                 "grid_id": gi,
#                 "seed": int(run_seed),
#                 "umap_n_neighbors": int(g["n_neighbors"]),
#                 "umap_n_components": int(g["n_components"]),
#                 "umap_min_dist": float(g["min_dist"]),
#                 "agglom_n_clusters": int(g["n_clusters"]),
#                 "agglom_linkage": g["linkage"],
#                 "agglom_metric": g["metric"],
#                 "distance_to_baseline": float(g["_score"]),
#                 "status": "ok",
#                 "error_msg": "",
#             }
#             try:
#                 umap_p = {
#                     "n_neighbors": g["n_neighbors"],
#                     "n_components": g["n_components"],
#                     "min_dist": g["min_dist"],
#                     "metric": UMAP_METRIC,
#                 }
#                 clu_p = {
#                     "n_clusters": g["n_clusters"],
#                     "linkage": g["linkage"],
#                     "metric": g["metric"],
#                 }
#                 topics_run, _model_run = fit_topics_once(
#                     docs=docs,
#                     embeddings=embeddings,
#                     seed=run_seed,
#                     umap_params=umap_p,
#                     cluster_params=clu_p,
#                     vectorizer_params=ROBUST_BASE_VECTORIZER,
#                     calc_probs=CALC_PROBS_IN_ROBUST,
#                 )
#                 m = compute_gap_metrics(df, topics_run, topic_col_name="_topic_eval")
#                 rec.update(m)
#             except Exception as e:
#                 rec.update({k: np.nan for k in ROBUST_METRIC_COLS})
#                 rec["status"] = "error"
#                 rec["error_msg"] = f"{type(e).__name__}: {e}"
#                 print(f"[Grid {gi}, seed={run_seed}] failed: {e}")

#             rec["runtime_sec"] = round(time.time() - t0, 2)
#             grid_records.append(rec)
#             print(f"[{done}/{total_runs}] grid={gi}, seed={run_seed}, status={rec['status']}")

#     results_grid_df = pd.DataFrame(grid_records)
#     results_grid_path = ROBUST_DIR / "results_grid.csv"
#     results_grid_df.to_csv(results_grid_path, index=False)
#     print(f"Saved: {results_grid_path}")

#     pgrid = results_grid_df.copy()
#     pgrid = pgrid[pgrid["status"] == "ok"].copy()
#     pgrid["js_distance"] = pd.to_numeric(pgrid["js_distance"], errors="coerce")
#     pgrid["frontier_js_distance"] = pd.to_numeric(pgrid["frontier_js_distance"], errors="coerce")
#     pgrid["topic_size_cv"] = pd.to_numeric(pgrid["topic_size_cv"], errors="coerce")
#     pgrid["max_topic_share"] = pd.to_numeric(pgrid["max_topic_share"], errors="coerce")

#     # Plot 1: topic imbalance vs JS distance
#     if not pgrid.empty and pgrid["js_distance"].notna().any():
#         fig, ax = plt.subplots(figsize=(8, 5))
#         sc = ax.scatter(
#             pgrid["topic_size_cv"],
#             pgrid["js_distance"],
#             c=pgrid["n_topics"],
#             cmap="viridis",
#             s=50,
#             alpha=0.85,
#         )
#         ax.set_xlabel("Topic size CV")
#         ax.set_ylabel("JS distance (CN vs US)")
#         ax.set_title("Grid Robustness: topic-size imbalance vs JS distance")
#         ax.grid(True, alpha=0.3)
#         cb = plt.colorbar(sc, ax=ax)
#         cb.set_label("n_topics")
#         plt.tight_layout()
#         p_js = ROBUST_DIR / "grid_imbalance_vs_jsd.png"
#         fig.savefig(p_js, dpi=180)
#         print(f"Saved: {p_js}")
#         plt.show()

#     # Plot 2: max topic share vs frontier JS distance
#     fig, ax = plt.subplots(figsize=(8, 5))
#     if not pgrid.empty and pgrid["frontier_js_distance"].notna().any():
#         sc = ax.scatter(
#             pgrid["max_topic_share"],
#             pgrid["frontier_js_distance"],
#             c=pgrid["n_topics"],
#             cmap="plasma",
#             s=50,
#             alpha=0.85,
#         )
#         cb = plt.colorbar(sc, ax=ax)
#         cb.set_label("n_topics")
#     else:
#         ax.text(0.5, 0.5, "frontier_js_distance unavailable", ha="center", va="center", transform=ax.transAxes)
#     ax.set_xlabel("Max topic share")
#     ax.set_ylabel("Frontier JS distance")
#     ax.set_title("Grid Robustness: concentration vs Frontier JS")
#     ax.grid(True, alpha=0.3)
#     plt.tight_layout()
#     p_fr = ROBUST_DIR / "grid_concentration_vs_frontier.png"
#     fig.savefig(p_fr, dpi=180)
#     print(f"Saved: {p_fr}")
#     plt.show()
# else:
#     print("RUN_GRID=False, skipped.")


In [ ]:
# # 【单元格中文说明】
# # - 固定已训练主题赋值，对论文行做分层自助重抽样，得到 gap 指标的置信区间与直方图；不重新拟合主题模型以控制算力。


# # ══════════════════════════════════════════════════════════════════════════════
# # 16.5 Bootstrap intervals (fixed topics)
# # ══════════════════════════════════════════════════════════════════════════════
# bootstrap_df = pd.DataFrame()
# bootstrap_summary_df = pd.DataFrame()

# if RUN_BOOTSTRAP:
#     df_boot_base = _ensure_country2(df.copy())
#     base_topic_col = _detect_topic_col(df_boot_base)

#     df_boot_base = df_boot_base[df_boot_base["country2"].isin(["CN", "US"])].reset_index(drop=True)
#     df_boot_base[base_topic_col] = pd.to_numeric(df_boot_base[base_topic_col], errors="coerce").fillna(-1).astype(int)

#     strata_cols = ["country2"]
#     if BOOTSTRAP_USE_YEAR_STRATA and "year" in df_boot_base.columns:
#         yv = pd.to_numeric(df_boot_base["year"], errors="coerce")
#         if yv.notna().any():
#             df_boot_base["_year_boot"] = yv.astype("Int64")
#             strata_cols = ["_year_boot", "country2"]

#     boot_indices = stratified_bootstrap_indices(df_boot_base, strata_cols, BOOTSTRAP_B, BOOTSTRAP_SEED)
#     print(f"Bootstrap strata={strata_cols}, iterations={len(boot_indices)}")

#     boot_records = []
#     for bi, idx in enumerate(boot_indices, start=1):
#         rec = {"bootstrap_iter": int(bi), "status": "ok", "error_msg": ""}
#         try:
#             samp = df_boot_base.iloc[idx].reset_index(drop=True)
#             topics_s = samp[base_topic_col].astype(int).values
#             m = compute_gap_metrics(samp, topics_s, topic_col_name="_topic_eval")
#             rec.update(m)
#         except Exception as e:
#             rec.update({k: np.nan for k in ROBUST_METRIC_COLS})
#             rec["status"] = "error"
#             rec["error_msg"] = f"{type(e).__name__}: {e}"

#         boot_records.append(rec)
#         if bi == 1 or bi % 20 == 0 or bi == len(boot_indices):
#             print(f"Bootstrap {bi}/{len(boot_indices)} status={rec['status']}")

#     bootstrap_df = pd.DataFrame(boot_records)
#     bootstrap_path = ROBUST_DIR / "bootstrap_metrics.csv"
#     bootstrap_df.to_csv(bootstrap_path, index=False)
#     print(f"Saved: {bootstrap_path}")

#     bootstrap_summary_df = summarize_runs(bootstrap_df, ROBUST_METRIC_COLS)
#     bootstrap_summary_path = ROBUST_DIR / "bootstrap_summary.csv"
#     bootstrap_summary_df.to_csv(bootstrap_summary_path, index=False)
#     print(f"Saved: {bootstrap_summary_path}")

#     js_series = pd.to_numeric(bootstrap_df.get("js_distance", pd.Series(dtype=float)), errors="coerce").dropna()
#     if not js_series.empty:
#         fig, ax = plt.subplots(figsize=(8, 5))
#         ax.hist(js_series.values, bins=25, alpha=0.85, color="#3C6FE0", edgecolor="white")
#         ax.set_title("Bootstrap distribution: JS distance")
#         ax.set_xlabel("JS distance")
#         ax.set_ylabel("Frequency")
#         ax.grid(True, axis="y", alpha=0.3)
#         plt.tight_layout()
#         p_js = ROBUST_DIR / "bootstrap_js_distance_hist.png"
#         fig.savefig(p_js, dpi=180)
#         print(f"Saved: {p_js}")
#         plt.show()

#     fr_series = pd.to_numeric(bootstrap_df.get("frontier_js_distance", pd.Series(dtype=float)), errors="coerce").dropna()
#     fig, ax = plt.subplots(figsize=(8, 5))
#     if not fr_series.empty:
#         ax.hist(fr_series.values, bins=25, alpha=0.85, color="#E03C31", edgecolor="white")
#     else:
#         ax.text(0.5, 0.5, "frontier_js_distance unavailable", ha="center", va="center", transform=ax.transAxes)
#     ax.set_title("Bootstrap distribution: Frontier JS distance")
#     ax.set_xlabel("Frontier JS distance")
#     ax.set_ylabel("Frequency")
#     ax.grid(True, axis="y", alpha=0.3)
#     plt.tight_layout()
#     p_fr = ROBUST_DIR / "bootstrap_frontier_js_hist.png"
#     fig.savefig(p_fr, dpi=180)
#     print(f"Saved: {p_fr}")
#     plt.show()
# else:
#     print("RUN_BOOTSTRAP=False, skipped.")


In [ ]:
# # 【单元格中文说明】
# # - 将种子标准差与自助区间整理为 Markdown 摘要，提示哪些指标更稳定、建议保留的基线超参组合。


# # ══════════════════════════════════════════════════════════════════════════════
# # 16.6 Short interpretation
# # ══════════════════════════════════════════════════════════════════════════════
# from IPython.display import display, Markdown

# seed_summary = (
#     results_seed_summary_df
#     if isinstance(results_seed_summary_df, pd.DataFrame) and not results_seed_summary_df.empty
#     else (pd.read_csv(ROBUST_DIR / "results_seed_summary.csv") if (ROBUST_DIR / "results_seed_summary.csv").exists() else pd.DataFrame())
# )
# boot_summary = (
#     bootstrap_summary_df
#     if isinstance(bootstrap_summary_df, pd.DataFrame) and not bootstrap_summary_df.empty
#     else (pd.read_csv(ROBUST_DIR / "bootstrap_summary.csv") if (ROBUST_DIR / "bootstrap_summary.csv").exists() else pd.DataFrame())
# )
# grid_df_local = (
#     results_grid_df
#     if isinstance(results_grid_df, pd.DataFrame) and not results_grid_df.empty
#     else (pd.read_csv(ROBUST_DIR / "results_grid.csv") if (ROBUST_DIR / "results_grid.csv").exists() else pd.DataFrame())
# )

# def _std_of(metric, summary_df):
#     if summary_df is None or summary_df.empty:
#         return np.nan
#     x = summary_df[summary_df["metric"] == metric]
#     return float(x["std"].iloc[0]) if len(x) > 0 and pd.notna(x["std"].iloc[0]) else np.nan

# seed_js_std = _std_of("js_distance", seed_summary)
# seed_fr_std = _std_of("frontier_js_distance", seed_summary)
# seed_mncs_std = _std_of("impact_gap_mncs_us_minus_cn", seed_summary)

# boot_js = boot_summary[boot_summary["metric"] == "js_distance"] if not boot_summary.empty else pd.DataFrame()
# boot_fr = boot_summary[boot_summary["metric"] == "frontier_js_distance"] if not boot_summary.empty else pd.DataFrame()

# def _ci_str(df_row):
#     if df_row.empty:
#         return "N/A"
#     r = df_row.iloc[0]
#     if pd.isna(r["q2_5"]) or pd.isna(r["q97_5"]):
#         return "N/A"
#     return f"[{r['q2_5']:.4f}, {r['q97_5']:.4f}]"

# rec_line = "- Recommended baseline config: keep current UMAP/Agglomerative baseline unless runtime budget allows broader search."
# if not grid_df_local.empty:
#     g = grid_df_local.copy()
#     g = g[g["status"] == "ok"].copy()
#     g["js_distance"] = pd.to_numeric(g["js_distance"], errors="coerce")
#     g["topic_size_cv"] = pd.to_numeric(g["topic_size_cv"], errors="coerce")
#     g["max_topic_share"] = pd.to_numeric(g["max_topic_share"], errors="coerce")
#     g = g.dropna(subset=["topic_size_cv"])
#     if not g.empty:
#         g["baseline_dist"] = (
#             (pd.to_numeric(g["umap_n_neighbors"], errors="coerce") - UMAP_N_NEIGHBORS).abs() / 10.0
#             + (pd.to_numeric(g["umap_n_components"], errors="coerce") - UMAP_N_COMPONENTS).abs()
#             + (pd.to_numeric(g["agglom_n_clusters"], errors="coerce") - AGGLO_N_CLUSTERS).abs() / 20.0
#         )
#         g = g.sort_values(["baseline_dist", "topic_size_cv", "max_topic_share", "js_distance"], ascending=[True, True, True, True])
#         best = g.iloc[0]
#         rec_line = (
#             "- Recommended baseline config: "
#             f"UMAP(n_neighbors={int(best['umap_n_neighbors'])}, n_components={int(best['umap_n_components'])}, min_dist={float(best.get('umap_min_dist', UMAP_MIN_DIST)):.2f}), "
#             f"Agglomerative(n_clusters={int(best['agglom_n_clusters'])}, linkage={best['agglom_linkage']}, metric={best['agglom_metric']})."
#         )

# lines = [
#     "### §16 Short interpretation",
#     "- SciPy `jensenshannon(p, q)` returns **JS distance**; this section also reports **JS divergence = distance²**.",
#     f"- Seed repeatability (n={len(results_seed_df) if isinstance(results_seed_df, pd.DataFrame) else 0}): std(JS distance)={seed_js_std:.4f} ; std(Frontier JS)={seed_fr_std:.4f} ; std(Impact MNCS gap)={seed_mncs_std:.4f}.",
#     f"- Bootstrap uncertainty (B={BOOTSTRAP_B}): JS distance 95% CI={_ci_str(boot_js)} ; Frontier JS 95% CI={_ci_str(boot_fr)}.",
#     "- Stable metrics are those with low seed std and narrow bootstrap interval; unstable metrics should be interpreted with larger uncertainty.",
#     "- Grid robustness emphasizes balanced topic-size profiles while keeping core gap metrics close to baseline.",
#     rec_line,
#     "- Outlier ratio is fixed at 0 under hierarchical clustering (no explicit noise class).",
#     "- Robustness artifacts are saved under `output/cluster_results/robustness/`.",
# ]

# md_text = "\n".join(lines)
# robustness_interpretation_md = md_text
# print(md_text)
# display(Markdown(md_text))
